# Paper combined - train the edit model, then run BCO experiments

**PART 1** trains the edit agent FROM SCRATCH on the 700-graph graded realistic curriculum (dup -> stub -> detour -> drop_cover -> subtle -> eval-mix -> clean) with the unified objective: 0.5*RTT + 0.5*WMC and a network-level adjustment budget (cap at target=0.2, W=10, paper mode, demand off; the BCO search uses the two-sided |adj-target| form). **PART 2** runs the paper experiments (E1-E6) using that fine-tuned model as `OUR_MODEL_PATH`. Algorithm budgets are set for a full top-to-bottom run that should fit into roughly 25-30 hours on the current CPU calibration; rerun the timing estimator cell after any budget change.

Run top-to-bottom: training must finish before the experiment config picks up the trained weights.


In [9]:
import torch

torch.__version__

'2.11.0+cpu'

In [10]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # select GPU 1
import sys, pickle, shutil, json, random as _random
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from hydra import compose, initialize_config_dir
from tqdm.auto import tqdm
from IPython.display import Image, display

from eval_lib.context import (ROOT_DIR, CFG_DIR, DATASETS_DIR,
                              MODEL_OUTPUTS_DIR, EDIT_MODEL_WEIGHTS_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from connectpt.routes_generator import utils as lrnu
from connectpt.routes_generator.improvement_learning import (
    _get_planned_current_routes, _make_route_context_state,
    load_raw_graphs_and_lc_routes, make_improvement_batch,
    rollout_lc_improvement, train_lc_improvement_cfg)
from connectpt.routes_generator.torch_utils import (
    get_batch_tensor_from_routes, dump_routes)
from connectpt.routes_generator.transit_time_estimator import (
    ROUTE_ACTION_EXTEND, ROUTE_ACTION_HALT, ROUTE_ACTION_TRIM_END,
    ROUTE_ACTION_TRIM_START, RouteGenBatchState)
from connectpt.routes_generator.citygraph_dataset import (
    STOP_KEY, DynamicCityGraphDataset)
from connectpt.routes_generator.bee_colony import get_adjustment_degrees
from torch_geometric.data import Batch
from eval_lib.results_io import save_table
from eval_lib import build_lc_cfg, run_lc, as_route_tensor
from eval_lib import plots as route_plots

pd.set_option("display.max_columns", None)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [11]:
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

from eval_lib.context import ROOT_DIR, EDIT_MODEL_WEIGHTS_DIR
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from eval_lib import *                      # runners, builders, BENCHMARK_SPECS, ...
from eval_lib import _run_baseline          # private: not pulled by `import *`
from eval_lib import plots as route_plots
from connectpt.routes_generator.bee_colony import get_adjustment_degrees

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.set_option("display.max_columns", None)
print("device:", device)

# --- wall-clock instrumentation (per-epoch / per-iteration) ---
import time as _time
TIMING = {"epoch_s": None, "dataset_graph_s": None, "bco_iter_s": {}, "base_iter_s": {}}


device: cpu


## Configuration

`copy_full`, `copy_boundary`, `copy_mixed`, and `lc_clean` are equal-size
tiers.  The three corrupted tiers are generated from LC routes with the four
copy/subcopy mutations below.  Multiplicity grows across the curriculum up to
five routes on one stop-to-stop leg.

In [12]:
# Training is OFF: experiments use an existing checkpoint (see OUR_MODEL_PATH).
# Set True to (re)train the edit model in PART 1.
RUN_TRAINING = False

# --- dataset: five tiers (four corrupted + clean), easy -> hard/clean ---
N_GRAPHS        = 1000   # full training pool: 200 graphs/tier (5 copy tiers)
RAW_N_NODES     = 50
RAW_GRAPH_TYPE  = "mixed"
RAW_GRAPH_SEED  = 0
TARGET_N_ROUTES = 12
MIN_ROUTE_LEN   = 8
MAX_ROUTE_LEN   = 15
LC_N_SAMPLES    = 1
LC_COMBOS = [(1.0,0.0,0.0,"demand"), (0.0,1.0,0.0,"route"), (0.0,0.0,1.0,"conn")]

# Old copy-redundancy tiers (eval_lib.route_copies.COPY_TIER_CFG):
# copy_full -> copy_boundary -> copy_mixed -> covered_dup -> lc_clean. Every
# corruption injects removable route duplicates; lc_clean is the clean tier.
from eval_lib.route_copies import (COPY_TIER_CFG as TIER_CFG,
                                   D_UN_CAP_PCT as D_UN_TARGET_PCT)
TIERS = list(TIER_CFG)
DATASET_DIRNAME = "lc_copytiers_n1000_n50_r12_len8_15_v1"
NEW_DATASET_DIR = DATASETS_DIR / DATASET_DIRNAME
SUBSET_PKL = NEW_DATASET_DIR / "raw_graphs_1000.pkl"
META_CSV   = NEW_DATASET_DIR / "meta.csv"
FORCE_REGEN = False
# Corrupt-routes knobs: CORRUPT_ALL_ROUTES forces every route of every graph to
# be damaged (no healthy halt-only slots); otherwise N_ROUTES_TO_CORRUPT sets a
# minimum corrupted-route count per graph (None = each tier's own behaviour).
CORRUPT_ALL_ROUTES = False
N_ROUTES_TO_CORRUPT = None
CORRUPT_TARGET = TARGET_N_ROUTES if CORRUPT_ALL_ROUTES else N_ROUTES_TO_CORRUPT

# --- objective: SINGLE SOURCE = eval_lib/params.py (training == BCO sweeps) ---
# 0.5*RTT + 0.5*WMC(median_weighted) + ADJ_WEIGHT*|adj - ADJ_TARGET|, demand off.
from eval_lib.params import (CONNECTIVITY_MODE, DISABLED_COST_COMPONENTS,
                             UNIFIED_COST_WEIGHTS, ADJ_WEIGHT, ADJ_TARGET,
                             ADJ_TRAIN_OBJECTIVE, ADJ_GAP, ADJ_MODE)
# Vary the cost weights per training batch instead of fixed 0.5/0.5: 30%
# route-only, 30% connectivity-only, 40% random (route, conn) mixes (demand is
# disabled, so the sampler renormalizes over the two enabled components). The
# sampled weights are fed to the agent as conditioning features, so one policy
# serves the whole alpha range (validation stays at the fixed 0.5/0.5 point).
VARY_WEIGHTS = True; OP_FRACTION = 0.3; MCW_FRACTION = 0.3
# Condition the agent on a per-batch adjustment TARGET ~ U[0.1, 0.4] (one extra
# gated global feature; W stays the fixed scalar ADJ_WEIGHT). Validation uses
# the midpoint target. The BCO eval path feeds the run's target to the bee.
CONDITION_ON_ADJ_TARGET = False  # adj OFF in training (no conditioning feature)
TRAIN_ADJ_WEIGHT = 0.0           # adj penalty weight in TRAINING reward (0 = off); eval/BCO still use ADJ_WEIGHT
ADJ_TARGET_MIN, ADJ_TARGET_MAX = 0.2, 1.0
ROUTE_W = UNIFIED_COST_WEIGHTS["route_time_weight"]
CONN_W = UNIFIED_COST_WEIGHTS["median_connectivity_weight"]

# --- anti-halt-collapse ---
FORCE_NONHALT_FIRST_STEP = False
FORCE_NONHALT_UNTIL_ITER = 150   # half the run: scratch policy needs the crutch far longer than the old finetune
POSITIVE_ONLY_TRIM_REWARD = False
ZERO_TRIM_REWARD = False
ENTROPY_WEIGHT = 0.05   # keep exploring past the forced-nonhalt window (halt logit is the "safe" one)

# --- critic norm + Huber + value clip (both runs) ---
CRITIC_OVERRIDES = ["++critic_normalize_returns=true", "++critic_huber=true",
                    "++critic_huber_delta=1.0", "++critic_value_clip=0.2"]

# --- training run (FROM SCRATCH on the graded realistic curriculum) ---
import torch as _torch
TRAIN_FROM_SCRATCH = True    # random init; no parent checkpoint is loaded
N_ITERATIONS = 700           # from-scratch needs more than the 100-epoch finetune
# GPU-aware budgets: batch 4 was CPU-budgeted and starves a GPU (a 4-graph
# GATv2 forward is launch-overhead-bound). keep_rollout_on_device=True stops
# the per-step GPU->CPU state snapshot + per-minibatch re-upload round trip.
BATCH_SIZE = 16 if _torch.cuda.is_available() else 4
KEEP_ROLLOUT_ON_DEVICE = _torch.cuda.is_available()  # keep rollout states on GPU (no per-step CPU round-trip); set False if OOM
TRAIN_FRACTION = 0.9
SPLIT_SEED   = 0
MAX_ROUTE_EDIT_STEPS = MAX_ROUTE_LEN
MAX_TRIM_ACTIONS_PER_ROUTE = 1

# --- init: from scratch by default (TRAIN_FROM_SCRATCH above); the parent
#     PRESERVED checkpoint is only used when TRAIN_FROM_SCRATCH=False ---
RESUME_FROM_CHECKPOINT = False
RESUME_TAG = "resume"
BASE_MODEL_PATH = EDIT_MODEL_WEIGHTS_DIR / "improvement_lc_redundancy_rttwmc_v1_PRESERVED.pt"
REQUIRE_BASE_MODEL = not TRAIN_FROM_SCRATCH


# --- cumulative curriculum over the copy tiers (easy -> hard -> clean),
#     mirroring the reference run: full-copy -> +boundary -> +mixed ->
#     +covered -> +clean. TIERS = [copy_full, copy_boundary, copy_mixed,
#     covered_dup, lc_clean]. ---
USE_CURRICULUM = True
CURRICULUM = [
    (round(0.15 * N_ITERATIONS), TIERS[:1], "full"),
    (round(0.30 * N_ITERATIONS), TIERS[:2], "+boundary"),
    (round(0.55 * N_ITERATIONS), TIERS[:3], "+mixed"),
    (round(0.75 * N_ITERATIONS), TIERS[:4], "+covered"),
    (N_ITERATIONS,               TIERS,     "+clean"),   # last ~25% includes lc_clean
]

# --- evaluation ---
BALANCED_EVAL_WEIGHTS = (0.5, 0.5)
TRAIN_EVAL_N_PER_TIER = 4   # balanced validation monitor graphs per tier
EVAL_N_PER_TIER = 10
RUN_NAME = "NEW_lc_copytiers_curric_noadj_v1"
HISTORY_CHECKPOINT_PATH = MODEL_OUTPUTS_DIR / f"{RUN_NAME}_training_history_partial.csv"
FULL_HISTORY_RUN = f"{RUN_NAME}_{RESUME_TAG}" if RESUME_FROM_CHECKPOINT else RUN_NAME
FULL_HISTORY_CHECKPOINT = MODEL_OUTPUTS_DIR / f"{FULL_HISTORY_RUN}_training_history_partial.csv"
TENSORBOARD_LOGDIR = MODEL_OUTPUTS_DIR / "tensorboard" / FULL_HISTORY_RUN
# TensorBoard: log ONLY the metrics shown on the reference history figure
# (lc_redundancy_rttwmc_v1_PRESERVED_history): the 6 actor-curve panels.
TENSORBOARD_SCALARS = [
    "train_reward_mean", "val_delta", "val_win_rate",
    "train_action_avg_actions_per_route",
    "val_component_delta_route", "val_component_delta_connectivity",
]
PRIOR_HISTORY_FILES = [HISTORY_CHECKPOINT_PATH] if RESUME_FROM_CHECKPOINT else []

print(f"{N_GRAPHS} graphs, tiers={TIERS}; curriculum_enabled={USE_CURRICULUM}; "
      f"schedule={[c[2]+'<'+str(c[0]) for c in CURRICULUM]}")
print(f"coverage cap d_un <= {D_UN_TARGET_PCT}% (dup tiers)")
print(f"FINE-TUNE: {N_ITERATIONS} epochs, batch={BATCH_SIZE}, graphs={N_GRAPHS}, resume={RESUME_FROM_CHECKPOINT}")
print(f"base model -> {BASE_MODEL_PATH} | exists={BASE_MODEL_PATH.exists()}")
print(f"connectivity_mode -> {CONNECTIVITY_MODE}")
print(f"plain reward (positive_only={POSITIVE_ONLY_TRIM_REWARD}, zero_trim={ZERO_TRIM_REWARD})")
print(f"dataset -> {NEW_DATASET_DIR}")



1000 graphs, tiers=['copy_full', 'copy_boundary', 'copy_mixed', 'covered_dup', 'lc_clean']; curriculum_enabled=True; schedule=['full<105', '+boundary<210', '+mixed<385', '+covered<525', '+clean<700']
coverage cap d_un <= 10.0% (dup tiers)
FINE-TUNE: 700 epochs, batch=4, graphs=1000, resume=False
base model -> D:\PythonProjects\connectpt\artifacts\model_weights\improvement\improvement_lc_redundancy_rttwmc_v1_PRESERVED.pt | exists=True
connectivity_mode -> median_weighted
plain reward (positive_only=False, zero_trim=False)
dataset -> D:\PythonProjects\connectpt\datasets\lc_copytiers_n1000_n50_r12_len8_15_v1


## Generate the four-tier route-copy dataset

Every corruption copies a whole donor route or a contiguous donor subroute
into another route slot.  Prefix, suffix, and interior replacements preserve
the recipient length.  Candidates with repeated stops are rejected, so the
dataset teaches inter-route redundancy rather than synthetic self-loops.

In [13]:
if RUN_TRAINING:
    from eval_lib.route_copies import (inject_route_copies,
                                       count_changed_routes as _count_changed_routes,
                                       redundancy_stats as _redundancy_stats,
                                       uncovered_demand_pct as _uncovered_demand_pct)
    def _to_fixed(routes):
        t = as_route_tensor(routes).long()
        if t.ndim == 3:
            t = t[0]
        if t.shape[0] < TARGET_N_ROUTES:
            t = torch.cat([t, torch.full((TARGET_N_ROUTES - t.shape[0], t.shape[1]), -1, dtype=t.dtype)], 0)
        else:
            t = t[:TARGET_N_ROUTES]
        if t.shape[1] < MAX_ROUTE_LEN:
            t = torch.cat([t, torch.full((t.shape[0], MAX_ROUTE_LEN - t.shape[1]), -1, dtype=t.dtype)], 1)
        elif t.shape[1] > MAX_ROUTE_LEN:
            t = t[:, :MAX_ROUTE_LEN]
        return t


    def _tensors(g):
        return {"node_locs": g[STOP_KEY].pos.detach().cpu().clone(),
                "street_adj": g.street_adj.detach().cpu().clone(),
                "demand": g.demand.detach().cpu().clone()}


    def generate_dataset():
        if NEW_DATASET_DIR.exists():
            shutil.rmtree(NEW_DATASET_DIR)
        NEW_DATASET_DIR.mkdir(parents=True, exist_ok=True)
        _random.seed(RAW_GRAPH_SEED); torch.manual_seed(RAW_GRAPH_SEED)
        ds = DynamicCityGraphDataset(min_nodes=RAW_N_NODES, max_nodes=RAW_N_NODES,
                                     data_type=RAW_GRAPH_TYPE, mumford_style=True, pos_only=False)
        raw = [ds.generate_graph(n_nodes=RAW_N_NODES) for _ in range(N_GRAPHS)]
        per = N_GRAPHS // len(TIERS)
        tensors_all = [_tensors(g) for g in raw]

        # --- batched learned construction on the GPU --------------------------
        # Group graphs by LC objective combo (one cfg / one set of cost weights
        # per combo), then construct each group in GPU batches of LC_GEN_BATCH
        # via run_lc_batch (one forward per batch) instead of one graph at a
        # time -- this is what actually keeps the GPU busy during dataset gen.
        LC_GEN_BATCH = 64 if torch.cuda.is_available() else 8
        lc_routes = [None] * N_GRAPHS
        for _ci, (d, rt, cn, ctag) in enumerate(LC_COMBOS):
            combo_idxs = [gi for gi in range(N_GRAPHS) if gi % len(LC_COMBOS) == _ci]
            if not combo_idxs:
                continue
            c = build_lc_cfg(run_name=f"copy_cur_{ctag}", n_routes=TARGET_N_ROUTES,
                             min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN,
                             demand_time_weight=d, route_time_weight=rt,
                             median_connectivity_weight=cn,
                             connectivity_mode=CONNECTIVITY_MODE)
            for _s in tqdm(range(0, len(combo_idxs), LC_GEN_BATCH),
                           desc=f"LC construct [{ctag}]"):
                chunk = combo_idxs[_s:_s + LC_GEN_BATCH]
                routes_b = run_lc_batch(
                    c, [tensors_all[gi] for gi in chunk],
                    run_name_prefix="copy_cur_", n_samples=LC_N_SAMPLES,
                    batch_size=len(chunk))
                for _j, gi in enumerate(chunk):
                    lc_routes[gi] = routes_b[_j]

        # --- per-graph corruption + dump (CPU) --------------------------------
        subset, meta = [], []
        for gi, g in enumerate(tqdm(raw, desc="corrupt + dump")):
            tier = TIERS[min(gi // per, len(TIERS) - 1)]
            tier_cfg = TIER_CFG[tier]
            rng = _random.Random(1000 + gi)
            tn = tensors_all[gi]
            ctag = LC_COMBOS[gi % len(LC_COMBOS)][3]
            routes = _to_fixed(lc_routes[gi])
            before = _redundancy_stats(routes)
            clean_routes = routes.clone()
            routes, event_counts, _ = inject_route_copies(
                routes, tier_cfg, rng, MIN_ROUTE_LEN, MAX_ROUTE_LEN,
                demand=tn["demand"], n_nodes=RAW_N_NODES,
                street_adj=tn["street_adj"])
            after = _redundancy_stats(routes)
            n_corrupted = _count_changed_routes(clean_routes, routes)
            gdir = NEW_DATASET_DIR / f"graph_{gi:04d}"; gdir.mkdir(parents=True, exist_ok=True)
            dump_routes(f"lc_copy_cur_graph_{gi:04d}_routes", routes, out_dir=gdir)
            subset.append(g)
            meta.append({
                "graph_index": gi, "tier": tier, "tier_kind": ("clean" if not tier_cfg["kinds"] else "copies"),
                "lc_combo": ctag,
                "applied_events": int(sum(event_counts.values())),
                "n_routes_to_corrupt": ("" if CORRUPT_TARGET is None else int(CORRUPT_TARGET)),
                "n_corrupted_routes": int(n_corrupted),
                "mutation_events": json.dumps(dict(event_counts), sort_keys=True),
                "d_un_after_pct": round(_uncovered_demand_pct(routes, tn["demand"], RAW_N_NODES), 2),
                "redun_before": round(before["redundancy"], 4),
                "redun_after": round(after["redundancy"], 4),
                "max_leg_use_before": before["max_leg_use"],
                "max_leg_use_after": after["max_leg_use"],
            })
        with SUBSET_PKL.open("wb") as fh:
            pickle.dump(subset, fh)
        pd.DataFrame(meta).to_csv(META_CSV, index=False)
        print(f"Saved {len(subset)} graphs -> {NEW_DATASET_DIR}")


    _have = len(list(NEW_DATASET_DIR.glob("graph_*"))) if NEW_DATASET_DIR.exists() else 0
    if SUBSET_PKL.exists() and _have == N_GRAPHS and META_CSV.exists() and not FORCE_REGEN:
        print(f"dataset already exists ({_have}) -> skip")
    else:
        if _have and _have != N_GRAPHS:
            print(f"found {_have} graphs, expected {N_GRAPHS} -> regenerate")
        _tg = _time.perf_counter()
        generate_dataset()
        TIMING["dataset_graph_s"] = (_time.perf_counter() - _tg) / max(1, N_GRAPHS)
        print(f"[timing] dataset gen: {TIMING['dataset_graph_s']:.2f} s/graph "
              f"(scales with N_GRAPHS: {N_GRAPHS} now)")

## Load, split, and define the cumulative curriculum

In [14]:
if RUN_TRAINING:
    # Data loading, the tier-stratified train/val split and the curriculum
    # schedule now live in TrainingDataModule (refactor C6/C10); the split
    # reproduces the previous inline logic exactly (pinned by
    # tests/test_training_data.py).
    from connectpt.routes_generator.training import TrainingDataModule

    _dm = TrainingDataModule(
        raw_graphs_path=SUBSET_PKL, lc_results_dir=NEW_DATASET_DIR, device=device,
        min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN,
        target_n_routes=TARGET_N_ROUTES,
    ).setup()
    graphs, seed_routes = _dm.graphs, _dm.seed_routes
    meta_df = pd.read_csv(META_CSV)
    N = len(graphs)
    print(f"loaded {N} graphs; seed_routes={tuple(seed_routes.shape)}")
    print("copy/subcopy corruption summary by tier:")
    display(meta_df.groupby("tier")[[
        "applied_events",
        "redun_before", "redun_after", "max_leg_use_after", "d_un_after_pct",
    ]].mean().round(3).reindex(TIERS))

    # Each tier contributes its own train/val so EVERY tier has
    # >= TRAIN_EVAL_N_PER_TIER validation graphs (robust for small N / smoke runs).
    TIER_OF = dict(zip(meta_df["graph_index"], meta_df["tier"]))
    (TRAIN_INDICES, VAL_INDICES, MONITOR_VAL_INDICES,
     _train_by_tier, _val_by_tier_t) = _dm.stratified_split(
        TIER_OF, TIERS, train_fraction=TRAIN_FRACTION,
        n_val_per_tier=TRAIN_EVAL_N_PER_TIER, seed=SPLIT_SEED)
    print(f"validation graphs={len(VAL_INDICES)}; balanced train-loop monitor={len(MONITOR_VAL_INDICES)}")
    print("train graphs per tier:", {tier: len(idx) for tier, idx in _train_by_tier.items()})

    # curriculum_fn / val_curriculum_fn map an iteration -> the active tiers'
    # train / val indices (cumulative easy -> hard -> clean schedule).
    curriculum_fn, val_curriculum_fn = TrainingDataModule.build_curriculum(
        CURRICULUM, _train_by_tier, _val_by_tier_t)


    def _active_tiers(iteration):
        for until, tiers, label in CURRICULUM:
            if iteration < until:
                return tiers, label
        return CURRICULUM[-1][1], CURRICULUM[-1][2]


    def stage_spans():
        '''[(start_iter, end_iter, label)] for curriculum shading.'''
        spans, prev = [], 0
        for until, _tiers, label in CURRICULUM:
            spans.append((prev + 1, until, label)); prev = until
        return spans


## Model and objective builders

Two helpers used by the fine-tune run below. `build_edit_run` composes the
hydra config, builds a **fresh** trim model + cost module, selects which of
the three cost components (`demand` / `route` / `connectivity`) are active,
and enables the optional adjustment-degree shaping through `adj_weight`.
`train_edit_run` wraps `train_lc_improvement_cfg` and returns the in-memory
history. Adjustment conditioning stays off, so the fine-tuned actor keeps the
same architecture as the RTT+connectivity checkpoint.


In [15]:
def build_edit_run(run_name, disabled_components, vary_weights=True,
                   critic_overrides=None, route_time_weight=None, adj_weight=0.0,
                   adj_target=None, adj_objective=None, adj_gap=None, adj_mode=None):
    """Compose cfg + build a fresh trim model/cost module for one training run."""
    critic_overrides = list(CRITIC_OVERRIDES if critic_overrides is None
                            else critic_overrides)
    adj_target = ADJ_TARGET if adj_target is None else adj_target
    adj_objective = ADJ_TRAIN_OBJECTIVE if adj_objective is None else adj_objective
    adj_gap = ADJ_GAP if adj_gap is None else adj_gap
    adj_mode = ADJ_MODE if adj_mode is None else adj_mode
    # Static training settings now live in cfg/train/edit.yaml (verified to
    # compose to the same cfg as the old override list). Only the per-run sweep
    # variables are passed as overrides here.
    overrides = [
        f"++run_name={run_name}",
        f"++adjustment_degree_weight={float(adj_weight)}",
        f"++adjustment_degree_target={float(adj_target)}",
        f"++adjustment_degree_objective={adj_objective}",
        f"++adjustment_degree_gap={float(adj_gap)}",
        f"++adjustment_degree_mode={adj_mode}",
        f"++keep_rollout_on_device={str(KEEP_ROLLOUT_ON_DEVICE).lower()}",
    ]
    if CONDITION_ON_ADJ_TARGET:
        overrides += [
            "++adjustment_conditioning=true",
            "++model.route_generator.kwargs.n_adjustment_cond_feats=1",
        ]
    overrides += list(critic_overrides)
    with initialize_config_dir(config_dir=str(CFG_DIR), version_base=None):
        cfg = compose(config_name="train/edit", overrides=overrides)
    _, run_name, _, cost_obj, model = lrnu.process_standard_experiment_cfg(
        cfg, run_name_prefix="improvement_")
    cost_obj.ignore_stops_oob = True
    cost_obj.set_enabled_components(disabled_components=disabled_components or None)
    if vary_weights:
        cost_obj.variable_weights = True
        cost_obj.pp_fraction = 0.0
        cost_obj.op_fraction = OP_FRACTION
        cost_obj.mcw_fraction = MCW_FRACTION
    else:
        cost_obj.variable_weights = False
    if route_time_weight is not None:
        cost_obj.route_time_weight = float(route_time_weight)
    best_path = EDIT_MODEL_WEIGHTS_DIR / f"{run_name}.pt"
    print(f"run_name={run_name} | enabled={list(cost_obj.enabled_component_names)} | "
          f"variable_weights={cost_obj.variable_weights} | edge_dim={model.edge_feat_dim}")
    print(f"  connectivity_mode={cost_obj.connectivity_mode}")
    print(f"  trim reward: zero_trim={cfg.get('zero_trim_reward')} "
          f"pos_only={cfg.get('positive_only_trim_reward')} | "
          f"critic_norm={cfg.get('critic_normalize_returns')}")
    print(f"  adj shaping: W={cfg.get('adjustment_degree_weight')} "
          f"target={cfg.get('adjustment_degree_target')} "
          f"objective={cfg.get('adjustment_degree_objective')} "
          f"mode={cfg.get('adjustment_degree_mode')} gap={cfg.get('adjustment_degree_gap')}")
    return cfg, cost_obj, model, run_name, best_path


def _pad_route_tensor(t, n_routes, width):
    if t.ndim == 2:
        t = t.unsqueeze(0)
    out = torch.full((t.shape[0], n_routes, width), -1,
                     dtype=t.dtype, device=t.device)
    nr = min(n_routes, t.shape[1])
    w = min(width, t.shape[2])
    out[:, :nr, :w] = t[:, :nr, :w]
    return out


def _rollout_adjustment_kwargs_for_model(model, *, target=None):
    n_adj_feats = int(getattr(model, "n_adjustment_cond_feats", 0) or 0)
    if n_adj_feats <= 0:
        return {}
    target = ADJ_TARGET if target is None else target
    kwargs = dict(adjustment_target=float(target),
                  adjustment_use_current=False,
                  adjustment_gap=ADJ_GAP,
                  adjustment_mode=ADJ_MODE)
    if n_adj_feats > 1:
        kwargs["adjustment_weight"] = float(ADJ_WEIGHT)
    return kwargs


def _adj_penalty_from_routes(routes, reference, cost_obj, *, weight, target,
                             objective, gap, mode):
    n_routes = max(int(routes.shape[1]), int(reference.shape[1]))
    width = max(int(routes.shape[2]), int(reference.shape[2]))
    routes = _pad_route_tensor(routes, n_routes, width)
    reference = _pad_route_tensor(reference, n_routes, width)
    adj = get_adjustment_degrees(
        routes, reference, cost_obj.symmetric_routes, gap=gap, mode=mode)
    # NETWORK-level form (matches MyCostModule._adjustment_penalty and the PPO
    # shaping): average the degrees first, then apply the objective transform.
    net_adj = adj.mean(dim=1)
    if objective == "cap":
        pen = (net_adj - float(target)).clamp(min=0.0)
    elif objective == "cap_sq":
        pen = (net_adj - float(target)).clamp(min=0.0) ** 2
    elif objective == "target":
        pen = (net_adj - float(target)).abs()
    else:
        pen = net_adj
    return float(weight) * pen, net_adj


# train_lc_improvement_cfg validates and saves best checkpoints through the
# module-level evaluate_lc_improvement. Patch it locally so validation is scored
# with the same network-level adjustment penalty (ADJ_TRAIN_OBJECTIVE) as the
# PPO reward shaping.
import connectpt.routes_generator.improvement_learning as _il_mod
if not hasattr(_il_mod, "_paper_combined_raw_evaluate_lc_improvement"):
    _il_mod._paper_combined_raw_evaluate_lc_improvement = _il_mod.evaluate_lc_improvement
_RAW_EVALUATE_LC_IMPROVEMENT = _il_mod._paper_combined_raw_evaluate_lc_improvement


def evaluate_lc_improvement_adj_aware(model, cost_obj, graphs, seed_routes, indices,
                                      device, min_route_len, max_route_len,
                                      batch_size=8, force_nonhalt_first_step=False,
                                      max_route_edit_steps=None,
                                      max_trim_actions_per_route=1,
                                      return_action_stats=False,
                                      target_n_routes=None,
                                      return_best_routes=False,
                                      adjustment_target=None,
                                      adjustment_weight=None,
                                      adjustment_use_current=False,
                                      adjustment_gap=0.1,
                                      adjustment_mode="paper"):
    if adjustment_target is None:
        _adj_kwargs = _rollout_adjustment_kwargs_for_model(model)
        adjustment_target = _adj_kwargs.get("adjustment_target")
        adjustment_weight = _adj_kwargs.get("adjustment_weight", adjustment_weight)
        adjustment_use_current = _adj_kwargs.get(
            "adjustment_use_current", adjustment_use_current)
        adjustment_gap = _adj_kwargs.get("adjustment_gap", adjustment_gap)
        adjustment_mode = _adj_kwargs.get("adjustment_mode", adjustment_mode)
    result = _RAW_EVALUATE_LC_IMPROVEMENT(
        model, cost_obj, graphs, seed_routes, indices, device,
        min_route_len, max_route_len, batch_size=batch_size,
        force_nonhalt_first_step=force_nonhalt_first_step,
        max_route_edit_steps=max_route_edit_steps,
        max_trim_actions_per_route=max_trim_actions_per_route,
        return_action_stats=return_action_stats,
        target_n_routes=target_n_routes,
        return_best_routes=return_best_routes,
        adjustment_target=adjustment_target,
        adjustment_weight=adjustment_weight,
        adjustment_use_current=adjustment_use_current,
        adjustment_gap=adjustment_gap,
        adjustment_mode=adjustment_mode)

    eval_weights = cost_obj.get_weights(device)
    raw_seed_costs, raw_final_costs = [], []
    seed_adj_pens, final_adj_pens = [], []
    seed_adjs, final_adjs = [], []

    batch_splits = list(indices.split(batch_size))
    for batch_indices, final_routes_cpu in zip(batch_splits, result["routes"]):
        graph_batch, route_batch = make_improvement_batch(
            graphs, seed_routes, batch_indices, device, training=False,
            target_n_routes=target_n_routes)
        final_routes = final_routes_cpu.to(device)
        if final_routes.ndim == 2:
            final_routes = final_routes.unsqueeze(0)

        seed_state = RouteGenBatchState(
            graph_batch, cost_obj, route_batch.shape[1],
            min_route_len, max_route_len,
            cost_weights=_il_mod._clone_cost_weights(eval_weights))
        seed_state.add_new_routes(route_batch)
        final_state = RouteGenBatchState(
            graph_batch, cost_obj, final_routes.shape[1],
            min_route_len, max_route_len,
            cost_weights=_il_mod._clone_cost_weights(eval_weights))
        final_state.add_new_routes(final_routes)

        raw_seed_costs.append(cost_obj(seed_state).cost.detach().cpu())
        raw_final_costs.append(cost_obj(final_state).cost.detach().cpu())
        sp, sa = _adj_penalty_from_routes(
            route_batch, route_batch, cost_obj,
            weight=TRAIN_ADJ_WEIGHT, target=ADJ_TARGET,
            objective=ADJ_TRAIN_OBJECTIVE, gap=ADJ_GAP, mode=ADJ_MODE)
        fp, fa = _adj_penalty_from_routes(
            final_routes, route_batch, cost_obj,
            weight=TRAIN_ADJ_WEIGHT, target=ADJ_TARGET,
            objective=ADJ_TRAIN_OBJECTIVE, gap=ADJ_GAP, mode=ADJ_MODE)
        seed_adj_pens.append(sp.detach().cpu())
        final_adj_pens.append(fp.detach().cpu())
        seed_adjs.append(sa.detach().cpu())
        final_adjs.append(fa.detach().cpu())

    raw_seed_costs = torch.cat(raw_seed_costs)
    raw_final_costs = torch.cat(raw_final_costs)
    seed_adj_pens = torch.cat(seed_adj_pens)
    final_adj_pens = torch.cat(final_adj_pens)
    seed_adjs = torch.cat(seed_adjs)
    final_adjs = torch.cat(final_adjs)
    adj_seed_costs = raw_seed_costs + seed_adj_pens
    adj_final_costs = raw_final_costs + final_adj_pens

    result.update({
        "raw_seed_cost": raw_seed_costs.mean().item(),
        "raw_final_cost": raw_final_costs.mean().item(),
        "raw_delta": (raw_seed_costs - raw_final_costs).mean().item(),
        "seed_adjustment_penalty": seed_adj_pens.mean().item(),
        "final_adjustment_penalty": final_adj_pens.mean().item(),
        "seed_adjustment_degree": seed_adjs.mean().item(),
        "final_adjustment_degree": final_adjs.mean().item(),
        "adjustment_penalty_delta": (seed_adj_pens - final_adj_pens).mean().item(),
        "adjustment_degree_delta": (seed_adjs - final_adjs).mean().item(),
        "seed_cost": adj_seed_costs.mean().item(),
        "final_cost": adj_final_costs.mean().item(),
        "delta": (adj_seed_costs - adj_final_costs).mean().item(),
        "win_rate": (adj_final_costs < adj_seed_costs).float().mean().item(),
    })
    return result


_il_mod.evaluate_lc_improvement = evaluate_lc_improvement_adj_aware


def train_edit_run(model, cost_obj, cfg, run_name, best_path,
                   n_iterations, batch_size, train_indices, val_indices,
                   curriculum_fn=None, val_curriculum_fn=None, checkpoint=None):
    """Train one edit model and return its in-memory history DataFrame."""
    result = train_lc_improvement_cfg(
        model=model, cost_obj=cost_obj, graphs=graphs, seed_routes=seed_routes,
        device=device, cfg=cfg, output_dir=MODEL_OUTPUTS_DIR, run_name=run_name,
        train_fraction=TRAIN_FRACTION, batch_size=batch_size,
        min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN, seed=SPLIT_SEED,
        max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
        max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE,
        target_n_routes=TARGET_N_ROUTES,
        train_indices=train_indices, val_indices=val_indices,
        best_model_path=best_path, n_iterations=n_iterations,
        force_nonhalt_first_step=FORCE_NONHALT_FIRST_STEP,
        curriculum_fn=curriculum_fn,
        val_curriculum_fn=val_curriculum_fn,
        history_checkpoint_path=checkpoint,
        tensorboard_logdir=globals().get("TENSORBOARD_LOGDIR"),
        tensorboard_scalars=globals().get("TENSORBOARD_SCALARS"),
    )
    df = pd.DataFrame(result["history"])
    save_table(df, f"{run_name}_training_history")
    print(f"history rows={len(df)}; best -> {best_path}")
    return df


print("builders ready: build_edit_run(), train_edit_run()")
print("adj-aware validation installed: val cost/delta/win_rate include the network-level ADJ_TRAIN_OBJECTIVE adjustment penalty")


builders ready: build_edit_run(), train_edit_run()
adj-aware validation installed: val cost/delta/win_rate include the network-level ADJ_TRAIN_OBJECTIVE adjustment penalty


## Clean LC baseline costs for history plots

Generate clean LC routes for the validation monitor graphs, score them with the same route/connectivity scalar used by the training-history plots, and save the result as a standalone CSV baseline.


In [16]:
# Minimal clean-LC baseline for training-history visualization.
# Produces one standalone CSV with per-curriculum-stage clean LC baseline costs.
RUN_CLEAN_LC_BASELINE = False
CLEAN_LC_BASELINE_PATH = MODEL_OUTPUTS_DIR / f"{RUN_NAME}_clean_lc_baseline_cost.csv"
CLEAN_LC_BASELINE_BATCH = 16
# Number of validation graphs per active tier used only for the baseline.
# Increase this for smoother publication plots; it does not affect training.
CLEAN_LC_BASELINE_N_PER_TIER = N_GRAPHS // len(TIERS)
# Use 0.0 to plot the pure route/connectivity clean-LC baseline. Set to
# TRAIN_ADJ_WEIGHT if you explicitly want the adjustment-penalized objective.
CLEAN_LC_BASELINE_ADJ_WEIGHT = 0.0

if RUN_CLEAN_LC_BASELINE:
    from connectpt.routes_generator.improvement_learning import _clone_cost_weights

    if "graphs" not in globals() or "seed_routes" not in globals():
        graphs, seed_routes = load_raw_graphs_and_lc_routes(SUBSET_PKL, NEW_DATASET_DIR)
    if "meta_df" not in globals():
        meta_df = pd.read_csv(META_CSV)
    _baseline_pool_by_tier = {
        tier: meta_df.loc[meta_df["tier"] == tier, "graph_index"].astype(int).tolist()
        for tier in TIERS
    }

    def _baseline_to_fixed(routes):
        t = as_route_tensor(routes).long()
        if t.ndim == 3:
            t = t[0]
        if t.shape[0] < TARGET_N_ROUTES:
            t = torch.cat([t, torch.full((TARGET_N_ROUTES - t.shape[0], t.shape[1]), -1, dtype=t.dtype)], 0)
        else:
            t = t[:TARGET_N_ROUTES]
        if t.shape[1] < MAX_ROUTE_LEN:
            t = torch.cat([t, torch.full((t.shape[0], MAX_ROUTE_LEN - t.shape[1]), -1, dtype=t.dtype)], 1)
        elif t.shape[1] > MAX_ROUTE_LEN:
            t = t[:, :MAX_ROUTE_LEN]
        return t

    def _baseline_tensors(g):
        return {"node_locs": g[STOP_KEY].pos.detach().cpu().clone(),
                "street_adj": g.street_adj.detach().cpu().clone(),
                "demand": g.demand.detach().cpu().clone()}

    # Clean-LC baseline cost: the unified objective (RTT+WMC, demand off, fixed
    # 0.5/0.5 weights) with the adjustment penalty off. Built via the library
    # CostFactory + objective YAML instead of build_edit_run + magic constants
    # (route/conn weights, adj target/objective/gap/mode all live in the YAML now).
    from connectpt.routes_generator.objectives import CostFactory
    _cost_base = CostFactory.build_unified("rtt_wmc_no_demand", for_training=True)
    _cost_base.variable_weights = False
    _cost_base.adjustment_degree_weight = float(CLEAN_LC_BASELINE_ADJ_WEIGHT)
    _cost_base.ignore_stops_oob = True
    _cost_base.to(device)
    _eval_weights = _cost_base.get_weights(device)

    _baseline_by_tier = {
        tier: [int(i) for i in _baseline_pool_by_tier[tier][:CLEAN_LC_BASELINE_N_PER_TIER]]
        for tier in TIERS
    }
    _baseline_indices = sorted({gi for _idxs in _baseline_by_tier.values() for gi in _idxs})
    print("clean LC baseline graphs per tier:", {tier: len(_idxs) for tier, _idxs in _baseline_by_tier.items()})
    _clean_routes = seed_routes.clone()
    for _ci, (_d, _rt, _cn, _ctag) in enumerate(LC_COMBOS):
        _idxs = [gi for gi in _baseline_indices if gi % len(LC_COMBOS) == _ci]
        if not _idxs:
            continue
        _lc_cfg = build_lc_cfg(
            run_name=f"clean_lc_baseline_{_ctag}",
            n_routes=TARGET_N_ROUTES,
            min_route_len=MIN_ROUTE_LEN,
            max_route_len=MAX_ROUTE_LEN,
            demand_time_weight=_d,
            route_time_weight=_rt,
            median_connectivity_weight=_cn,
            connectivity_mode=CONNECTIVITY_MODE)
        for _s in range(0, len(_idxs), CLEAN_LC_BASELINE_BATCH):
            _chunk = _idxs[_s:_s + CLEAN_LC_BASELINE_BATCH]
            _routes_b = run_lc_batch(
                _lc_cfg, [_baseline_tensors(graphs[gi]) for gi in _chunk],
                run_name_prefix="clean_lc_baseline_",
                n_samples=LC_N_SAMPLES,
                batch_size=len(_chunk))
            for _j, gi in enumerate(_chunk):
                _clean_routes[gi] = _baseline_to_fixed(_routes_b[_j])

    def _mean_cost(routes_src, idxs):
        idxs = torch.as_tensor(list(map(int, idxs)), dtype=torch.long)
        if len(idxs) == 0:
            return float("nan")
        costs = []
        for _chunk in idxs.split(CLEAN_LC_BASELINE_BATCH):
            _gb, _rb = make_improvement_batch(
                graphs, routes_src, _chunk, device, training=False,
                target_n_routes=TARGET_N_ROUTES)
            _state = RouteGenBatchState(
                _gb, _cost_base, _rb.shape[1], MIN_ROUTE_LEN, MAX_ROUTE_LEN,
                cost_weights=_clone_cost_weights(_eval_weights))
            _state.add_new_routes(_rb)
            _res = _cost_base(_state)
            costs.append(_res.cost.detach().cpu())
        return torch.cat(costs).mean().item()

    _tier_of = dict(zip(meta_df["graph_index"].astype(int), meta_df["tier"]))
    if USE_CURRICULUM:
        _stage_rows = []
        for _until, _tiers, _label in CURRICULUM:
            _idxs = [gi for tier in _tiers for gi in _baseline_by_tier.get(tier, [])]
            _stage_rows.append({
                "until_epoch": int(_until),
                "curriculum_stage": _label,
                "active_tiers": ";".join(_tiers),
                "n_graphs": len(_idxs),
                "n_per_tier_target": int(CLEAN_LC_BASELINE_N_PER_TIER),
                "clean_lc_cost": _mean_cost(_clean_routes, _idxs),
                "seed_cost": _mean_cost(seed_routes, _idxs),
            })
    else:
        _idxs = _baseline_indices
        _stage_rows = [{
            "until_epoch": int(N_ITERATIONS),
            "curriculum_stage": "all",
            "active_tiers": ";".join(sorted({_tier_of[i] for i in _idxs})),
            "n_graphs": len(_idxs),
            "n_per_tier_target": int(CLEAN_LC_BASELINE_N_PER_TIER),
            "clean_lc_cost": _mean_cost(_clean_routes, _idxs),
            "seed_cost": _mean_cost(seed_routes, _idxs),
        }]

    clean_lc_baseline_df = pd.DataFrame(_stage_rows)
    CLEAN_LC_BASELINE_PATH.parent.mkdir(parents=True, exist_ok=True)
    clean_lc_baseline_df.to_csv(CLEAN_LC_BASELINE_PATH, index=False)
    print(f"clean LC baseline -> {CLEAN_LC_BASELINE_PATH}")
    display(clean_lc_baseline_df.round(4))




## From-scratch training - route + connectivity + adj (W=10)

This run starts from the final-experiments RTT + connectivity checkpoint,
keeps `route` and `connectivity` active at 0.5/0.5 (demand off), and adds
network-level adjustment-budget reward shaping (cap at target=0.2, W=10, paper mode). The budget is 100
epochs over the current 500-graph copy-redundancy curriculum, batch 4, with
balanced per-tier validation monitoring. The resulting checkpoint is saved
under a new `adjcap` run name and is used by the evaluation cells below.

If `RESUME_FROM_CHECKPOINT=True`, this cell reloads the adj-finetune output
first; otherwise it initializes from `BASE_MODEL_PATH`.


In [17]:
if RUN_TRAINING:
    # Train from scratch: route + connectivity (0.5/0.5) + adj penalty (W=ADJ_WEIGHT); demand off.
    cfg, cost_obj, model, run_name, BEST_MODEL_PATH = build_edit_run(
        run_name=RUN_NAME,
        disabled_components=DISABLED_COST_COMPONENTS,
        vary_weights=VARY_WEIGHTS,
        route_time_weight=ROUTE_W,
        adj_weight=TRAIN_ADJ_WEIGHT,
        adj_target=ADJ_TARGET,
        adj_objective=ADJ_TRAIN_OBJECTIVE,
        adj_gap=ADJ_GAP, adj_mode=ADJ_MODE)
    cost_obj.median_connectivity_weight = float(CONN_W)   # fix conn weight (demand off)
    print(f"train objective: route={ROUTE_W} conn={CONN_W} demand=off | "
          f"adj W={TRAIN_ADJ_WEIGHT} (train OFF) target={ADJ_TARGET} "
          f"objective={ADJ_TRAIN_OBJECTIVE} mode={ADJ_MODE} gap={ADJ_GAP}")

    # From-scratch by default; resume / parent-checkpoint init only when asked.
    if RESUME_FROM_CHECKPOINT and BEST_MODEL_PATH.exists():
        model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
        print(f"resumed weights <- {BEST_MODEL_PATH}")
    elif TRAIN_FROM_SCRATCH:
        print(f"training FROM SCRATCH (random init) -> {BEST_MODEL_PATH.name}")
    elif BASE_MODEL_PATH.exists():
        model.load_state_dict(torch.load(BASE_MODEL_PATH, map_location=device))
        print(f"initialized from parent PRESERVED checkpoint <- {BASE_MODEL_PATH}")
    elif REQUIRE_BASE_MODEL:
        raise FileNotFoundError(
            "No base RTT+connectivity model found. Checked: "
            + str(BASE_MODEL_PATH)
        )
    else:
        print("base checkpoint missing; training from scratch")

    _t_train0 = _time.perf_counter()
    history_df = train_edit_run(
        model, cost_obj, cfg, FULL_HISTORY_RUN, BEST_MODEL_PATH,
        n_iterations=N_ITERATIONS, batch_size=BATCH_SIZE,
        train_indices=TRAIN_INDICES, val_indices=MONITOR_VAL_INDICES,
        curriculum_fn=(curriculum_fn if USE_CURRICULUM else None),
        val_curriculum_fn=(val_curriculum_fn if USE_CURRICULUM else None),
        checkpoint=FULL_HISTORY_CHECKPOINT)
    print(f"continuation history -> {FULL_HISTORY_RUN}_training_history "
          f"(partial: {FULL_HISTORY_CHECKPOINT.name})")
    TIMING["epoch_s"] = (_time.perf_counter() - _t_train0) / max(1, N_ITERATIONS)
    print(f"[timing] training: {TIMING['epoch_s']:.2f} s/epoch over {N_ITERATIONS} epochs")

## Train the edit model (cumulative curriculum)

Fine-tune the edit model over the cumulative curriculum (duplicate -> boundary
-> mixed -> covered -> clean). Training history is checkpointed to a partial CSV;
on resume the prior history is prepended.

In [18]:
if RUN_TRAINING:
    if "pd" not in globals():
        import pandas as pd
    if "plt" not in globals():
        import matplotlib.pyplot as plt
    from pathlib import Path

    # --- stitch history: prior part(s) (e.g. machine-1) + this run's continuation ---
    _parts, _labels = [], []
    for _f in (PRIOR_HISTORY_FILES if "PRIOR_HISTORY_FILES" in globals() else []):
        _f = Path(_f)
        if _f.exists():
            _parts.append(pd.read_csv(_f)); _labels.append(f"{_f.name}({len(_parts[-1])})")
    if "history_df" in globals():
        _parts.append(history_df.copy()); _labels.append(f"in-memory({len(history_df)})")
    elif "FULL_HISTORY_CHECKPOINT" in globals() and Path(FULL_HISTORY_CHECKPOINT).exists():
        _parts.append(pd.read_csv(FULL_HISTORY_CHECKPOINT))
        _labels.append(f"{Path(FULL_HISTORY_CHECKPOINT).name}({len(_parts[-1])})")
    if not _parts:
        raise FileNotFoundError("No history found. Train at least one epoch first.")
    h = pd.concat(_parts, ignore_index=True)
    h["epoch"] = range(1, len(h) + 1)          # continuous axis across stitched parts
    print(f"history stitched: {' + '.join(_labels)} => {len(h)} epochs")

    def _num(col):
        return pd.to_numeric(h[col], errors="coerce") if col in h.columns else None

    # curriculum stage spans derived from the stitched data (robust to stitching)
    if "curriculum_stage" in h.columns and h["curriculum_stage"].notna().any():
        _spans = []
        for _epoch, _label in h[["epoch", "curriculum_stage"]].dropna().itertuples(index=False, name=None):
            _epoch = int(_epoch)
            if _spans and _spans[-1][2] == _label:
                _spans[-1] = (_spans[-1][0], _epoch, _label)
            else:
                _spans.append((_epoch, _epoch, _label))
    elif "stage_spans" in globals():
        _spans = stage_spans()
    else:
        _spans = []
    _colors = ["#eaf3ff", "#eafbea", "#fff6e6", "#fdeaea", "#f0eaff"]
    def _shade(ax):
        for k, (s, e, lab) in enumerate(_spans):
            ax.axvspan(s, e, color=_colors[k % len(_colors)], alpha=0.6, zorder=0)
            ax.axvline(s, color="gray", lw=0.6, ls=":")

    fig, ax = plt.subplots(2, 3, figsize=(17, 8), constrained_layout=True)
    panels = [("train_reward_mean","train reward"), ("val_delta","val cost delta (+=улучш.)"),
              ("val_win_rate","val win rate"), ("train_action_avg_actions_per_route","avg edits/route"),
              ("val_component_delta_route","val route delta"),
              ("val_component_delta_connectivity","val conn delta")]
    for a,(col,title) in zip(ax.flat, panels):
        _shade(a); y=_num(col)
        if y is not None and y.notna().any():
            a.plot(h["epoch"], y, marker="o", ms=2, color="tab:blue", zorder=3)
        a.axhline(0,color="k",lw=0.7); a.set_title(title); a.set_xlabel("epoch"); a.grid(alpha=0.2)
    # подписи стадий сверху
    for s,e,lab in _spans:
        ax[0,0].text((s+e)/2, ax[0,0].get_ylim()[1], lab, ha="center", va="bottom", fontsize=8)
    fig.suptitle("Actor curves + curriculum stages (заливка = стадия; история сшита)",
                 fontsize=13, fontweight="bold")
    plt.show()    # keep the inline image as before

    # --- TensorBoard: persist the full (stitched) history + this figure ---
    # Inline image stays above; this just ALSO mirrors every scalar column to a
    # TensorBoard run so the history is browsable/zoomable and comparable across
    # runs.  Launch:  tensorboard --logdir <MODEL_OUTPUTS_DIR>/tensorboard
    from torch.utils.tensorboard import SummaryWriter

    def _tb_tag(col):
        for pre, grp in (("train_ppo_", "ppo/"), ("train_critic_", "critic/"),
                         ("train_action_", "action/"),
                         ("train_component_", "component/train_"),
                         ("val_component_", "component/val_"),
                         ("train_", "train/"), ("val_", "val/")):
            if col.startswith(pre):
                return grp + col[len(pre):]
        return "misc/" + col

    _tb_run = FULL_HISTORY_RUN if "FULL_HISTORY_RUN" in globals() else RUN_NAME
    _tb_dir = MODEL_OUTPUTS_DIR / "tensorboard" / _tb_run
    _tb_dir.mkdir(parents=True, exist_ok=True)
    _writer = SummaryWriter(log_dir=str(_tb_dir))
    _epochs = h["epoch"].astype(int).tolist()
    _tb_cols = globals().get("TENSORBOARD_SCALARS")
    _n_scalars = 0
    for _col in h.columns:
        if _col == "epoch":
            continue
        if _tb_cols is not None and _col not in _tb_cols:
            continue
        _y = pd.to_numeric(h[_col], errors="coerce")
        if not _y.notna().any():
            continue
        for _ep, _v in zip(_epochs, _y.tolist()):
            if _v == _v:  # skip NaN
                _writer.add_scalar(_col, float(_v), _ep)
        _n_scalars += 1
    _writer.add_figure("actor_curves", fig, global_step=_epochs[-1])
    _writer.flush(); _writer.close()
    plt.close(fig)
    print(f"[tensorboard] {_n_scalars} scalar series ({len(_epochs)} epochs) -> {_tb_dir}")
    print(f"[tensorboard] launch:  tensorboard --logdir \"{MODEL_OUTPUTS_DIR / 'tensorboard'}\"")

## Critic diagnostics (per-component critic MSE / explained variance)

In [19]:
if RUN_TRAINING:
    crit_cols = [c for c in h.columns if "critic" in c.lower()]
    print("critic columns:", crit_cols)
    if crit_cols:
        n=len(crit_cols)
        fig, ax = plt.subplots(1, n, figsize=(5*n, 4), squeeze=False, constrained_layout=True)
        for a, col in zip(ax[0], crit_cols):
            _shade(a); y=_num(col)
            if y is not None and y.notna().any():
                a.plot(h["epoch"], y, marker="o", ms=2, color="tab:orange", zorder=3)
            a.set_title(col, fontsize=9); a.set_xlabel("epoch"); a.grid(alpha=0.2)
            if "explained" in col: a.axhline(0, color="k", lw=0.7)
        fig.suptitle("Critic metrics + curriculum stages", fontsize=13, fontweight="bold")
        plt.show(); plt.close(fig)
        display(h[["epoch","curriculum_stage"]+crit_cols].iloc[::max(1,len(h)//15)].round(4))

## Evaluate balanced policy by tier

The table reports before/after route-level redundancy, `ATT`, `RTT`,
connectivity, and demand percentages by transfer bucket (`d0`, `d1`, `d2`,
`d_un`). `Adj(current, seed)` remains available only in the example plots.

In [20]:
if RUN_TRAINING:
    from eval_lib.route_copies import redundancy_stats as _redundancy_stats
    # Standalone eval: if the training cell wasn't run, rebuild the model + cost
    # module from the saved checkpoint so this cell (and the viz below) work alone.
    if "model" not in globals() or "cost_obj" not in globals():
        cfg, cost_obj, model, run_name, BEST_MODEL_PATH = build_edit_run(
            run_name=RUN_NAME, disabled_components=DISABLED_COST_COMPONENTS,
            vary_weights=VARY_WEIGHTS, route_time_weight=ROUTE_W,
            adj_weight=TRAIN_ADJ_WEIGHT, adj_target=ADJ_TARGET,
            adj_objective=ADJ_TRAIN_OBJECTIVE, adj_gap=ADJ_GAP, adj_mode=ADJ_MODE)
        cost_obj.median_connectivity_weight = float(CONN_W)
        if not BEST_MODEL_PATH.exists():
            raise FileNotFoundError(
                f"No trained checkpoint at {BEST_MODEL_PATH}. Train first, or check RUN_NAME.")
        model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
        model.eval()
        print(f"[eval] loaded trained model <- {BEST_MODEL_PATH.name} (training cell not run)")
    _required_eval_globals = (
        "TIERS", "EVAL_N_PER_TIER", "_val_by_tier", "BALANCED_EVAL_WEIGHTS",
        "graphs", "seed_routes", "model", "cost_obj", "device",
        "TARGET_N_ROUTES", "MIN_ROUTE_LEN", "MAX_ROUTE_LEN",
        "MAX_ROUTE_EDIT_STEPS", "MAX_TRIM_ACTIONS_PER_ROUTE",
        "ADJ_TARGET", "ADJ_WEIGHT", "ADJ_GAP", "ADJ_MODE",
        "make_improvement_batch", "rollout_lc_improvement",
        "get_batch_tensor_from_routes", "torch", "tqdm",
    )
    _missing_eval_globals = [name for name in _required_eval_globals if name not in globals()]
    if _missing_eval_globals:
        raise RuntimeError(
            "Balanced eval needs initialized config, dataset split, and model. "
            "Run the notebook cells from imports through training first. Missing: "
            + ", ".join(_missing_eval_globals)
        )

    def _redun_t(routes_2d):
        return _redundancy_stats(routes_2d)["redundancy"]


    def _mean_metric(result, key):
        value = result.get_metrics()[key]
        return float(value.detach().float().mean().item())


    val_by_tier = {
        tier: indices[:EVAL_N_PER_TIER]
        for tier, indices in _val_by_tier.items()
    }

    base_w = cost_obj.get_weights(device)
    def mkw(route_weight, conn_weight):
        weights = {key: (value.clone() if torch.is_tensor(value) else value)
                   for key, value in base_w.items()}
        weights["demand_time_weight"] = torch.as_tensor(0.0, device=device)
        weights["route_time_weight"] = torch.as_tensor(float(route_weight), device=device)
        weights["median_connectivity_weight"] = torch.as_tensor(float(conn_weight), device=device)
        return weights


    rows = []
    visual_examples = {}
    conn_metric_key = ("median_connectivity_weighted"
                       if cost_obj.use_weighted_connectivity
                       else "median_connectivity")
    transfer_metric_keys = {
        "d0": "$d_0$", "d1": "$d_1$", "d2": "$d_2$", "d_un": "$d_{un}$",
    }
    model.eval()
    weights = mkw(*BALANCED_EVAL_WEIGHTS)
    rollout_kwargs = _rollout_adjustment_kwargs_for_model(model)
    for tier in TIERS:
        idxs = val_by_tier[tier]
        if not idxs:
            continue
        metrics = {
            "redun_before": [], "redun_after": [],
            "ATT_before": [], "ATT_after": [],
            "RTT_before": [], "RTT_after": [],
            "CONN_before": [], "CONN_after": [],
            "d0_before": [], "d0_after": [],
            "d1_before": [], "d1_after": [],
            "d2_before": [], "d2_after": [],
            "d_un_before": [], "d_un_after": [],
        }
        for gi in tqdm(idxs, desc=f"eval balanced/{tier}", leave=False):
            graph_batch, route_batch = make_improvement_batch(
                graphs, seed_routes, torch.tensor([gi]), device,
                training=False, target_n_routes=TARGET_N_ROUTES)
            with torch.no_grad():
                output = rollout_lc_improvement(
                    model, cost_obj, graph_batch, route_batch,
                    MIN_ROUTE_LEN, MAX_ROUTE_LEN,
                    greedy=True, cost_weights=weights,
                    max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
                    max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE,
                    **rollout_kwargs)
            final_state, seed_result, final_result = output[:3]
            improved = get_batch_tensor_from_routes(
                final_state.routes, device, max_route_len=route_batch.shape[-1])
            metrics["redun_before"].append(_redun_t(route_batch[0]))
            metrics["redun_after"].append(_redun_t(improved[0]))
            metrics["ATT_before"].append(_mean_metric(seed_result, "ATT"))
            metrics["ATT_after"].append(_mean_metric(final_result, "ATT"))
            metrics["RTT_before"].append(_mean_metric(seed_result, "RTT"))
            metrics["RTT_after"].append(_mean_metric(final_result, "RTT"))
            metrics["CONN_before"].append(_mean_metric(seed_result, conn_metric_key))
            metrics["CONN_after"].append(_mean_metric(final_result, conn_metric_key))
            for metric_name, metric_key in transfer_metric_keys.items():
                metrics[f"{metric_name}_before"].append(_mean_metric(seed_result, metric_key))
                metrics[f"{metric_name}_after"].append(_mean_metric(final_result, metric_key))

            if tier not in visual_examples:
                nr = min(improved.shape[1], route_batch.shape[1])
                width = min(improved.shape[-1], route_batch.shape[-1])
                adj = get_adjustment_degrees(
                    improved[:, :nr, :width], route_batch[:, :nr, :width],
                    cost_obj.symmetric_routes, gap=ADJ_GAP, mode=ADJ_MODE
                ).mean().item()
                visual_examples[tier] = {
                    "graph_index": gi,
                    "seed": route_batch[0].detach().cpu(),
                    "improved": improved[0].detach().cpu(),
                    "Adj": adj,
                    "cost_before": float(seed_result.cost.detach().float().mean().item()),
                    "cost_after": float(final_result.cost.detach().float().mean().item()),
                    "redun_before": metrics["redun_before"][-1],
                    "redun_after": metrics["redun_after"][-1],
                    "ATT_before": metrics["ATT_before"][-1],
                    "ATT_after": metrics["ATT_after"][-1],
                    "RTT_before": metrics["RTT_before"][-1],
                    "RTT_after": metrics["RTT_after"][-1],
                    "CONN_before": metrics["CONN_before"][-1],
                    "CONN_after": metrics["CONN_after"][-1],
                }

        means = {key: float(np.mean(values)) for key, values in metrics.items()}
        rows.append({
            "tier": tier, "n": len(idxs),
            "redun_before": means["redun_before"],
            "redun_after": means["redun_after"],
            "ATT_before": means["ATT_before"], "ATT_after": means["ATT_after"],
            "RTT_before": means["RTT_before"], "RTT_after": means["RTT_after"],
            "CONN_before": means["CONN_before"], "CONN_after": means["CONN_after"],
            "d0_before": means["d0_before"], "d0_after": means["d0_after"],
            "d1_before": means["d1_before"], "d1_after": means["d1_after"],
            "d2_before": means["d2_before"], "d2_after": means["d2_after"],
            "d_un_before": means["d_un_before"], "d_un_after": means["d_un_after"],
        })

    eval_df = pd.DataFrame(rows).round(4)
    display(eval_df)
    save_table(eval_df, f"{RUN_NAME}_eval_by_tier")
    # Persist the per-tier visual examples (route tensors + metrics) so the viz
    # cell can run standalone and survive kernel loss on the server.
    _vx_path = MODEL_OUTPUTS_DIR / f"{RUN_NAME}_visual_examples.pt"
    torch.save(visual_examples, _vx_path)
    print(f"[eval] saved visual_examples -> {_vx_path}")
    print("ATT/RTT/CONN are minutes; d0/d1/d2/d_un are demand percentages by transfer bucket.")

## Visual validation examples

For the balanced preference vector, show one seed and the corresponding edited
network from every tier.  The right-hand panels emphasize removed and added
segments relative to the corrupted seed.

In [21]:
if RUN_TRAINING:
    if "visual_examples" not in globals() or not visual_examples:
        _vx_path = MODEL_OUTPUTS_DIR / f"{RUN_NAME}_visual_examples.pt"
        if _vx_path.exists():
            try:
                visual_examples = torch.load(_vx_path, map_location="cpu", weights_only=False)
            except TypeError:
                visual_examples = torch.load(_vx_path, map_location="cpu")
            print(f"[viz] loaded visual_examples <- {_vx_path.name}")
        else:
            visual_examples = {}
    if not visual_examples:
        print("Run the evaluation cell first.")
    else:
        tiers_to_plot = [tier for tier in TIERS if tier in visual_examples]
        fig, axes = plt.subplots(
            len(tiers_to_plot), 2,
            figsize=(18, 7 * len(tiers_to_plot)),
            squeeze=False, constrained_layout=True)
        for row_idx, tier in enumerate(tqdm(tiers_to_plot, desc="render tiers")):
            example = visual_examples[tier]
            graph = graphs[example["graph_index"]]
            route_plots.plot_plain_route_set(
                axes[row_idx, 0], example["seed"], graph,
                title=f"{tier}: corrupted seed (graph {example['graph_index']})",
                subtitle=(f"cost={example.get('cost_before', float('nan')):.3f}; "
                          f"redun={example['redun_before']:.3f}; "
                          f"ATT={example['ATT_before']:.2f}; RTT={example['RTT_before']:.2f}; "
                          f"CONN={example['CONN_before']:.2f}"))
            route_plots.plot_route_diff(
                axes[row_idx, 1], example["improved"], example["seed"], graph,
                title=f"{tier}: edited network vs seed",
                subtitle=(f"cost {example.get('cost_before', float('nan')):.3f}->{example.get('cost_after', float('nan')):.3f}; "
                          f"redun {example['redun_before']:.3f}->{example['redun_after']:.3f}; "
                          f"Adj={example['Adj']:.3f}\n"
                          f"ATT {example['ATT_before']:.2f}->{example['ATT_after']:.2f}; "
                          f"RTT {example['RTT_before']:.2f}->{example['RTT_after']:.2f}; "
                          f"CONN {example['CONN_before']:.2f}->{example['CONN_after']:.2f}"))
        fig.suptitle("Balanced validation: copy/subcopy corruption repair", fontsize=15, fontweight="bold")
        plt.show()
        plt.close(fig)

## Agent action GIFs

Run the greedy balanced policy step by step for one graph from every tier.
Each GIF starts from the corrupted seed and adds a frame after every agent
action (`extend`, `trim_start`, `trim_end`, or `halt`) with its reward and cost.

In [22]:
# GIF_OUTPUT_DIR = MODEL_OUTPUTS_DIR / "agent_action_gifs"
# GIF_FPS = 2
# GIF_FORCE_NONHALT_FIRST_STEP = False  # match the balanced eval rollout
# ACTION_NAMES = {
#     ROUTE_ACTION_EXTEND: "extend", ROUTE_ACTION_TRIM_START: "trim_start",
#     ROUTE_ACTION_TRIM_END: "trim_end", ROUTE_ACTION_HALT: "halt",
# }


# def _is_trim_action(action_kind):
#     return int(action_kind) in (ROUTE_ACTION_TRIM_START, ROUTE_ACTION_TRIM_END)


# def _action_text(action_kind, action):
#     action_kind = int(action_kind)
#     name = ACTION_NAMES[action_kind]
#     if action_kind == ROUTE_ACTION_HALT:
#         return name
#     if _is_trim_action(action_kind):
#         return f"{name}(position={int(action[0])})"
#     return f"{name}({int(action[0])} -> {int(action[1])})"


# def _put_route(routes, route_idx, route):
#     routes = routes.clone()
#     routes[:, route_idx] = -1
#     route = route[route > -1].to(routes.device)
#     if route.numel():
#         route_tensor = get_batch_tensor_from_routes(
#             [[route]], routes.device, max_route_len=routes.shape[-1])
#         routes[:, route_idx] = route_tensor[:, 0]
#     return routes


# def _collect_agent_action_frames(graph_index):
#     graph_batch, route_batch = make_improvement_batch(
#         graphs, seed_routes, torch.tensor([graph_index]), device,
#         training=False, target_n_routes=TARGET_N_ROUTES)
#     weights = mkw(*BALANCED_EVAL_WEIGHTS)
#     reward_scale = float(getattr(cfg, "reward_scale", 1.0))
#     diff_reward = bool(getattr(cfg, "diff_reward", True))
#     incumbent_reward = bool(getattr(cfg, "incumbent_reward", False))
#     zero_trim_reward = bool(getattr(cfg, "zero_trim_reward", False))
#     positive_only_trim_reward = bool(getattr(cfg, "positive_only_trim_reward", False))
#     edit_step_penalty = float(getattr(cfg, "edit_step_penalty", 0.0))
#     forced_halt_penalty = float(getattr(cfg, "forced_halt_penalty", 0.0))
#     context_len = max(
#         int(route_batch.shape[-1]), int(MAX_ROUTE_LEN),
#         int(graph_batch[STOP_KEY].num_nodes))
#     working_routes = torch.full(
#         (1, route_batch.shape[1], context_len), -1,
#         dtype=route_batch.dtype, device=device)
#     working_routes[..., :route_batch.shape[-1]] = route_batch
#     display_routes = working_routes.clone()
#     frames = [{"routes": display_routes[0].detach().cpu(), "label": "corrupted seed"}]
#     rows = []

#     model.eval()
#     with torch.no_grad():
#         for route_idx in range(route_batch.shape[1]):
#             context_routes = working_routes.clone()
#             context_routes[:, route_idx] = -1
#             route_state = _make_route_context_state(
#                 cost_obj, graph_batch, working_routes, route_idx,
#                 MIN_ROUTE_LEN, MAX_ROUTE_LEN, weights,
#                 invalid_directly_connected=not bool((context_routes >= 0).any().item()))
#             context_counts = route_state.n_finished_routes.detach().clone()
#             route_state = model.setup_planning(route_state)
#             prev_cost = cost_obj(route_state).cost.detach().clone()
#             trim_count = 0

#             for step_idx in range(MAX_ROUTE_EDIT_STEPS + 1):
#                 if route_state.is_done().all():
#                     break
#                 force_halt = step_idx >= MAX_ROUTE_EDIT_STEPS
#                 trim_allowed = (MAX_TRIM_ACTIONS_PER_ROUTE is None or
#                                 trim_count < int(MAX_TRIM_ACTIONS_PER_ROUTE))
#                 if force_halt:
#                     step_kinds = torch.full((1,), ROUTE_ACTION_HALT, dtype=torch.long, device=device)
#                     step_actions = torch.full((1, 2), -1, dtype=torch.long, device=device)
#                 else:
#                     step_kinds, step_actions, _, _ = model.step_route_action(
#                         route_state, greedy=True,
#                         allow_halt=not (GIF_FORCE_NONHALT_FIRST_STEP and step_idx == 0),
#                         allow_trim_start=trim_allowed, allow_trim_end=trim_allowed)

#                 cost_before = float(prev_cost.cpu()[0])
#                 action_kind = int(step_kinds[0].item())
#                 is_trim = _is_trim_action(action_kind)
#                 route_state.apply_route_actions(step_kinds, step_actions)
#                 planned_route = _get_planned_current_routes(
#                     route_state, working_routes[:, route_idx], context_counts)[0]
#                 frame_routes = _put_route(display_routes, route_idx, planned_route)
#                 new_cost = cost_obj(route_state).cost.detach().clone()
#                 if incumbent_reward:
#                     step_reward = torch.clamp_min(prev_cost - new_cost, 0) * reward_scale
#                 elif diff_reward:
#                     step_reward = (prev_cost - new_cost) * reward_scale
#                 else:
#                     step_reward = torch.zeros_like(new_cost)
#                     if action_kind == ROUTE_ACTION_HALT:
#                         step_reward = -new_cost * reward_scale
#                 if positive_only_trim_reward and is_trim:
#                     step_reward = step_reward.clamp_min(0)
#                 elif zero_trim_reward and is_trim:
#                     step_reward = torch.zeros_like(step_reward)
#                 if action_kind != ROUTE_ACTION_HALT:
#                     step_reward = step_reward - edit_step_penalty
#                 if force_halt:
#                     step_reward = step_reward - forced_halt_penalty
#                 if not (zero_trim_reward and not positive_only_trim_reward and is_trim):
#                     prev_cost = new_cost
#                 trim_count += int(is_trim)

#                 label = (_action_text(action_kind, step_actions[0]) +
#                          f" | reward={float(step_reward.cpu()[0]):+.4f}" +
#                          f" | cost={float(new_cost.cpu()[0]):.4f}")
#                 frames.append({"routes": frame_routes[0].detach().cpu(), "label": label})
#                 rows.append({
#                     "route": route_idx, "step": step_idx + 1,
#                     "action": _action_text(action_kind, step_actions[0]),
#                     "action_kind": ACTION_NAMES[action_kind],
#                     "cost_before": cost_before, "cost_after": float(new_cost.cpu()[0]),
#                     "reward": float(step_reward.cpu()[0]), "forced_halt": force_halt,
#                 })
#                 if action_kind == ROUTE_ACTION_HALT:
#                     break

#             final_route = _get_planned_current_routes(
#                 route_state, working_routes[:, route_idx], context_counts)[0]
#             display_routes = _put_route(display_routes, route_idx, final_route)
#             working_routes = _put_route(working_routes, route_idx, final_route)
#     return frames, pd.DataFrame(rows)


# def save_agent_action_gif(tier, example):
#     frames, steps_df = _collect_agent_action_frames(example["graph_index"])
#     graph = graphs[example["graph_index"]]
#     fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)

#     def _draw(frame_idx):
#         ax.clear()
#         frame = frames[frame_idx]
#         route_plots.plot_route_diff(
#             ax, frame["routes"], example["seed"], graph,
#             title=f"{tier}: greedy balanced actions (graph {example['graph_index']})",
#             subtitle=f"frame {frame_idx + 1}/{len(frames)} | {frame['label']}")
#         return []

#     GIF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#     gif_path = GIF_OUTPUT_DIR / f"{RUN_NAME}_{tier}_agent_actions.gif"
#     animation = FuncAnimation(
#         fig, _draw, frames=len(frames), interval=1000 / GIF_FPS, blit=False)
#     try:
#         animation.save(gif_path, writer=PillowWriter(fps=GIF_FPS))
#         plt.close(fig)
#         display(Image(filename=str(gif_path)))
#     except Exception as exc:
#         plt.close(fig)
#         print(f"{tier}: could not save GIF with PillowWriter: {exc}")
#         fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
#         _draw(len(frames) - 1)
#         plt.show(); plt.close(fig)
#         gif_path = None
#     return gif_path, steps_df


# if not visual_examples:
#     print("Run the evaluation cell first.")
# else:
#     saved_gifs, gif_steps_by_tier = {}, {}
#     for tier in tqdm([tier for tier in TIERS if tier in visual_examples], desc="render GIFs"):
#         gif_path, steps_df = save_agent_action_gif(tier, visual_examples[tier])
#         saved_gifs[tier] = gif_path
#         gif_steps_by_tier[tier] = steps_df
#         save_table(steps_df, f"{RUN_NAME}_{tier}_agent_action_steps")
#         print(f"{tier}: {len(steps_df) + 1} frames -> {gif_path}")
#         display(steps_df.groupby("action_kind")["reward"].agg(["count", "sum", "mean"]).round(4))

## Post-training convergence: RPC + type2 vs RPC + trim/extend (Mumford1, 100 iters)

Right after PART 1 training, run two of the 5-model BCO variants -- random
path-combiner (RPC) rebuild paired with the hand-designed `type2` edit vs the
freshly trained `trim/extend` edit bee -- on Mumford1 for 100 BCO iterations
(balanced alpha=0.5, adjustment penalty off) and plot the convergence history.

In [23]:
if RUN_TRAINING:
    import matplotlib.pyplot as plt
    from omegaconf import OmegaConf
    import eval_lib.helpers as _eh

    POST_TRAIN_CITY  = "Mumford1"
    POST_TRAIN_ITERS = 100
    POST_TRAIN_ALPHA = 0.5   # balanced RTT/WMC; adjustment penalty OFF (as in the 5-model run)

    # Drive the extend/trim edit bee with the checkpoint we just trained.
    _eh.EDIT_MODEL_WEIGHTS_PATH     = BEST_MODEL_PATH
    _eh.EDIT_MODEL_N_ADJ_COND_FEATS = (1 if CONDITION_ON_ADJ_TARGET else 0)
    print(f"[post-train] edit bee model <- {BEST_MODEL_PATH.name} "
          f"(adj_cond_feats={_eh.EDIT_MODEL_N_ADJ_COND_FEATS})")

    _spec = next(s for s in BENCHMARK_SPECS if s["city"] == POST_TRAIN_CITY)
    _ptl_tensors, _ptl_init = load_benchmark_graph(_spec)

    # Two of the 5-model variants: RPC (random path-combiner) rebuild auto-fills
    # the remaining bee slots; the edit slots are either type2 or trim/extend.
    POST_TRAIN_MODELS = [
        dict(label="RPC + trim/extend", n_type2=0, n_type5=5),
        dict(label="RPC + type2",       n_type2=5, n_type5=0),
    ]
    _ptl_conv, _ptl_rows = {}, []
    for _m in POST_TRAIN_MODELS:
        _cfg = build_bco_cfg(
            run_name=f"posttrain_{POST_TRAIN_CITY}_{_m['label']}".replace(" ", "_").replace("/", "_"),
            n_routes=_spec["n_routes"], min_route_len=_spec["min_route_len"],
            max_route_len=_spec["max_route_len"],
            use_neural_bees=False, n_bees=10,
            n_type1_bees=0, n_type2_bees=_m["n_type2"], n_type5_bees=_m["n_type5"],
            route_time_weight=POST_TRAIN_ALPHA,
            median_connectivity_weight=1.0 - POST_TRAIN_ALPHA,
            connectivity_mode=CONNECTIVITY_MODE)
        OmegaConf.update(_cfg, "n_iterations", int(POST_TRAIN_ITERS), force_add=True)
        _hist_out = {}
        _t0 = _time.perf_counter()
        run_bco(_cfg, _ptl_init, tensors=_ptl_tensors,
                run_name_scope=f"{POST_TRAIN_CITY}_", cost_history_out=_hist_out)
        _dt = _time.perf_counter() - _t0
        _h = _hist_out.get("history")
        _y = (np.asarray(_h.numpy() if hasattr(_h, "numpy") else _h).reshape(-1)
              if _h is not None else None)
        if _y is not None and _y.size:
            _ptl_conv[_m["label"]] = _y
            _final, _best = float(_y[-1]), float(np.min(_y))
        else:
            _final = _best = float("nan")
        _ptl_rows.append(dict(model=_m["label"], final_cost=round(_final, 4),
                              best_cost=round(_best, 4),
                              iters=int(POST_TRAIN_ITERS), seconds=round(_dt, 1)))
        print(f"[post-train] {_m['label']:18} final={_final:.4f} best={_best:.4f} ({_dt:.0f}s)")

    display(pd.DataFrame(_ptl_rows))

    if _ptl_conv:
        fig, ax = plt.subplots(figsize=(8, 5))
        for _lab, _y in _ptl_conv.items():
            ax.plot(range(1, len(_y) + 1), _y, marker="o", ms=2, label=_lab)
        ax.set_xlabel("BCO iteration")
        ax.set_ylabel("best objective cost (lower = better)")
        ax.set_title(f"Post-training convergence -- {POST_TRAIN_CITY} "
                     f"(alpha={POST_TRAIN_ALPHA}, adj off, {POST_TRAIN_ITERS} iters)")
        ax.grid(alpha=0.3); ax.legend()
        plt.show(); plt.close(fig)
        _conv_df = pd.DataFrame({k: pd.Series(v) for k, v in _ptl_conv.items()})
        _conv_df.index.name = "bco_iteration"
        save_table(_conv_df.reset_index(), f"posttrain_convergence_{POST_TRAIN_CITY}")
    else:
        print("[post-train] no convergence history captured")

---
# PART 2 - BCO experiments (E1-E6)

Uses the model trained above (`BEST_MODEL_PATH`). All baselines are re-run live on a single seed with small budgets.

## Configuration

`SMOKE=True` -> Mandl only, 1 seed, tiny iteration budgets (just checks the
pipeline builds tables/figures). `SMOKE=False` runs Mandl and Mumford0-3
with the 25-30 hour budgeted settings below.


In [24]:
SMOKE = False
QUICK = False  # minimal budgets/sweeps just to verify the notebook runs end-to-end
CITIES = ["Mandl"] if SMOKE else ["Mandl", "Mumford0", "Mumford1", "Mumford2", "Mumford3"]
SEEDS = [0]                # single seed everywhere
# Unified objective parameters -- SINGLE SOURCE: eval_lib/params.py (also used
# by the PART 1 training config above).
from eval_lib.params import (CONNECTIVITY_MODE, DISABLED_COST_COMPONENTS,
                             UNIFIED_COST_WEIGHTS, ADJ_WEIGHT, ADJ_TARGET,
                             ADJ_OBJECTIVE, ADJ_GAP, ADJ_MODE)
# E1u (unified objective: 0.5*RTT + 0.5*WMC + adj for ALL methods) runs live.
E1U_FULL_CITIES = ["Mandl", "Mumford0", "Mumford1", "Mumford2", "Mumford3"]
E1U_TRIM12_ONLY_CITIES = []
E1U_CITIES = []  # E1 narrow off: training-only run
E1U_TABLE_SUFFIX = "_nbco_only_target03_iter200"
# E1 narrow run selector (Initial is always the reference). Here: neural
# BCO only -- Our NBCO is computed separately in its own file.
E1U_RUN_NEURAL_BCO = False
E1U_RUN_OUR_MODEL = False
E2_TABLE_SUFFIX = "_mumford1_5model"
# E1 narrow full run: only neural BCO and Our NBCO, alpha sweep with target 0.5.
E1U_ALPHA_GRID = [0.0, 0.5, 1.0]
E1U_ADJ_TARGET = 0.3
E1U_BCO_ITERATIONS = 200
# Variant A -- reuse a previously-saved init network instead of regenerating
# it via the (machine-dependent) LC GNN. When non-empty, load_benchmark_graph
# loads init from final_main_unified_<city><E1_INIT_SOURCE_STEM>_routes.pt
# (the 'Initial ...' entry); empty string or missing file -> regenerate via LC.
E1_INIT_SOURCE_STEM = "_our_only_target03_iter500"
RUN_E2_OUR_PARETO = False
RUN_E2_5MODEL = False        # EKB-only run: other experiments off
RUN_E2_ROUTE_VIZ = False
RUN_M0_NBCO_VARIANTS = False
RUN_MACSA_EXPERIMENTS = False
# NSGA-II is the heaviest baseline. When False, skip live NSGA-II runs and
# drop any precomputed NSGA-II rows from the comparison CSVs.
RUN_NSGAII_BASELINES = False
# E1u reruns only our model by default. Baseline algorithms are
# reused from final_main_unified_<city> CSV/route dumps unless
# their cache is missing or E1U_RERUN_BASELINES=True.
E1U_REUSE_BASELINE_CACHE = True
E1U_RERUN_BASELINES = False
E1U_RERUN_OUR_MODEL = True

# Edit model used by the experiments (training is OFF here).
OUR_MODEL_PATH = EDIT_MODEL_WEIGHTS_DIR / "improvement_lc_rttconn_adj_w10_t02_finetune100.pt"

# BCO edit/trim bees also load the edit model via eval_lib's default path.
import eval_lib.helpers as _eh
_eh.EDIT_MODEL_WEIGHTS_PATH = OUR_MODEL_PATH
# The finetune100 checkpoint is unconditioned.
_eh.EDIT_MODEL_N_ADJ_COND_FEATS = 0
# Experiment init for every city = LC construction + covered_dup tier (overridden
# to the LC generator in the helpers cell below); BENCHMARK_INIT_MODE is unused.
import eval_lib.baselines as _eb
_eb.BENCHMARK_INIT_MODE = "nx"
if QUICK:
    _eh.BCO_N_ITERATIONS = 5   # tiny BCO budget for quick sweeps

# Version B of our NBCO (edit model as the REBUILD bee) makes the trim model run a
# full rollout as a constructor -> ~250x slower per BCO iteration (258 s/it vs 1 s/it
# on Mandl, effectively hangs on Mumford). OFF by default; opt-in for Mandl-only probes.
INCLUDE_EDIT_REBUILD = False

# Early stopping OFF: every method runs its full iteration budget so the
# convergence curves below are complete (no premature termination).
EARLY_STOP_PATIENCE = None   # no early stopping (full iteration budget)

# CPU-budgeted full profiles. BCO uses the paper-scale population size
# (10 bees), with reduced-but-not-tiny iteration counts. E2 uses a separate
# BCO budget because the Pareto sweep multiplies runs by grids x models.
DEFAULT_ALGO_SETTINGS = (
    dict(sa=120, hh=120, ga=5, ga_pop=6, nsgaii=(8, 6),
         bco=8, bco_bees=10, sa_args={}, bco_args={})
    if SMOKE else
    dict(sa=15000, hh=15000, ga=500, ga_pop=20, nsgaii=(800, 100),
         bco=500, bco_bees=10, sa_args={}, bco_args={})
)

MANDL_ALGO_SETTINGS = dict(
    sa=(120 if SMOKE else 15000),
    hh=(120 if SMOKE else 15000),
    ga=(5 if SMOKE else 500),
    ga_pop=(6 if SMOKE else 20),
    nsgaii=((8, 6) if SMOKE else (800, 100)),
    bco=(8 if SMOKE else 500),
    bco_bees=10,
    sa_args=dict(initial_temp=0.08, final_temp=0.003, cooling_rate=6.5e-5,
                 reheating_threshold=300, reheating_factor=1.02),
    bco_args=dict(worse_accept_temperature=0.02, worse_accept_decay=0.985,
                  worse_accept_min_temperature=0.001,
                  worse_selection_temperature=0.02,
                  worse_selection_decay=0.985,
                  worse_selection_uniform_mix=0.10,
                  worse_selection_elite_count=2),
)

# Larger Mumford graphs need more search than the Mandl/Mumford0-1 settings.
# E1 currently runs the full baseline suite on Mumford2/3, so scale SA/HH/GA
# there instead of globally inflating the smaller-city probes.
SA_ITERS_BY_CITY = {"Mandl": 15000, "Mumford0": 25000, "Mumford1": 40000,
                    "Mumford2": 120000, "Mumford3": 180000}
HH_ITERS_BY_CITY = {"Mandl": 15000, "Mumford0": 25000, "Mumford1": 50000,
                    "Mumford2": 90000, "Mumford3": 130000}
GA_ITERS_BY_CITY = {"Mandl": 500, "Mumford0": 700, "Mumford1": 900,
                    "Mumford2": 1400, "Mumford3": 1800}
GA_POP_BY_CITY = {"Mandl": 20, "Mumford0": 24, "Mumford1": 32,
                  "Mumford2": 48, "Mumford3": 64}
# HH random-walk init-repair can need a much longer cap on larger route sets.
HH_MAX_REPAIR_ITERS = 5000
HH_MAX_REPAIR_ITERS_BY_CITY = {"Mandl": 5000, "Mumford0": 10000,
                               "Mumford1": 30000, "Mumford2": 80000,
                               "Mumford3": 120000}

# E2 is an expensive sweep (alpha x target x models); keep iterations bounded.
E2_BCO_ITERATIONS = 200


def algo_settings(city):
    settings = dict(DEFAULT_ALGO_SETTINGS)
    if city == "Mandl":
        settings.update(MANDL_ALGO_SETTINGS)
    if not (SMOKE or QUICK):
        settings["sa"] = SA_ITERS_BY_CITY.get(city, settings["sa"])
        settings["hh"] = HH_ITERS_BY_CITY.get(city, settings["hh"])
        settings["ga"] = GA_ITERS_BY_CITY.get(city, settings["ga"])
        settings["ga_pop"] = GA_POP_BY_CITY.get(city, settings["ga_pop"])
        settings["hh_max_repair_iters"] = HH_MAX_REPAIR_ITERS_BY_CITY.get(
            city, HH_MAX_REPAIR_ITERS)
    settings["bco_args"] = dict(DEFAULT_ALGO_SETTINGS.get("bco_args", {}),
                                 **settings.get("bco_args", {}))
    if QUICK:
        settings.update(sa=40, hh=40, ga=5, ga_pop=6, nsgaii=(10, 8),
                        bco=8, bco_bees=10)
    # SA temperature schedule: a LINEAR anneal spanning the FULL (city-specific)
    # iteration budget + periodic reheating, applied to ALL cities. Previously
    # only Mandl was tuned; Mumford used the yaml defaults with NO reheating and
    # a budget-agnostic cooling_rate, so SA could stay stuck at the seed.
    # cost_norm in the SA loop normalizes acceptance by the initial cost, so one
    # schedule transfers across cities (initial cost ~2-3 for every benchmark).
    _sa_n = int(settings["sa"])
    if city in {"Mumford2", "Mumford3"} and not (SMOKE or QUICK):
        _t0, _t1 = 0.22, 0.004
        _reheat_threshold, _reheat_factor = max(1000, _sa_n // 25), 1.8
    else:
        _t0, _t1 = 0.15, 0.005
        _reheat_threshold, _reheat_factor = max(500, _sa_n // 20), 1.5
    settings["sa_args"] = dict(
        schedule="linear", initial_temp=_t0, final_temp=_t1,
        cooling_rate=(_t0 - _t1) / max(_sa_n, 1),
        reheating_threshold=_reheat_threshold,
        reheating_factor=_reheat_factor)
    return settings


print("cities:", CITIES)
print("our model:", OUR_MODEL_PATH.name, "| exists:", OUR_MODEL_PATH.exists())
print("connectivity_mode:", CONNECTIVITY_MODE)
print("disabled cost components:", DISABLED_COST_COMPONENTS)
print("run NSGA-II baselines:", RUN_NSGAII_BASELINES)
print("E1u narrow cities:", E1U_CITIES,
      "| methods: neural BCO + Our NBCO",
      "| alphas:", E1U_ALPHA_GRID, "| target:", E1U_ADJ_TARGET,
      "| bco_iter:", E1U_BCO_ITERATIONS,
      "| suffix:", E1U_TABLE_SUFFIX)
print("E1u large-graph algo settings:", {
    c: {k: algo_settings(c).get(k) for k in ("sa", "hh", "ga", "ga_pop", "hh_max_repair_iters")}
    for c in E1U_FULL_CITIES})
print("enabled experiments:", {"E1_narrow": bool(E1U_CITIES),
                              "MACSA": RUN_MACSA_EXPERIMENTS,
                              "E2_our_pareto": RUN_E2_OUR_PARETO,
                              "E2_5model": RUN_E2_5MODEL,
                              "M0_variants": RUN_M0_NBCO_VARIANTS})
print("unified weights (RTT+WMC):", UNIFIED_COST_WEIGHTS,
      "| adj:", f"{ADJ_WEIGHT}*|adj-{ADJ_TARGET}| ({ADJ_OBJECTIVE})")
print("E2 BCO iterations:", E2_BCO_ITERATIONS)



cities: ['Mandl', 'Mumford0', 'Mumford1', 'Mumford2', 'Mumford3']
our model: improvement_lc_rttconn_adj_w10_t02_finetune100.pt | exists: True
connectivity_mode: median_weighted
disabled cost components: ['demand']
run NSGA-II baselines: False
E1u narrow cities: [] | methods: neural BCO + Our NBCO | alphas: [0.0, 0.5, 1.0] | target: 0.3 | bco_iter: 200 | suffix: _nbco_only_target03_iter200
E1u large-graph algo settings: {'Mandl': {'sa': 15000, 'hh': 15000, 'ga': 500, 'ga_pop': 20, 'hh_max_repair_iters': 5000}, 'Mumford0': {'sa': 25000, 'hh': 25000, 'ga': 700, 'ga_pop': 24, 'hh_max_repair_iters': 10000}, 'Mumford1': {'sa': 40000, 'hh': 50000, 'ga': 900, 'ga_pop': 32, 'hh_max_repair_iters': 30000}, 'Mumford2': {'sa': 120000, 'hh': 90000, 'ga': 1400, 'ga_pop': 48, 'hh_max_repair_iters': 80000}, 'Mumford3': {'sa': 180000, 'hh': 130000, 'ga': 1800, 'ga_pop': 64, 'hh_max_repair_iters': 120000}}
enabled experiments: {'E1_narrow': False, 'MACSA': False, 'E2_our_pareto': False, 'E2_5model': Fals

## Shared metrics: adjustment-degree vs seed + redundancy

In [25]:
from omegaconf import OmegaConf
from tqdm.auto import tqdm
import functools as _functools

# Generic plumbing lives in eval_lib (paper.py / route_copies.py); the mechanical
# cfg-builders live in eval_lib.experiments. This cell keeps only the experiment
# design: per-city budgets, BCO variants, LC init.
from eval_lib.paper import (UNIFIED_ADJ, PAPER_DIR, bco_cfg_set,
                            set_cfg_value as _set_cfg_value,
                            unify_weights as _unify_weights,
                            eval_routes_cfg as _eval_routes_cfg,
                            ravel_hist as _ravel_hist,
                            paper_row as _row, full_metrics as _full_metrics,
                            adj_vs_init, redundancy_pct, conn_metric,
                            save_paper_table, reset_paper_table,
                            append_paper_row, save_paper_fig, save_paper_routes)
from eval_lib.route_copies import (inject_realistic_tier, pad_routes,
                                   uncovered_demand_pct)
import eval_lib.experiments as _experiments
from eval_lib.experiments import ExperimentContext, abl_variant, our_nbco_versions

# LC init optimizes the same unified objective (RTT + WMC, demand off).
LC_INIT_WEIGHTS = UNIFIED_COST_WEIGHTS

# Bind the cfg-builders to this notebook's runtime context so the experiment
# cells keep calling the short _sa_cfg / _variant_bco_cfg / _our_model_cfg names.
_EXP_CTX = ExperimentContext(
    algo_settings=algo_settings, connectivity_mode=CONNECTIVITY_MODE,
    unified_weights=UNIFIED_COST_WEIGHTS, unified_adj=UNIFIED_ADJ,
    early_stop_patience=EARLY_STOP_PATIENCE, hh_max_repair_iters=HH_MAX_REPAIR_ITERS,
    our_model_path=OUR_MODEL_PATH, bco_n_type1_bees=BCO_N_TYPE1_BEES,
    include_edit_rebuild=INCLUDE_EDIT_REBUILD)
_sa_cfg = _functools.partial(_experiments.sa_cfg, _EXP_CTX)
_ga_cfg = _functools.partial(_experiments.ga_cfg, _EXP_CTX)
_hh_cfg = _functools.partial(_experiments.hh_cfg, _EXP_CTX)
_variant_bco_cfg = _functools.partial(_experiments.variant_bco_cfg, _EXP_CTX)
_our_model_cfg = _functools.partial(_experiments.our_model_cfg, _EXP_CTX)
OUR_NBCO_VERSIONS = our_nbco_versions(INCLUDE_EDIT_REBUILD)


# --- experiment init: LC construction base + the REALISTIC corruption tier
#     (eval_lib.route_copies.REALISTIC_TIER_CFG): a couple of street-valid
#     partial duplicates, 1-2 detour routes the agent should straighten, and a
#     few dropped low-demand stops -- instead of the old covered_dup tier that
#     cloned half the network and glued phantom (non-street) legs. ---
import random as _random
LC_INIT_SEED = 0
EXP_INIT_TIER = "realistic"   # tier injected into the experiment init network
_LC_INIT_CACHE = {}


def _lc_base_routes(spec):
    """Clean LC construction routes for a city (no tier corruption)."""
    tensors = load_benchmark_tensors(spec["city"])
    cfg = build_lc_cfg(run_name=f"lc_init_{spec['city']}", n_routes=spec["n_routes"],
                       min_route_len=spec["min_route_len"],
                       max_route_len=spec["max_route_len"],
                       connectivity_mode=CONNECTIVITY_MODE, **LC_INIT_WEIGHTS)
    _set_cfg_value(cfg, "experiment.seed", int(LC_INIT_SEED))
    routes = run_lc(cfg, tensors=tensors, run_name_prefix=f"lc_init_{spec['city']}_", n_samples=1)[3]
    return tensors, pad_routes(routes, spec["n_routes"], spec["max_route_len"])


def _e1_init_from_file(spec):
    """Variant A: reuse a saved init network for a city -- load the
    'Initial ...' routes from
    final_main_unified_<city><E1_INIT_SOURCE_STEM>_routes.pt so init is
    machine-independent and identical across methods. Returns a padded
    (1, n_routes, max_len) tensor, or None to fall back to LC generation."""
    stem = globals().get("E1_INIT_SOURCE_STEM", "")
    if not stem:
        return None
    path = PAPER_DIR / f"final_main_unified_{spec['city']}{stem}_routes.pt"
    if not path.exists():
        return None
    try:
        dump = torch.load(path, map_location="cpu", weights_only=False)
        routes = dump.get("routes", {})
        key = next((k for k in routes if str(k).startswith("Initial")), None)
        if key is None:
            return None
        return pad_routes(routes[key], spec["n_routes"], spec["max_route_len"])[None]
    except Exception as exc:
        print(f"[init] {spec['city']}: file load failed ({exc}); regenerating via LC")
        return None

def load_benchmark_graph(spec, init_mode=None):     # overrides eval_lib.load_benchmark_graph
    city = spec["city"]
    if city not in _LC_INIT_CACHE:
        _file_init = _e1_init_from_file(spec)
        if _file_init is not None:
            tensors = load_benchmark_tensors(city)
            init = as_route_tensor(_file_init)
            if init.ndim == 2:
                init = init[None]
            _LC_INIT_CACHE[city] = (tensors, init)
            _dun = uncovered_demand_pct(init[0], tensors["demand"],
                                        tensors["node_locs"].shape[0])
            print(f"[init] {city}: loaded from saved "
                  f"final_main_unified_{city}{globals().get('E1_INIT_SOURCE_STEM',)} dump "
                  f"-> redun={redundancy_pct(init):.1f}% d_un={_dun:.1f}%")
        else:
            tensors, clean = _lc_base_routes(spec)
            rng = _random.Random(LC_INIT_SEED)
            init, _applied = inject_realistic_tier(
                clean, rng, spec["min_route_len"], spec["max_route_len"],
                street_adj=tensors["street_adj"], demand=tensors["demand"],
                n_nodes=tensors["node_locs"].shape[0])
            init = as_route_tensor(init)
            if init.ndim == 2:
                init = init[None]                      # -> (1, n_routes, max_len) for runners
            _LC_INIT_CACHE[city] = (tensors, init)
            _dun = uncovered_demand_pct(init[0], tensors["demand"],
                                        tensors["node_locs"].shape[0])
            print(f"[init] {city}: LC base + tier '{EXP_INIT_TIER}' "
                  f"({dict(_applied)}) -> redun={redundancy_pct(init):.1f}% "
                  f"d_un={_dun:.1f}%")
    return _LC_INIT_CACHE[city]


# 2x2 ablation BCO variant (paper-style: GNN constructor swapped for RPC).
#   construct: "GNN" = neural route-construction rebuild (type-1, use_neural_bees)
#             "RPC" = random path-combiner rebuild (type-3, path_mix_rebuild)
#   edit:     "trim/extend" = our edit-model bee (type-5)
#             "type2"       = old local-endpoint edit (type-2)
ABL_MODELS = [
    "GNN + trim/extend", "RPC + trim/extend",
    "GNN + type2", "RPC + type2", "trim 12 + extend 12",
]


# Unified objective (RTT + WMC + two-sided |adj-target|) shared by E1u and MACSA.




## EKB case study

Load the Ekaterinburg instance, inspect its projected coordinates, and render the supplied seed routes as a regular matplotlib figure.


In [26]:
# EKB case study: load supplied routes and render them as a regular figure.
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from eval_lib import plots as route_plots
from eval_lib.ekb import (EKB_COORD_CRS, EKB_STATIC_MAP_PATH,
                          ekb_connectivity_stats, ekb_route_stats,
                          ekb_spec, load_ekb_routes, load_ekb_tensors,
                          make_ekb_crop_case, make_ekb_street_underlay_adj,
                          project_ekb_coords)

RUN_EKB_CASE_STUDY = False
EKB_FORCE_CPU = True
EKB_SCORE_SEED = False
USE_EKB_CROP = False        # full EKB network (no geographic crop)
EKB_CROP_TARGET_NODES = 300
EKB_CROP_MIN_ROUTE_LEN = 11
EKB_CROP_KEEP_LARGEST_COMPONENT = True
EKB_ROUTE_FIG_NODE_SIZE = 8

if RUN_EKB_CASE_STUDY:
    EKB_FULL_TENSORS = load_ekb_tensors()
    EKB_FULL_INIT = load_ekb_routes()
    if USE_EKB_CROP:
        EKB_TENSORS, EKB_INIT, EKB_CROP_META = make_ekb_crop_case(
            EKB_FULL_TENSORS, EKB_FULL_INIT,
            target_nodes=EKB_CROP_TARGET_NODES,
            min_route_len=EKB_CROP_MIN_ROUTE_LEN,
            keep_largest_component=EKB_CROP_KEEP_LARGEST_COMPONENT)
        EKB_CASE_TAG = f"crop{EKB_CROP_META['n_nodes']}"
    else:
        EKB_TENSORS = EKB_FULL_TENSORS
        EKB_INIT = EKB_FULL_INIT
        EKB_CROP_META = {"crop_enabled": False}
        EKB_CASE_TAG = "full"
    EKB_SPEC = ekb_spec(EKB_INIT)
    EKB_STATS = ekb_route_stats(EKB_INIT)
    _ekb_latlon = project_ekb_coords(EKB_TENSORS["node_locs"], EKB_COORD_CRS)
    print("EKB coords:", EKB_TENSORS["node_locs"].shape,
          "CRS", EKB_COORD_CRS,
          "lat", (round(float(_ekb_latlon[:, 0].min()), 4), round(float(_ekb_latlon[:, 0].max()), 4)),
          "lon", (round(float(_ekb_latlon[:, 1].min()), 4), round(float(_ekb_latlon[:, 1].max()), 4)))
    EKB_CONNECTIVITY = ekb_connectivity_stats(EKB_TENSORS, EKB_INIT)
    print("EKB spec:", EKB_SPEC, "route stats:", EKB_STATS)
    print("EKB connectivity:", {
        "components": EKB_CONNECTIVITY["n_components"],
        "component_sizes": EKB_CONNECTIVITY["component_sizes"][:5],
        "isolated_nodes": EKB_CONNECTIVITY["isolated_nodes"],
        "symmetric": EKB_CONNECTIVITY["symmetric"],
        "cross_component_demand_pct": round(EKB_CONNECTIVITY["cross_component_demand_pct"], 4),
        "route_nodes_outside_giant": EKB_CONNECTIVITY["route_nodes_outside_giant"],
        "self_loop_legs": len(EKB_CONNECTIVITY["route_self_loops"]),
    })
    if USE_EKB_CROP:
        print("EKB crop:", {
            "case": EKB_CASE_TAG,
            "nodes": f"{EKB_CROP_META['n_nodes']}/{EKB_CROP_META['n_nodes_full']}",
            "routes": f"{EKB_CROP_META['n_routes']}/{EKB_CROP_META['n_routes_full']}",
            "radius_km": round(EKB_CROP_META['radius_m'] / 1000.0, 3),
            "keep_largest_component": EKB_CROP_META['keep_largest_component'],
        })
    print("EKB force CPU:", EKB_FORCE_CPU)

    EKB_SEED_ROW = None
    if EKB_SCORE_SEED:
        _ekb_cfg = _unify_weights(_eval_routes_cfg("EKB", EKB_SPEC))
        _set_cfg_value(_ekb_cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
        _set_cfg_value(_ekb_cfg, "experiment.cpu", bool(EKB_FORCE_CPU))
        _ekb_res = _run_baseline(
            None, _ekb_cfg, EKB_INIT, "EKB_seed_eval_", {}, tensors=EKB_TENSORS,
            use_weighted_connectivity=True, connectivity_mode=CONNECTIVITY_MODE)
        EKB_SEED_ROW = _row("EKB", "Initial EKB routes", "ekb_seed",
                            _ekb_res[1], _ekb_res[3], EKB_INIT)
        display(pd.DataFrame([EKB_SEED_ROW]).round(4))

    _ekb_rr = EKB_INIT[0] if EKB_INIT.ndim == 3 else EKB_INIT
    _ekb_plot_adj = make_ekb_street_underlay_adj(EKB_TENSORS["street_adj"])
    _ekb_subtitle = (f"{EKB_STATS['n_routes']} routes | {EKB_STATS['unique_stops']} stops | "
                     f"len {EKB_STATS['min_len']}-{EKB_STATS['max_len']} "
                     f"mean={EKB_STATS['mean_len']:.1f}")
    if EKB_SEED_ROW is not None:
        _ekb_subtitle += (f"\ncost={EKB_SEED_ROW['cost']:.3f}  RTT={EKB_SEED_ROW['RTT']:.0f}  "
                          f"WMC={EKB_SEED_ROW['WMC']:.2f}  redun={EKB_SEED_ROW['redun%']:.0f}%")

    fig, ax = plt.subplots(1, 1, figsize=(10.5, 10), constrained_layout=True)
    route_plots.plot_plain_route_set(
        ax, _ekb_rr, EKB_TENSORS["node_locs"], _ekb_plot_adj,
        title=f"EKB {EKB_CASE_TAG} supplied routes", subtitle=_ekb_subtitle,
        palette="tab20", with_overlap_curves=True,
        show_node_labels=False, node_size=EKB_ROUTE_FIG_NODE_SIZE)
    fig.suptitle(f"EKB {EKB_CASE_TAG} seed route network", fontsize=15, fontweight="bold")
    EKB_STATIC_CASE_MAP_PATH = (EKB_STATIC_MAP_PATH if EKB_CASE_TAG == "full"
                                else EKB_STATIC_MAP_PATH.with_name(
                                    f"ekb_{EKB_CASE_TAG}_seed_routes_static.png"))
    EKB_STATIC_CASE_MAP_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(EKB_STATIC_CASE_MAP_PATH, dpi=220, bbox_inches="tight")
    print(f"[EKB] static route map saved -> {EKB_STATIC_CASE_MAP_PATH}")
    plt.show(); plt.close(fig)


### EKB NBCO GNN + trim/extend

Run our NBCO variant on the Ekaterinburg instance and draw before/after route sets in both plain and diff modes. Overlapping route edges are rendered as curved arcs so duplicate coverage stays visible instead of collapsing into one line.


In [27]:
# EKB NBCO: GNN rebuild bees + trim/extend edit bees, then plain/diff route figures.
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from eval_lib import plots as route_plots
from eval_lib.ekb import (EKB_COORD_CRS, ekb_route_stats, ekb_spec,
                          load_ekb_routes, load_ekb_tensors,
                          make_ekb_crop_case, make_ekb_street_underlay_adj)

RUN_EKB_NBCO = True
EKB_FORCE_CPU = globals().get("EKB_FORCE_CPU", True)
USE_EKB_CROP = globals().get("USE_EKB_CROP", False)   # full network by default
EKB_CROP_TARGET_NODES = globals().get("EKB_CROP_TARGET_NODES", 300)
EKB_CROP_MIN_ROUTE_LEN = globals().get("EKB_CROP_MIN_ROUTE_LEN", 11)
EKB_CROP_KEEP_LARGEST_COMPONENT = globals().get("EKB_CROP_KEEP_LARGEST_COMPONENT", True)
EKB_PROCESS_NEURAL_BEES_SEQUENTIALLY = True
EKB_NBCO_FORCE_RERUN = False
EKB_NBCO_ITERATIONS = 25
EKB_NBCO_ADJ_TARGET = ADJ_TARGET
EKB_NBCO_FINAL_LABEL = "Our NBCO (GNN + trim/extend)"
_EKB_SEQ_SUFFIX = "_seqbees" if EKB_PROCESS_NEURAL_BEES_SEQUENTIALLY else ""
_EKB_CASE_SUFFIX = f"_crop{int(EKB_CROP_TARGET_NODES)}" if USE_EKB_CROP else ""
EKB_NBCO_TABLE = f"final_ekb{_EKB_CASE_SUFFIX}_nbco_gnn_trimextend_iter{EKB_NBCO_ITERATIONS}{_EKB_SEQ_SUFFIX}"
EKB_NBCO_ROUTES_PATH = PAPER_DIR / f"{EKB_NBCO_TABLE}_routes.pt"
EKB_NBCO_CSV_PATH = PAPER_DIR / f"{EKB_NBCO_TABLE}.csv"
EKB_NBCO_OVERLAP_CURVES = True
EKB_NBCO_NODE_SIZE = 8

def _load_active_ekb_case():
    full_tensors = load_ekb_tensors()
    full_init = load_ekb_routes()
    if USE_EKB_CROP:
        tensors, init_routes, crop_meta = make_ekb_crop_case(
            full_tensors, full_init,
            target_nodes=EKB_CROP_TARGET_NODES,
            min_route_len=EKB_CROP_MIN_ROUTE_LEN,
            keep_largest_component=EKB_CROP_KEEP_LARGEST_COMPONENT)
        case_tag = f"crop{crop_meta['n_nodes']}"
    else:
        tensors, init_routes = full_tensors, full_init
        crop_meta = {"crop_enabled": False}
        case_tag = "full"
    return tensors, as_route_tensor(init_routes), ekb_spec(init_routes), crop_meta, case_tag


if RUN_EKB_NBCO:
    EKB_TENSORS, EKB_INIT, EKB_SPEC, EKB_CROP_META, EKB_CASE_TAG = _load_active_ekb_case()
    EKB_INIT = as_route_tensor(EKB_INIT)
    EKB_NBCO_RESULTS = {"Initial EKB routes": EKB_INIT}
    print(f"[EKB NBCO] active case={EKB_CASE_TAG} spec={EKB_SPEC}")
    if USE_EKB_CROP:
        print("[EKB NBCO] crop meta:", {
            "nodes": f"{EKB_CROP_META['n_nodes']}/{EKB_CROP_META['n_nodes_full']}",
            "routes": f"{EKB_CROP_META['n_routes']}/{EKB_CROP_META['n_routes_full']}",
            "radius_km": round(EKB_CROP_META['radius_m'] / 1000.0, 3),
        })
else:
    EKB_NBCO_RESULTS = {}
EKB_NBCO_ROWS = []
EKB_NBCO_HISTORY = {}
EKB_NBCO_MUTATION_COUNTS = {}


def _ekb_score_fixed_routes(routes, tag):
    scoped_tag = f"{EKB_CASE_TAG}_{tag}"
    cfg = _unify_weights(_eval_routes_cfg("EKB", EKB_SPEC))
    _set_cfg_value(cfg, "run_name", f"EKB_{scoped_tag}")
    _set_cfg_value(cfg, "experiment.cpu", bool(EKB_FORCE_CPU))
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
    return _run_baseline(
        None, cfg, routes, f"EKB_{scoped_tag}_", {}, tensors=EKB_TENSORS,
        use_weighted_connectivity=True, connectivity_mode=CONNECTIVITY_MODE,
        adjustment_seed_routes=EKB_INIT,
        **dict(UNIFIED_ADJ, adjustment_degree_target=float(EKB_NBCO_ADJ_TARGET)))


def _ekb_row(method, source, metrics, routes, duration_s=None, seed_cost=None):
    routes = as_route_tensor(routes)
    row = _row("EKB", method, source, metrics, routes, EKB_INIT, duration_s=duration_s)
    row.update(ekb_case=EKB_CASE_TAG,
               n_nodes=int(EKB_TENSORS["node_locs"].shape[0]),
               n_routes_active=int(EKB_SPEC["n_routes"]),
               adj_target=float(EKB_NBCO_ADJ_TARGET),
               n_iterations=int(EKB_NBCO_ITERATIONS),
               n_bees=int(algo_settings("EKB")["bco_bees"]),
               process_neural_bees_sequentially=
               bool(EKB_PROCESS_NEURAL_BEES_SEQUENTIALLY))
    if seed_cost is None:
        row["seed_cost"] = float(row["cost"])
        row["cost_delta"] = 0.0
        row["cost_delta_pct"] = 0.0
    else:
        row["seed_cost"] = float(seed_cost)
        row["cost_delta"] = float(row["cost"] - seed_cost)
        row["cost_delta_pct"] = (100.0 * row["cost_delta"] / abs(seed_cost)
                                   if abs(seed_cost) > 1e-12 else np.nan)
    return row


def _ekb_metric_subtitle(label):
    if EKB_NBCO_DF.empty or "method" not in EKB_NBCO_DF:
        return ""
    sub = EKB_NBCO_DF[EKB_NBCO_DF["method"] == label]
    if sub.empty:
        return ""
    r = sub.iloc[0]
    return (f"cost={r['cost']:.3f}  d={r.get('cost_delta', np.nan):.3f}  "
            f"seed={r.get('seed_cost', np.nan):.3f}\n"
            f"RTT={r['RTT']:.0f}  WMC={r['WMC']:.2f}  adj={r['adj_vs_seed']:.2f}  "
            f"d_un={r['d_un']:.1f}%  redun={r['redun%']:.0f}%")


if not RUN_EKB_NBCO:
    EKB_NBCO_DF = pd.DataFrame()
    print("[EKB NBCO] skipped (RUN_EKB_NBCO=False)")
elif (EKB_NBCO_ROUTES_PATH.exists() and EKB_NBCO_CSV_PATH.exists()
        and not EKB_NBCO_FORCE_RERUN):
    _dump = torch.load(EKB_NBCO_ROUTES_PATH, weights_only=False)
    EKB_NBCO_RESULTS = {k: as_route_tensor(v) for k, v in _dump["routes"].items()}
    EKB_NBCO_DF = pd.read_csv(EKB_NBCO_CSV_PATH)
    print(f"[EKB NBCO] loaded cached routes/table: {EKB_NBCO_ROUTES_PATH.name}")
else:
    reset_paper_table(EKB_NBCO_TABLE)
    print("[EKB NBCO] scoring initial EKB routes ...", flush=True)
    _seed_res = _ekb_score_fixed_routes(EKB_INIT, "nbco_seed_eval")
    EKB_SEED_ROW = _ekb_row("Initial EKB routes", f"ekb_{EKB_CASE_TAG}_nbco_seed",
                            _seed_res[1], EKB_INIT)
    EKB_NBCO_ROWS.append(EKB_SEED_ROW)
    append_paper_row(EKB_SEED_ROW, EKB_NBCO_TABLE, ndigits=4)

    if RUN_EKB_NBCO:
        print(f"[EKB NBCO] running {EKB_NBCO_FINAL_LABEL} on {EKB_CASE_TAG}: "
              f"iters={EKB_NBCO_ITERATIONS}, target={EKB_NBCO_ADJ_TARGET} ...",
              flush=True)
        _cfg = _our_model_cfg("EKB", EKB_SPEC, adj_target=EKB_NBCO_ADJ_TARGET,
                              run_name_suffix=f"{EKB_CASE_TAG}_", seed=SEEDS[0], use_gnn=True,
                              force_cpu=EKB_FORCE_CPU)
        _set_cfg_value(_cfg, "experiment.cpu", bool(EKB_FORCE_CPU))
        _set_cfg_value(
            _cfg, "process_neural_bees_sequentially",
            bool(EKB_PROCESS_NEURAL_BEES_SEQUENTIALLY))
        print("[EKB NBCO] experiment.cpu=", bool(_cfg.experiment.get("cpu", False)))
        print("[EKB NBCO] process_neural_bees_sequentially=",
              bool(_cfg.get("process_neural_bees_sequentially", False)))
        bco_cfg_set(_cfg, n_iterations=int(EKB_NBCO_ITERATIONS),
                    **dict(UNIFIED_ADJ, adjustment_degree_target=float(EKB_NBCO_ADJ_TARGET)))
        _set_cfg_value(_cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)

        # Live checkpoint: overwrite final_ekb_best_solution with the current
        # BCO incumbent (routes + characteristics) on every improvement, so
        # stopping the run early never loses the best solution.
        EKB_BEST_STEM = "final_ekb_best_solution"
        def _ekb_save_best_checkpoint(iteration, best_networks, best_costs,
                                      best_adj, best_metrics, metric_names):
            _best_rt = as_route_tensor(best_networks[0]).cpu()
            _mrow = {str(_n): float(_v) for _n, _v in
                     zip(metric_names, best_metrics[0].tolist())}
            _mrow.update(method=EKB_NBCO_FINAL_LABEL,
                         checkpoint_iteration=int(iteration),
                         objective_cost=float(best_costs[0]),
                         adj_vs_seed=float(best_adj[0]),
                         case_tag=EKB_CASE_TAG,
                         adj_target=float(EKB_NBCO_ADJ_TARGET),
                         n_iterations=int(EKB_NBCO_ITERATIONS))
            save_paper_table(pd.DataFrame([_mrow]).round(6), EKB_BEST_STEM)
            save_paper_routes(EKB_BEST_STEM,
                              {"Initial EKB routes": as_route_tensor(EKB_INIT),
                               EKB_NBCO_FINAL_LABEL: _best_rt},
                              meta={"city": "EKB", "best_method": EKB_NBCO_FINAL_LABEL,
                                    "checkpoint_iteration": int(iteration),
                                    "case_tag": EKB_CASE_TAG,
                                    "objective_cost": float(best_costs[0])})
            _w = globals().get("_ekb_tb_writer")
            if _w is not None:
                _w.add_scalar("ekb/objective_cost", float(best_costs[0]), int(iteration))
                _w.add_scalar("ekb/adj_vs_seed", float(best_adj[0]), int(iteration))
                for _n, _v in zip(metric_names, best_metrics[0].tolist()):
                    _w.add_scalar(f"ekb/{_n}", float(_v), int(iteration))
                _w.flush()

        from torch.utils.tensorboard import SummaryWriter as _SummaryWriter
        _ekb_tb_dir = MODEL_OUTPUTS_DIR / "tensorboard" / f"ekb_{EKB_CASE_TAG}_iter{EKB_NBCO_ITERATIONS}"
        _ekb_tb_dir.mkdir(parents=True, exist_ok=True)
        _ekb_tb_writer = _SummaryWriter(log_dir=str(_ekb_tb_dir))
        print(f"[EKB NBCO][tensorboard] -> {_ekb_tb_dir}")
        print(f"  view live:  tensorboard --logdir {MODEL_OUTPUTS_DIR / 'tensorboard'}")

        _t0 = _time.perf_counter()
        try:
            _res = run_bco(
                _cfg, EKB_INIT, tensors=EKB_TENSORS, run_name_scope=f"EKB_{EKB_CASE_TAG}_",
                cost_history_out=EKB_NBCO_HISTORY,
                mutation_counts_out=EKB_NBCO_MUTATION_COUNTS,
                iteration_callback=_ekb_save_best_checkpoint)
        except KeyboardInterrupt:
            print(f"[EKB NBCO] interrupted -- best-so-far kept in "
                  f"{EKB_BEST_STEM}.csv + {EKB_BEST_STEM}_routes.pt")
            raise
        _dt = _time.perf_counter() - _t0
        if globals().get("_ekb_tb_writer") is not None:
            _ekb_tb_writer.close()
        _routes = as_route_tensor(_res[3])
        _row_out = _ekb_row(EKB_NBCO_FINAL_LABEL, f"ekb_{EKB_CASE_TAG}_nbco_gnn_trimextend",
                            _res[1], _routes, duration_s=_dt,
                            seed_cost=EKB_SEED_ROW["cost"])
        EKB_NBCO_ROWS.append(_row_out)
        EKB_NBCO_RESULTS[EKB_NBCO_FINAL_LABEL] = _routes
        append_paper_row(_row_out, EKB_NBCO_TABLE, ndigits=4)
        torch.save({"history": _ravel_hist(EKB_NBCO_HISTORY.get("history")),
                    "mutation_counts": EKB_NBCO_MUTATION_COUNTS},
                   PAPER_DIR / f"{EKB_NBCO_TABLE}_history.pt")

    EKB_NBCO_DF = pd.DataFrame(EKB_NBCO_ROWS)
    save_paper_table(EKB_NBCO_DF.round(4), EKB_NBCO_TABLE)
    save_paper_routes(EKB_NBCO_TABLE, EKB_NBCO_RESULTS,
                      EKB_TENSORS["node_locs"], EKB_TENSORS["street_adj"],
                      meta={"city": "EKB", "case_tag": EKB_CASE_TAG,
                            "crop": EKB_CROP_META,
                            "objective": "NBCO GNN + trim/extend",
                            "adj_target": float(EKB_NBCO_ADJ_TARGET),
                            "n_iterations": int(EKB_NBCO_ITERATIONS),
                            "process_neural_bees_sequentially":
                            bool(EKB_PROCESS_NEURAL_BEES_SEQUENTIALLY),
                            "coord_crs": EKB_COORD_CRS})

if RUN_EKB_NBCO and not EKB_NBCO_DF.empty:
    display(EKB_NBCO_DF.round(4))

_final_key = next((k for k in EKB_NBCO_RESULTS if "Our NBCO" in k), None)
if not RUN_EKB_NBCO:
    pass
elif _final_key is None:
    print("[EKB NBCO] no final NBCO route set available yet; set RUN_EKB_NBCO=True or load a cached dump.")
else:
    _init_rr = EKB_INIT[0] if EKB_INIT.ndim == 3 else EKB_INIT
    _final_rt = as_route_tensor(EKB_NBCO_RESULTS[_final_key])
    _final_rr = _final_rt[0] if _final_rt.ndim == 3 else _final_rt
    _plot_adj = make_ekb_street_underlay_adj(EKB_TENSORS["street_adj"])
    _changes = route_plots.summarize_route_changes(_final_rr, _init_rr)
    print("[EKB NBCO] route changes:", _changes,
          "stats before", ekb_route_stats(_init_rr),
          "stats after", ekb_route_stats(_final_rr))

    fig, axes = plt.subplots(1, 2, figsize=(18, 8.5), squeeze=False,
                             constrained_layout=True)
    route_plots.plot_plain_route_set(
        axes[0, 0], _init_rr, EKB_TENSORS["node_locs"], _plot_adj,
        title=f"EKB {EKB_CASE_TAG} initial routes", subtitle=_ekb_metric_subtitle("Initial EKB routes"),
        palette="tab20", with_overlap_curves=EKB_NBCO_OVERLAP_CURVES,
        show_node_labels=False, node_size=EKB_NBCO_NODE_SIZE)
    route_plots.plot_plain_route_set(
        axes[0, 1], _final_rr, EKB_TENSORS["node_locs"], _plot_adj,
        title=EKB_NBCO_FINAL_LABEL, subtitle=_ekb_metric_subtitle(_final_key),
        palette="tab20", with_overlap_curves=EKB_NBCO_OVERLAP_CURVES,
        show_node_labels=False, node_size=EKB_NBCO_NODE_SIZE)
    fig.suptitle(f"EKB {EKB_CASE_TAG} NBCO GNN + trim/extend: plain before / after",
                 fontsize=15, fontweight="bold")
    plt.show(); plt.close(fig)

    fig, ax = plt.subplots(1, 1, figsize=(10.5, 10), constrained_layout=True)
    _diff_sub = ("init: " + _ekb_metric_subtitle("Initial EKB routes").replace("\n", "\ninit: ") + "\n"
                 "result: " + _ekb_metric_subtitle(_final_key).replace("\n", "\nresult: ") + "\n"
                 f"changed_routes={_changes['changed_routes']}  "
                 f"+edges={_changes['added_edges']}  -edges={_changes['removed_edges']}  "
                 f"+stops={_changes['added_stops']}  -stops={_changes['removed_stops']}")
    route_plots.plot_route_diff(
        ax, _final_rr, _init_rr, EKB_TENSORS["node_locs"], _plot_adj,
        title=f"EKB {EKB_CASE_TAG} diff: NBCO vs initial", subtitle=_diff_sub,
        palette="tab20", with_overlap_curves=EKB_NBCO_OVERLAP_CURVES,
        show_node_labels=False, node_size=EKB_NBCO_NODE_SIZE)
    fig.suptitle(f"EKB {EKB_CASE_TAG} NBCO GNN + trim/extend: diff vs initial",
                 fontsize=15, fontweight="bold")
    plt.show(); plt.close(fig)


[EKB NBCO] active case=full spec={'city': 'EKB', 'n_routes': 67, 'min_route_len': 11, 'max_route_len': 54}
[EKB NBCO] scoring initial EKB routes ...
,67.000,4.903,10.717,1606.335,18.087,43.778,10.111,28.024,71816.500,0.000,15.305,15.599,1.249,0.000
[paper] row -> D:\PythonProjects\connectpt\artifacts\paper_results\final_ekb_nbco_gnn_trimextend_iter25_seqbees.csv
[EKB NBCO] running Our NBCO (GNN + trim/extend) on full: iters=25, target=0.2 ...
[EKB_full_our_nbco_gnn_rebuild_trimext] route_selection=uniform-random | bee split: neural_rebuild=5 local_endpoint_edit=0 path_mix_rebuild=0 construction_extend=0 extend_trim_edit=5 trim_only=0 trim_then_extend=0 (total=10)
[EKB NBCO] experiment.cpu= True
[EKB NBCO] process_neural_bees_sequentially= True
[EKB NBCO][tensorboard] -> D:\PythonProjects\connectpt\artifacts\lc_improvement_outputs\tensorboard\ekb_full_iter25
  view live:  tensorboard --logdir D:\PythonProjects\connectpt\artifacts\lc_improvement_outputs\tensorboard


  0%|          | 0/25 [00:59<?, ?it/s]


[EKB NBCO] interrupted -- best-so-far kept in final_ekb_best_solution.csv + final_ekb_best_solution_routes.pt


KeyboardInterrupt: 

# EKB alpha sweep

Run only the Ekaterinburg alpha sweep with adjustment target 0.3. Results are written to a separate table and route dump so existing paper results are not overwritten.


In [ ]:
# EKB alpha sweep: target=0.3, alpha=0.0..1.0 step 0.2, 50 BCO iterations.
import time as _time
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from IPython.display import display
from eval_lib.ekb import ekb_spec, load_ekb_routes, load_ekb_tensors, make_ekb_crop_case

RUN_EKB_ALPHA_SWEEP = True
EKB_SWEEP_FORCE_RERUN = False
EKB_SWEEP_ITERATIONS = 50
EKB_SWEEP_ADJ_TARGET = 0.3
EKB_SWEEP_ALPHA_GRID = [0.0, 0.5, 1.0]
EKB_FORCE_CPU = globals().get("EKB_FORCE_CPU", True)
USE_EKB_CROP = globals().get("USE_EKB_CROP", False)
EKB_CROP_TARGET_NODES = globals().get("EKB_CROP_TARGET_NODES", 300)
EKB_CROP_MIN_ROUTE_LEN = globals().get("EKB_CROP_MIN_ROUTE_LEN", 11)
EKB_CROP_KEEP_LARGEST_COMPONENT = globals().get("EKB_CROP_KEEP_LARGEST_COMPONENT", True)
EKB_PROCESS_NEURAL_BEES_SEQUENTIALLY = False


def _ekb_sweep_load_case():
    full_tensors = load_ekb_tensors()
    full_init = load_ekb_routes()
    if USE_EKB_CROP:
        tensors, init_routes, crop_meta = make_ekb_crop_case(
            full_tensors, full_init,
            target_nodes=EKB_CROP_TARGET_NODES,
            min_route_len=EKB_CROP_MIN_ROUTE_LEN,
            keep_largest_component=EKB_CROP_KEEP_LARGEST_COMPONENT)
        case_tag = f"crop{crop_meta['n_nodes']}"
    else:
        tensors, init_routes = full_tensors, full_init
        crop_meta = {"crop_enabled": False}
        case_tag = "full"
    return tensors, as_route_tensor(init_routes), ekb_spec(init_routes), crop_meta, case_tag


def _ekb_sweep_score(routes, *, alpha, tag, seed_routes):
    cfg = _unify_weights(_eval_routes_cfg("EKB", EKB_SWEEP_SPEC))
    _set_cfg_value(cfg, "run_name", f"EKB_{EKB_SWEEP_CASE_TAG}_{tag}")
    _set_cfg_value(cfg, "experiment.cpu", bool(EKB_FORCE_CPU))
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.route_time_weight", float(alpha))
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.median_connectivity_weight", float(1.0 - alpha))
    return _run_baseline(
        None, cfg, routes, f"EKB_{EKB_SWEEP_CASE_TAG}_{tag}_", {}, tensors=EKB_SWEEP_TENSORS,
        use_weighted_connectivity=True, connectivity_mode=CONNECTIVITY_MODE,
        adjustment_seed_routes=seed_routes,
        **dict(UNIFIED_ADJ, adjustment_degree_target=float(EKB_SWEEP_ADJ_TARGET)))


def _ekb_sweep_row(method, source, metrics, routes, *, alpha, duration_s=None, seed_cost=None):
    row = _row("EKB", method, source, metrics, routes, EKB_SWEEP_INIT, duration_s=duration_s)
    row.update(ekb_case=EKB_SWEEP_CASE_TAG,
               n_nodes=int(EKB_SWEEP_TENSORS["node_locs"].shape[0]),
               alpha=float(alpha),
               adj_target=float(EKB_SWEEP_ADJ_TARGET),
               n_iterations=int(EKB_SWEEP_ITERATIONS),
               n_bees=int(algo_settings("EKB")["bco_bees"]),
               process_neural_bees_sequentially=bool(EKB_PROCESS_NEURAL_BEES_SEQUENTIALLY))
    if seed_cost is not None and "cost" in row:
        row["seed_cost"] = float(seed_cost)
        row["delta_cost"] = float(row["cost"]) - float(seed_cost)
    return row


_EKB_SWEEP_CASE_SUFFIX = f"_crop{int(EKB_CROP_TARGET_NODES)}" if USE_EKB_CROP else ""
EKB_SWEEP_TABLE = (
    f"final_ekb{_EKB_SWEEP_CASE_SUFFIX}_alpha_sweep_target{str(EKB_SWEEP_ADJ_TARGET).replace('.', 'p')}"
    f"_iter{EKB_SWEEP_ITERATIONS}")
EKB_SWEEP_ROUTES_PATH = PAPER_DIR / f"{EKB_SWEEP_TABLE}_routes.pt"
EKB_SWEEP_CSV_PATH = PAPER_DIR / f"{EKB_SWEEP_TABLE}.csv"

if not RUN_EKB_ALPHA_SWEEP:
    EKB_SWEEP_DF = pd.DataFrame()
    print("[EKB sweep] skipped (RUN_EKB_ALPHA_SWEEP=False)")
elif EKB_SWEEP_ROUTES_PATH.exists() and EKB_SWEEP_CSV_PATH.exists() and not EKB_SWEEP_FORCE_RERUN:
    _payload = torch.load(EKB_SWEEP_ROUTES_PATH, map_location="cpu", weights_only=False)
    EKB_SWEEP_ROUTES = {k: as_route_tensor(v) for k, v in _payload.get("routes", {}).items()}
    EKB_SWEEP_DF = pd.read_csv(EKB_SWEEP_CSV_PATH)
    print(f"[EKB sweep] loaded cache: {EKB_SWEEP_CSV_PATH.name}")
    display(EKB_SWEEP_DF.round(4))
else:
    EKB_SWEEP_TENSORS, EKB_SWEEP_INIT, EKB_SWEEP_SPEC, EKB_SWEEP_CROP_META, EKB_SWEEP_CASE_TAG = _ekb_sweep_load_case()
    print(f"[EKB sweep] case={EKB_SWEEP_CASE_TAG} spec={EKB_SWEEP_SPEC} "
          f"alphas={EKB_SWEEP_ALPHA_GRID} target={EKB_SWEEP_ADJ_TARGET} "
          f"iters={EKB_SWEEP_ITERATIONS} cpu={EKB_FORCE_CPU}")
    reset_paper_table(EKB_SWEEP_TABLE)
    EKB_SWEEP_ROWS = []
    EKB_SWEEP_ROUTES = {"Initial EKB routes": EKB_SWEEP_INIT}

    for _alpha in tqdm(EKB_SWEEP_ALPHA_GRID, desc="EKB alpha sweep"):
        _seed_res = _ekb_sweep_score(
            EKB_SWEEP_INIT, alpha=_alpha, tag=f"seed_alpha{_alpha:g}", seed_routes=EKB_SWEEP_INIT)
        _seed_row = _ekb_sweep_row(
            "Initial EKB routes", f"ekb_{EKB_SWEEP_CASE_TAG}_seed_alpha{_alpha:g}",
            _seed_res[1], _seed_res[3], alpha=_alpha)
        EKB_SWEEP_ROWS.append(_seed_row)
        append_paper_row(_seed_row, EKB_SWEEP_TABLE, ndigits=4)

        _cfg = _our_model_cfg(
            "EKB", EKB_SWEEP_SPEC, adj_target=EKB_SWEEP_ADJ_TARGET,
            run_name_suffix=f"{EKB_SWEEP_CASE_TAG}_alpha{_alpha:g}_",
            seed=SEEDS[0], use_gnn=True, force_cpu=EKB_FORCE_CPU)
        _set_cfg_value(_cfg, "experiment.cpu", bool(EKB_FORCE_CPU))
        _set_cfg_value(_cfg, "experiment.cost_function.kwargs.route_time_weight", float(_alpha))
        _set_cfg_value(_cfg, "experiment.cost_function.kwargs.median_connectivity_weight", float(1.0 - _alpha))
        _set_cfg_value(_cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
        _set_cfg_value(_cfg, "process_neural_bees_sequentially", bool(EKB_PROCESS_NEURAL_BEES_SEQUENTIALLY))
        bco_cfg_set(_cfg, n_iterations=int(EKB_SWEEP_ITERATIONS),
                    **dict(UNIFIED_ADJ, adjustment_degree_target=float(EKB_SWEEP_ADJ_TARGET)))
        _t0 = _time.perf_counter()
        _run_name, _metrics, _unserved, _routes, _mutation_counts = run_bco(
            _cfg, EKB_SWEEP_INIT, tensors=EKB_SWEEP_TENSORS,
            run_name_scope=f"EKB_{EKB_SWEEP_CASE_TAG}_alpha{_alpha:g}_")
        _dt = _time.perf_counter() - _t0
        _routes = as_route_tensor(_routes)
        _label = f"Our NBCO alpha={_alpha:g}"
        EKB_SWEEP_ROUTES[_label] = _routes
        _score_res = _ekb_sweep_score(
            _routes, alpha=_alpha, tag=f"our_alpha{_alpha:g}", seed_routes=EKB_SWEEP_INIT)
        _row_out = _ekb_sweep_row(
            "Our NBCO (GNN + trim/extend)", f"ekb_{EKB_SWEEP_CASE_TAG}_our_alpha{_alpha:g}",
            _score_res[1], _score_res[3], alpha=_alpha, duration_s=_dt,
            seed_cost=_seed_row.get("cost"))
        EKB_SWEEP_ROWS.append(_row_out)
        append_paper_row(_row_out, EKB_SWEEP_TABLE, ndigits=4)
        save_paper_routes(EKB_SWEEP_TABLE, EKB_SWEEP_ROUTES,
                          EKB_SWEEP_TENSORS["node_locs"], EKB_SWEEP_TENSORS["street_adj"],
                          meta={"city": "EKB", "case_tag": EKB_SWEEP_CASE_TAG,
                                "crop": EKB_SWEEP_CROP_META,
                                "alpha_grid": EKB_SWEEP_ALPHA_GRID,
                                "adj_target": float(EKB_SWEEP_ADJ_TARGET),
                                "n_iterations": int(EKB_SWEEP_ITERATIONS),
                                "process_neural_bees_sequentially":
                                bool(EKB_PROCESS_NEURAL_BEES_SEQUENTIALLY)})

    EKB_SWEEP_DF = pd.DataFrame(EKB_SWEEP_ROWS)
    save_paper_table(EKB_SWEEP_DF.round(4), EKB_SWEEP_TABLE)
    save_paper_routes(EKB_SWEEP_TABLE, EKB_SWEEP_ROUTES,
                      EKB_SWEEP_TENSORS["node_locs"], EKB_SWEEP_TENSORS["street_adj"],
                      meta={"city": "EKB", "case_tag": EKB_SWEEP_CASE_TAG,
                            "crop": EKB_SWEEP_CROP_META,
                            "alpha_grid": EKB_SWEEP_ALPHA_GRID,
                            "adj_target": float(EKB_SWEEP_ADJ_TARGET),
                            "n_iterations": int(EKB_SWEEP_ITERATIONS),
                            "process_neural_bees_sequentially":
                            bool(EKB_PROCESS_NEURAL_BEES_SEQUENTIALLY)})
    display(EKB_SWEEP_DF.round(4))
    print(f"[EKB sweep] saved -> {EKB_SWEEP_CSV_PATH.name} + {EKB_SWEEP_ROUTES_PATH.name}")


### EKB best solution (overwritten each run)

The best achieved EKB network (BCO incumbent) is written to fixed-name `artifacts/paper_results/final_ekb_best_solution.{csv,_routes.pt}` and overwritten on every run.

In [ ]:
# Save the single BEST achieved EKB solution (routes + characteristics) to a
# FIXED-stem file, OVERWRITTEN on every run (independent of the iter/seqbees-
# suffixed cache files above). The BCO returns its best incumbent, so the
# 'Our NBCO' result is the best solution.
EKB_BEST_STEM = "final_ekb_best_solution"
if not RUN_EKB_NBCO or EKB_NBCO_DF.empty:
    print("[EKB best] skipped (EKB NBCO not run).")
else:
    _ekb_best_key = next((k for k in EKB_NBCO_RESULTS if "Our NBCO" in k), None)
    if _ekb_best_key is None:
        print("[EKB best] no Our NBCO solution to save.")
    else:
        _ekb_best_routes = as_route_tensor(EKB_NBCO_RESULTS[_ekb_best_key])
        _ekb_best_row = EKB_NBCO_DF[EKB_NBCO_DF["method"] == _ekb_best_key].round(6)
        save_paper_table(_ekb_best_row, EKB_BEST_STEM)            # overwrites .csv
        save_paper_routes(                                        # overwrites _routes.pt
            EKB_BEST_STEM,
            {"Initial EKB routes": as_route_tensor(EKB_INIT),
             _ekb_best_key: _ekb_best_routes},
            EKB_TENSORS["node_locs"], EKB_TENSORS["street_adj"],
            meta={"city": "EKB", "best_method": _ekb_best_key,
                  "case_tag": globals().get("EKB_CASE_TAG", "full"),
                  "adj_target": float(EKB_NBCO_ADJ_TARGET),
                  "n_iterations": int(EKB_NBCO_ITERATIONS),
                  "process_neural_bees_sequentially":
                  bool(EKB_PROCESS_NEURAL_BEES_SEQUENTIALLY),
                  "source_table": EKB_NBCO_TABLE})
        display(_ekb_best_row)
        print(f"[EKB best] overwritten -> {EKB_BEST_STEM}.csv + {EKB_BEST_STEM}_routes.pt "
              f"(method={_ekb_best_key})")


In [ ]:
# === Mumford0 neural-BCO variants (alpha=0.5, target=0.3, 200 BCO iterations) ===
M0_NBCO_CITY = "Mumford0"
M0_NBCO_ALPHA = 0.5
M0_NBCO_ADJ_TARGET = 0.3
M0_NBCO_ITERS = 200
M0_NBCO_GNN_TRIMEXT_BEES = 10
M0_NBCO_OTHER_BEES = 24
M0_NBCO_TABLE = ("final_mumford0_nbco_variants_alpha05_target03_iter200"
                 if RUN_M0_NBCO_VARIANTS
                 else "final_mumford0_nbco_variants_alpha05_target03_iter200_skipped")

M0_NBCO_VARIANTS = [] if not RUN_M0_NBCO_VARIANTS else [
    dict(method="Our NBCO (GNN + trim/extend)", run_name="gnn_trimextend",
         n_bees=M0_NBCO_GNN_TRIMEXT_BEES,
         use_neural_bees=True, n_type1_bees=5, n_type2_bees=0,
         n_type4_bees=0, n_type5_bees=5, n_type6_bees=0, n_type7_bees=0),
    dict(method="Trim 12 + extend 12", run_name="trim12_extend12",
         n_bees=M0_NBCO_OTHER_BEES,
         use_neural_bees=True, n_type1_bees=0, n_type2_bees=0,
         n_type4_bees=12, n_type5_bees=0, n_type6_bees=12, n_type7_bees=0),
]


def _m0_nbco_split(v):
    return (f"n_bees={v['n_bees']}, type1={v['n_type1_bees']}, "
            f"type2={v['n_type2_bees']}, "
            f"type4={v['n_type4_bees']}, type5={v['n_type5_bees']}, "
            f"type6={v['n_type6_bees']}, type7={v['n_type7_bees']}")


def _m0_nbco_cfg(spec, variant, *, seed=None):
    weights = dict(demand_time_weight=0.0,
                   route_time_weight=float(M0_NBCO_ALPHA),
                   median_connectivity_weight=float(1.0 - M0_NBCO_ALPHA))
    cfg = build_bco_cfg(
        run_name=(f"{M0_NBCO_CITY}_nbco_{variant['run_name']}_"
                  f"alpha05_target03_iter200"),
        n_routes=spec["n_routes"], min_route_len=spec["min_route_len"],
        max_route_len=spec["max_route_len"],
        use_neural_bees=variant["use_neural_bees"], n_bees=int(variant["n_bees"]),
        n_type1_bees=variant["n_type1_bees"],
        n_type2_bees=variant["n_type2_bees"],
        n_type4_bees=variant["n_type4_bees"],
        n_type5_bees=variant["n_type5_bees"],
        n_type6_bees=variant["n_type6_bees"],
        n_type7_bees=variant["n_type7_bees"],
        connectivity_mode=CONNECTIVITY_MODE,
        **algo_settings(M0_NBCO_CITY).get("bco_args", {}), **weights)
    bco_cfg_set(cfg, n_iterations=int(M0_NBCO_ITERS))
    bco_cfg_set(cfg, **dict(UNIFIED_ADJ,
                            adjustment_degree_target=float(M0_NBCO_ADJ_TARGET)))
    bco_cfg_set(cfg, type4_allow_halt=False, type5_allow_halt=False,
                type6_allow_halt=False, type7_allow_halt=False)
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
    if seed is not None:
        _set_cfg_value(cfg, "experiment.seed", int(seed))
    return cfg


def _m0_eval_routes_cfg(spec, run_name):
    cfg = _unify_weights(_eval_routes_cfg(M0_NBCO_CITY, spec))
    _set_cfg_value(cfg, "run_name", run_name)
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.demand_time_weight", 0.0)
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.route_time_weight",
                   float(M0_NBCO_ALPHA))
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.median_connectivity_weight",
                   float(1.0 - M0_NBCO_ALPHA))
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
    return cfg


def _m0_cost_weights(cost_obj, device):
    weights = {key: (value.clone() if torch.is_tensor(value) else value)
               for key, value in cost_obj.get_weights(device).items()}
    weights["demand_time_weight"] = torch.as_tensor(0.0, device=device)
    weights["route_time_weight"] = torch.as_tensor(float(M0_NBCO_ALPHA), device=device)
    weights["median_connectivity_weight"] = torch.as_tensor(
        float(1.0 - M0_NBCO_ALPHA), device=device)
    return weights


def _run_m0_rl_only(spec, tensors, init_routes):
    """Greedy eval of the trained RL edit agent on the same Mumford0 init."""
    eval_cfg = _m0_eval_routes_cfg(spec, f"{M0_NBCO_CITY}_rl_only_eval")
    dataloader = make_tensor_dataloader(eval_cfg.eval.dataset, tensors)
    device, _run_name, _, cost_obj, _ = lrnu.process_standard_experiment_cfg(
        eval_cfg, run_name_prefix=f"{M0_NBCO_CITY}_rl_only_", weights_required=False)
    cost_obj.use_weighted_connectivity = True
    cost_obj.connectivity_mode = CONNECTIVITY_MODE
    graph_batch = next(iter(dataloader))
    if device.type != "cpu":
        graph_batch = graph_batch.cuda()
    route_batch = as_route_tensor(init_routes).to(device)
    model = build_edit_model(device, weights_path=OUR_MODEL_PATH)
    max_steps = globals().get("MAX_ROUTE_EDIT_STEPS", None)
    max_trims = globals().get("MAX_TRIM_ACTIONS_PER_ROUTE", 1)
    rollout_kwargs = _rollout_adjustment_kwargs_for_model(
        model, target=M0_NBCO_ADJ_TARGET)
    with torch.no_grad():
        output = rollout_lc_improvement(
            model, cost_obj, graph_batch, route_batch,
            spec["min_route_len"], spec["max_route_len"],
            greedy=True, cost_weights=_m0_cost_weights(cost_obj, device),
            return_actions=True, max_route_edit_steps=max_steps,
            max_trim_actions_per_route=max_trims, **rollout_kwargs)
    final_state = output[0]
    route_actions, route_action_kinds = output[5], output[6]
    routes = get_batch_tensor_from_routes(
        final_state.routes, device, max_route_len=spec["max_route_len"])
    action_stats = summarize_route_action_stats(route_actions, route_action_kinds)
    eval_res = _run_baseline(
        None, _m0_eval_routes_cfg(spec, f"{M0_NBCO_CITY}_rl_only_score"),
        routes.detach().cpu(), f"{M0_NBCO_CITY}_rlscore_", {}, tensors=tensors,
        use_weighted_connectivity=True, connectivity_mode=CONNECTIVITY_MODE,
        adjustment_seed_routes=init_routes,
        **dict(UNIFIED_ADJ, adjustment_degree_target=float(M0_NBCO_ADJ_TARGET)))
    return eval_res, action_stats


_m0_spec = next(s for s in BENCHMARK_SPECS if s["city"] == M0_NBCO_CITY)
_m0_tensors, _m0_init = load_benchmark_graph(_m0_spec)
_m0_seed = as_route_tensor(_m0_init)
reset_paper_table(M0_NBCO_TABLE)

M0_NBCO_ROWS = []
_m0_init_label = f"Initial (LC+{EXP_INIT_TIER})"
M0_NBCO_ROUTES = {_m0_init_label: _m0_seed}
M0_NBCO_HISTORY = {}
M0_NBCO_MUTATION_COUNTS = {}

print(f"=== {M0_NBCO_CITY} NBCO/RL variants: alpha={M0_NBCO_ALPHA}, "
      f"target={M0_NBCO_ADJ_TARGET}, BCO iters={M0_NBCO_ITERS} ===")
print("  -> scoring initial routes ...", flush=True)
_m0_seed_res = _run_baseline(
    None, _m0_eval_routes_cfg(_m0_spec, f"{M0_NBCO_CITY}_nbco_init_eval"),
    _m0_seed, f"{M0_NBCO_CITY}_nbco_init_", {}, tensors=_m0_tensors,
    use_weighted_connectivity=True, connectivity_mode=CONNECTIVITY_MODE,
    adjustment_seed_routes=_m0_seed,
    **dict(UNIFIED_ADJ, adjustment_degree_target=float(M0_NBCO_ADJ_TARGET)))
_m0_init_row = _row(M0_NBCO_CITY, _m0_init_label, "mumford0_nbco_init",
                   _m0_seed_res[1], _m0_seed, _m0_seed)
_m0_init_row.update(alpha=float(M0_NBCO_ALPHA),
                    adj_target=float(M0_NBCO_ADJ_TARGET),
                    n_iterations=0, n_bees=0,
                    bee_split="initial route set",
                    seed_cost=float(_m0_init_row["cost"]),
                    cost_delta=0.0, cost_delta_pct=0.0,
                    is_initial=True)
M0_NBCO_ROWS.append(_m0_init_row)
append_paper_row(_m0_init_row, M0_NBCO_TABLE, ndigits=4)
for _variant in tqdm(M0_NBCO_VARIANTS, desc=f"{M0_NBCO_CITY} NBCO variants"):
    _label = _variant["method"]
    _history, _mutation_counts = {}, {}
    print(f"  -> {_label} ({_m0_nbco_split(_variant)}) ...", flush=True)
    _t0 = _time.perf_counter()
    try:
        _res = run_bco(
            _m0_nbco_cfg(_m0_spec, _variant, seed=SEEDS[0]), _m0_init,
            mutation_counts_out=_mutation_counts, tensors=_m0_tensors,
            run_name_scope=f"{M0_NBCO_CITY}_", cost_history_out=_history)
        _dt = _time.perf_counter() - _t0
        _routes = as_route_tensor(_res[3])
        _row_out = _row(M0_NBCO_CITY, _label, "mumford0_nbco_variants",
                        _res[1], _routes, _m0_seed, duration_s=_dt)
        _row_out.update(alpha=float(M0_NBCO_ALPHA),
                        adj_target=float(M0_NBCO_ADJ_TARGET),
                        n_iterations=int(M0_NBCO_ITERS),
                        n_bees=int(_variant["n_bees"]),
                        bee_split=_m0_nbco_split(_variant),
                        seed_cost=float(_m0_init_row["cost"]),
                        cost_delta=float(_row_out["cost"] - _m0_init_row["cost"]),
                        cost_delta_pct=(100.0 * float(_row_out["cost"] - _m0_init_row["cost"]) /
                                        abs(float(_m0_init_row["cost"]))
                                        if abs(float(_m0_init_row["cost"])) > 1e-12 else np.nan),
                        is_initial=False)
        M0_NBCO_ROWS.append(_row_out)
        M0_NBCO_ROUTES[_label] = _routes
        M0_NBCO_HISTORY[_label] = _ravel_hist(_history.get("history"))
        M0_NBCO_MUTATION_COUNTS[_label] = _mutation_counts
        append_paper_row(_row_out, M0_NBCO_TABLE, ndigits=4)
        print(f"  [done] {_label}: RTT={_row_out['RTT']:.0f} "
              f"WMC={_row_out['WMC']:.2f} adj={_row_out['adj_vs_seed']:.2f} "
              f"cost={_row_out['cost']:.4f} ({_dt:.0f}s)", flush=True)
    except Exception as exc:
        print(f"  {M0_NBCO_CITY}/{_label} FAILED: {exc}", flush=True)

if RUN_M0_NBCO_VARIANTS:
    print("  -> RL only (greedy edit-agent eval) ...", flush=True)
    _t0 = _time.perf_counter()
    try:
        _rl_res, _rl_action_stats = _run_m0_rl_only(_m0_spec, _m0_tensors, _m0_init)
        _dt = _time.perf_counter() - _t0
        _rl_routes = as_route_tensor(_rl_res[3])
        _rl_row = _row(M0_NBCO_CITY, "RL only", "mumford0_rl_only_eval",
                      _rl_res[1], _rl_routes, _m0_seed, duration_s=_dt)
        _rl_row.update(alpha=float(M0_NBCO_ALPHA),
                       adj_target=float(M0_NBCO_ADJ_TARGET),
                       n_iterations=None, n_bees=None,
                       bee_split="RL greedy rollout; no BCO population",
                       seed_cost=float(_m0_init_row["cost"]),
                       cost_delta=float(_rl_row["cost"] - _m0_init_row["cost"]),
                       cost_delta_pct=(100.0 * float(_rl_row["cost"] - _m0_init_row["cost"]) /
                                       abs(float(_m0_init_row["cost"]))
                                       if abs(float(_m0_init_row["cost"])) > 1e-12 else np.nan),
                       is_initial=False)
        M0_NBCO_ROWS.append(_rl_row)
        M0_NBCO_ROUTES["RL only"] = _rl_routes
        M0_NBCO_HISTORY["RL only"] = []
        M0_NBCO_MUTATION_COUNTS["RL only"] = _rl_action_stats
        append_paper_row(_rl_row, M0_NBCO_TABLE, ndigits=4)
        print(f"  [done] RL only: RTT={_rl_row['RTT']:.0f} "
              f"WMC={_rl_row['WMC']:.2f} adj={_rl_row['adj_vs_seed']:.2f} "
              f"cost={_rl_row['cost']:.4f} ({_dt:.0f}s)", flush=True)
    except Exception as exc:
        print(f"  {M0_NBCO_CITY}/RL only FAILED: {exc}", flush=True)
else:
    print("[M0_NBCO] variants and RL-only skipped (RUN_M0_NBCO_VARIANTS=False)", flush=True)

M0_NBCO_VARIANTS_DF = pd.DataFrame(M0_NBCO_ROWS).round(4)
if not M0_NBCO_VARIANTS_DF.empty:
    display(M0_NBCO_VARIANTS_DF)
    save_paper_table(M0_NBCO_VARIANTS_DF, M0_NBCO_TABLE)
    save_paper_routes(M0_NBCO_TABLE, M0_NBCO_ROUTES,
                      _m0_tensors["node_locs"], _m0_tensors["street_adj"],
                      meta={"city": M0_NBCO_CITY,
                            "objective": "alpha*RTT + (1-alpha)*WMC + adj",
                            "alpha": M0_NBCO_ALPHA,
                            "adj_target": M0_NBCO_ADJ_TARGET,
                            "n_iterations": M0_NBCO_ITERS,
                            "gnn_trimextend_bees": M0_NBCO_GNN_TRIMEXT_BEES,
                            "other_bco_bees": M0_NBCO_OTHER_BEES,
                            "connectivity_mode": CONNECTIVITY_MODE,
                            "init_tier": EXP_INIT_TIER})
    torch.save(M0_NBCO_HISTORY, PAPER_DIR / f"{M0_NBCO_TABLE}_history.pt")
    torch.save(M0_NBCO_MUTATION_COUNTS,
               PAPER_DIR / f"{M0_NBCO_TABLE}_mutation_counts.pt")
    print(f"[paper] histories/mutation counts -> {PAPER_DIR / (M0_NBCO_TABLE + '_history.pt')}")

    _viz_items = list(M0_NBCO_ROUTES.items())
    _fig, _axes = plt.subplots(2, len(_viz_items),
                               figsize=(4.2 * len(_viz_items), 8.2),
                               squeeze=False)
    _coords = _m0_tensors["node_locs"]
    _street_adj = _m0_tensors["street_adj"]
    _demand = _m0_tensors["demand"]
    _ref_routes = M0_NBCO_ROUTES[f"Initial (LC+{EXP_INIT_TIER})"]
    _m0_init_cost = None
    _m0_init_key = f"Initial (LC+{EXP_INIT_TIER})"
    if _m0_init_key in M0_NBCO_VARIANTS_DF.get("method", pd.Series(dtype=str)).values:
        _m0_init_cost = float(M0_NBCO_VARIANTS_DF.loc[
            M0_NBCO_VARIANTS_DF["method"] == _m0_init_key, "cost"].iloc[0])
    elif not M0_NBCO_VARIANTS_DF.empty:
        _m0_init_cost = float(M0_NBCO_VARIANTS_DF["cost"].iloc[0])

    def _m0_metric_line(_r, prefix, delta=None):
        _d = "" if delta is None else f"  d={delta:.3f}"
        return (f"{prefix}: cost={_r['cost']:.3f}{_d}  RTT={_r['RTT']:.0f}  "
                f"WMC={_r['WMC']:.2f}  adj={_r['adj_vs_seed']:.2f}  "
                f"redun={_r['redun%']:.0f}%")

    _m0_sub = {}
    _m0_cmp_sub = {}
    _m0_init_line = ""
    if _m0_init_key in M0_NBCO_VARIANTS_DF.get("method", pd.Series(dtype=str)).values:
        _m0_init_r = M0_NBCO_VARIANTS_DF.loc[
            M0_NBCO_VARIANTS_DF["method"] == _m0_init_key].iloc[0]
        _m0_init_line = _m0_metric_line(_m0_init_r, "init", 0.0)
    for _, _r in M0_NBCO_VARIANTS_DF.iterrows():
        _delta = (float(_r["cost"]) - _m0_init_cost
                  if _m0_init_cost is not None else float("nan"))
        _prefix = "init" if _r["method"] == _m0_init_key else "result"
        _line = _m0_metric_line(_r, _prefix, _delta)
        _m0_sub[_r["method"]] = _line
        _m0_cmp_sub[_r["method"]] = ((_m0_init_line + "\n" + _line)
                                     if _m0_init_line and _r["method"] != _m0_init_key
                                     else _line)
    for _col, (_label, _routes) in enumerate(_viz_items):
        _short = (_label.replace("Our NBCO ", "")
                         .replace(" (GNN + trim/extend)", "GNN+trim/extend"))
        route_plots.plot_plain_route_set(
            _axes[0, _col], _routes, _coords, _street_adj,
            title=_short, subtitle=_m0_sub.get(_label, "routes"), palette="tab20")
        if _col == 0:
            route_plots.plot_demand_graph(
                _axes[1, _col], _demand, _coords, _street_adj,
                title="OD demand", subtitle="bottom row: diff vs init",
                top_frac=0.20)
        else:
            route_plots.plot_route_diff(
                _axes[1, _col], _routes, _ref_routes, _coords, _street_adj,
                title=_short, subtitle=_m0_cmp_sub.get(_label, "diff vs init"), palette="tab20")
    _fig.suptitle(f"{M0_NBCO_CITY}: NBCO/RL variants, alpha={M0_NBCO_ALPHA}, "
                  f"target={M0_NBCO_ADJ_TARGET}", y=1.02, fontsize=15)
    plt.tight_layout()
    save_paper_fig(_fig, f"{M0_NBCO_TABLE}_routes")
    plt.show()
    plt.close(_fig)
else:
    print("No successful Mumford0 NBCO variant runs.")


## E2 - Pareto fronts on RTT x WMC (Mumford0, covered_dup tier)

Both panels run on **Mumford0** from the **covered_dup** init tier (removable
redundancy), optimizing `alpha*RTT + (1-alpha)*WMC + adj(|.-target|)` with
`use_weighted_connectivity=True`. `alpha` = `route_time_weight` (so alpha=1 ->
pure RTT, alpha=0 -> pure WMC).

- **Fig 1 (our model):** RTT x WMC Pareto front of `GNN + trim/extend`, swept
  over `alpha` x `adj_target` (one curve per target).
- **Fig 2 (4-model comparison):** `{GNN, RPC} x {trim/extend, type2}`, alpha-
  swept at a fixed `adj_target=ADJ_TARGET` -> shows the trim/extend
  bee's contribution to the RTT x WMC front.

In [ ]:
# === E2 -- Pareto fronts on RTT x WMC (Mumford1, realistic init tier) ===
E2_CITY = "Mumford1" if not SMOKE else "Mandl"
E2_ALPHA_GRID = [round(0.25 * i, 2) for i in range(5)] if not SMOKE else [0.0, 0.5, 1.0]
_E2_TARGET_GRID_FULL = [round(0.2 * i, 2) for i in range(1, 6)] if not SMOKE else [0.2, 0.6]
E2_TARGET_GRID = _E2_TARGET_GRID_FULL if RUN_E2_OUR_PARETO else []
E2_FIX_TARGET = min(_E2_TARGET_GRID_FULL, key=lambda _t: abs(_t - 0.2))  # alpha-slice target
E2_FIX_ALPHA = min(E2_ALPHA_GRID, key=lambda _a: abs(_a - 0.5))    # target-slice alpha
E2_FIXED_TARGET = E2_FIX_TARGET
E2_ADJ_OBJECTIVE = "target"
E2_DEFAULT_ADJ_WEIGHT = float(ADJ_WEIGHT)
E2_FIG2_ADJ_WEIGHT = 0.0
print("E2 grid:", {"city": E2_CITY, "alpha": E2_ALPHA_GRID,
                   "adj_target": E2_TARGET_GRID, "fixed_target": E2_FIXED_TARGET,
                   "iters": E2_BCO_ITERATIONS,
                   "adj_objective": E2_ADJ_OBJECTIVE,
                   "run_our_pareto": RUN_E2_OUR_PARETO,
                   "run_5model": RUN_E2_5MODEL})


In [ ]:
def _e2_ctx():
    spec = next(s for s in BENCHMARK_SPECS if s["city"] == E2_CITY)
    tensors, init = load_benchmark_graph(spec)
    s = algo_settings(E2_CITY)
    return dict(city=E2_CITY, spec=spec, tensors=tensors, init=init,
                nb=int(s["bco_bees"]), settings=s)


def _e2_adj_kwargs(adj_target, adj_weight=None):
    weight = E2_DEFAULT_ADJ_WEIGHT if adj_weight is None else float(adj_weight)
    return dict(UNIFIED_ADJ,
                adjustment_degree_weight=float(weight),
                adjustment_degree_target=float(adj_target),
                adjustment_degree_objective=E2_ADJ_OBJECTIVE)


def _e2_eval_cfg(ctx, alpha, tag):
    spec = ctx["spec"]
    cfg = _unify_weights(_eval_routes_cfg(ctx["city"], spec))
    _set_cfg_value(cfg, "run_name", f"{ctx['city']}_{tag}")
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.demand_time_weight", 0.0)
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.route_time_weight", float(alpha))
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.median_connectivity_weight", float(1.0 - alpha))
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
    return cfg


_E2_SEED_SCORE_CACHE = {}


def _score_rttwmc(ctx, routes, *, alpha, adj_target, tag, adj_weight=None):
    cfg = _e2_eval_cfg(ctx, alpha, tag)
    return _run_baseline(
        None, cfg, routes, f"{ctx['city']}_{tag}_", {},
        tensors=ctx["tensors"], use_weighted_connectivity=True,
        connectivity_mode=CONNECTIVITY_MODE,
        adjustment_seed_routes=ctx["init"],
        **_e2_adj_kwargs(adj_target, adj_weight=adj_weight))[1]


def _e2_seed_metrics(ctx, alpha, adj_target, adj_weight=None):
    weight = E2_DEFAULT_ADJ_WEIGHT if adj_weight is None else float(adj_weight)
    key = (float(alpha), float(adj_target), float(weight))
    if key not in _E2_SEED_SCORE_CACHE:
        _E2_SEED_SCORE_CACHE[key] = _score_rttwmc(
            ctx, ctx["init"], alpha=alpha, adj_target=adj_target,
            adj_weight=weight, tag=f"e2_seed_a{alpha}_t{adj_target}_w{weight}")
    return _E2_SEED_SCORE_CACHE[key]


def _run_rttwmc(ctx, construct, edit, *, alpha, adj_target, tag, seed=None,
                adj_weight=None):
    """One BCO run optimizing alpha*RTT + (1-alpha)*WMC + optional adj.

    ``adj_weight=0.0`` disables adjustment-degree acceptance; RPC rows use this
    in the main E2 comparison instead of a separate no-adj cell.
    """
    var = abl_variant(construct, edit, ctx["nb"])
    spec = ctx["spec"]
    weights = {"demand_time_weight": 0.0, "route_time_weight": float(alpha),
               "median_connectivity_weight": float(1.0 - alpha)}
    cfg = build_bco_cfg(
        run_name=f"{ctx['city']}_{tag}_{var['run_name']}",
        n_routes=spec["n_routes"], min_route_len=spec["min_route_len"],
        max_route_len=spec["max_route_len"], use_neural_bees=var["use_neural_bees"],
        n_bees=int(var.get("n_bees", ctx["nb"])), n_type1_bees=var["n_type1_bees"],
        n_type2_bees=var["n_type2_bees"],
        n_type4_bees=var.get("n_type4_bees", 0),
        n_type5_bees=var.get("n_type5_bees", 0),
        n_type6_bees=var.get("n_type6_bees", 0),
        n_type7_bees=var.get("n_type7_bees", 0),
        type4_allow_halt=bool(var.get("type4_allow_halt", False)),
        type5_allow_halt=bool(var.get("type5_allow_halt", False)),
        type6_allow_halt=bool(var.get("type6_allow_halt", False)),
        type7_allow_halt=bool(var.get("type7_allow_halt", False)),
        connectivity_mode=CONNECTIVITY_MODE,
        **ctx["settings"].get("bco_args", {}), **weights)
    bco_cfg_set(cfg, n_iterations=int(E2_BCO_ITERATIONS))
    bco_cfg_set(cfg, **_e2_adj_kwargs(adj_target, adj_weight=adj_weight))
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
    if seed is not None:
        _set_cfg_value(cfg, "experiment.seed", int(seed))
    t = _time.perf_counter()
    res = run_bco(cfg, ctx["init"], tensors=ctx["tensors"], run_name_scope=f"{ctx['city']}_")
    duration_s = _time.perf_counter() - t
    return res[3], res[1], duration_s


def _e2_row(ctx, series_col, label, alpha, target, routes, metrics,
            duration_s=None, seed_metrics=None, adj_weight=None):
    rt = as_route_tensor(routes)
    row = {"city": ctx["city"], "method": label, series_col: label,
           "alpha": float(alpha), "is_initial": False,
           "adj_target": float(target),
           "adj_weight": float(E2_DEFAULT_ADJ_WEIGHT if adj_weight is None else adj_weight),
           "duration_s": (round(float(duration_s), 1) if duration_s is not None else None),
           **_full_metrics(metrics, rt, ctx["init"])}
    if seed_metrics is not None:
        seed_full = _full_metrics(seed_metrics, ctx["init"], ctx["init"])
        seed_cost = float(seed_full["cost"])
        row["seed_cost"] = seed_cost
        row["cost_delta"] = float(row["cost"] - seed_cost)
        row["cost_delta_pct"] = (100.0 * row["cost_delta"] / abs(seed_cost)
                                  if abs(seed_cost) > 1e-12 else float("nan"))
    return row


def _e2_initial_row(ctx, series_col, label, alpha, target, seed_metrics,
                    adj_weight=None):
    row = _e2_row(ctx, series_col, label, alpha, target, ctx["init"],
                  seed_metrics, duration_s=None, seed_metrics=seed_metrics,
                  adj_weight=adj_weight)
    row["method"] = f"Initial (LC+{EXP_INIT_TIER})"
    row["source"] = "initial_route_set"
    row["is_initial"] = True
    row["duration_s"] = 0.0
    row["seed_cost"] = float(row["cost"])
    row["cost_delta"] = 0.0
    row["cost_delta_pct"] = 0.0
    return row


_AXIS_LABEL = {"RTT": "RTT (lower = better)", "WMC": "WMC (lower = better)",
              "rtt_cost": "normalized RTT cost (lower = better)",
              "wmc_cost": "normalized WMC cost (lower = better)",
              "adj_vs_seed": "adjustment degree vs seed (lower = less change)",
              "cost": "objective cost (lower = better)",
              "cost_delta": "cost delta vs init (lower = better)"}


def _plot_fronts(df, series_col, title, save_name, xcol="RTT", ycol="WMC"):
    fig, ax = plt.subplots(figsize=(8, 6))
    for lab, sub in df.groupby(series_col):
        sub = sub.sort_values(xcol)
        ax.plot(sub[xcol], sub[ycol], marker="o", alpha=0.85, label=str(lab))
        for _, r in sub.iterrows():
            tag = f"a={r['alpha']:g}"
            if "adj_target" in r and pd.notna(r["adj_target"]):
                tag += f",t={r['adj_target']:g}"
            if "adj_weight" in r and pd.notna(r["adj_weight"]) and float(r["adj_weight"]) == 0.0:
                tag += ",adj0"
            ax.annotate(tag, (r[xcol], r[ycol]), fontsize=6, alpha=0.7,
                        xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel(_AXIS_LABEL.get(xcol, xcol)); ax.set_ylabel(_AXIS_LABEL.get(ycol, ycol))
    ax.set_title(title); ax.legend(title=series_col); ax.grid(alpha=0.3)
    save_paper_fig(fig, save_name); plt.show(); plt.close(fig)


_ctx = _e2_ctx()
_sfx = ("_smoke" if SMOKE else "") + E2_TABLE_SUFFIX
print(f"=== E2 Pareto fronts on RTT x WMC ({E2_CITY}, tier={EXP_INIT_TIER}, iters={E2_BCO_ITERATIONS}) ===")

# --- Fig 1: OUR model (GNN + trim/extend), Pareto over alpha x target ---
_e2_our_table = f"final_e2_ourpareto_{E2_CITY}{_sfx}"
reset_paper_table(_e2_our_table)
def _e2_route_key(alpha, target):
    return f"target={float(target):g} | alpha={float(alpha):g}"


_our_rows = []
_our_routes = {f"Initial (LC+{EXP_INIT_TIER})": _ctx["init"]}
for _t in tqdm(E2_TARGET_GRID, desc=f"E2 {E2_CITY} our-model targets"):
    for _a in E2_ALPHA_GRID:
        try:
            _seed_m = _e2_seed_metrics(_ctx, _a, _t)
            _seed_row = _e2_initial_row(_ctx, "adj_target", f"target={_t:g}",
                                       _a, _t, _seed_m)
            _seed_row["objective"] = f"RTT_x_{CONNECTIVITY_MODE}_adj_target"
            _our_rows.append(_seed_row)
            append_paper_row(_seed_row, _e2_our_table, ndigits=4)
            r, m, dt = _run_rttwmc(_ctx, "GNN", "trim/extend", alpha=_a, adj_target=_t,
                                   tag=f"our_a{_a}_t{_t}", seed=SEEDS[0])
            row = _e2_row(_ctx, "adj_target", f"target={_t:g}", _a, _t, r, m,
                          duration_s=dt, seed_metrics=_seed_m)
            row["method"] = "Our NBCO (GNN + trim/extend)"
            row["source"] = "e2_our_gnn_trimextend"
            row["objective"] = f"RTT_x_{CONNECTIVITY_MODE}_adj_target"
            _our_rows.append(row)
            _our_routes[_e2_route_key(_a, _t)] = as_route_tensor(r)
            append_paper_row(row, _e2_our_table, ndigits=4)
            print(f"  [E2 our] a={_a:g} t={_t:g}: cost={row['cost']:.4f} "
                  f"delta={row.get('cost_delta', float('nan')):.4f} ({dt:.0f}s)", flush=True)
        except Exception as exc:
            print(f"  our alpha={_a} target={_t} FAILED: {exc}")
E2_OUR_DF = pd.DataFrame(_our_rows)
if not E2_OUR_DF.empty:
    display(E2_OUR_DF.round(4))
    save_paper_table(E2_OUR_DF.round(4), _e2_our_table)
    save_paper_routes(_e2_our_table, _our_routes,
                      _ctx["tensors"]["node_locs"], _ctx["tensors"]["street_adj"],
                      meta={"city": E2_CITY, "objective": "E2 alpha x target",
                            "connectivity_mode": CONNECTIVITY_MODE,
                            "adj_objective": E2_ADJ_OBJECTIVE,
                            "init_tier": EXP_INIT_TIER,
                            "alpha_grid": E2_ALPHA_GRID,
                            "target_grid": E2_TARGET_GRID,
                            "n_iterations": E2_BCO_ITERATIONS})
    E2_OUR_RUN_DF = E2_OUR_DF[~E2_OUR_DF["is_initial"].astype(bool)].copy()
    _plot_fronts(E2_OUR_RUN_DF, "adj_target",
                 f"E2 ({E2_CITY}): our model RTT x WMC -- alpha x target",
                 f"final_e2_ourpareto_combined_{E2_CITY}{_sfx}")
    _plot_fronts(E2_OUR_RUN_DF, "adj_target",
                 f"E2 ({E2_CITY}): our model adj x RTT -- alpha x target",
                 f"final_e2_ourpareto_adjrtt_{E2_CITY}{_sfx}",
                 xcol="adj_vs_seed", ycol="RTT")
    _plot_fronts(E2_OUR_RUN_DF, "adj_target",
                 f"E2 ({E2_CITY}): our model cost-delta x WMC",
                 f"final_e2_ourpareto_costdelta_{E2_CITY}{_sfx}",
                 xcol="cost_delta", ycol="WMC")
    _alpha_slice = E2_OUR_RUN_DF[np.isclose(E2_OUR_RUN_DF["adj_target"], E2_FIX_TARGET)]
    if not _alpha_slice.empty:
        _plot_fronts(_alpha_slice, "adj_target",
                     f"E2 ({E2_CITY}): alpha sweep on RTT x WMC (target={E2_FIX_TARGET:g} fixed)",
                     f"final_e2_alphasweep_{E2_CITY}{_sfx}")
    _target_slice = E2_OUR_RUN_DF[np.isclose(E2_OUR_RUN_DF["alpha"], E2_FIX_ALPHA)]
    if not _target_slice.empty:
        _plot_fronts(_target_slice, "alpha",
                     f"E2 ({E2_CITY}): adj-target sweep on RTT x WMC (alpha={E2_FIX_ALPHA:g} fixed)",
                     f"final_e2_targetsweep_{E2_CITY}{_sfx}")


In [ ]:
# --- Fig 2: 5-model comparison, alpha-swept with adjustment penalty OFF ---
# Fig 2 compares route-quality objectives only; adj_weight=0 for every model.
E2_MODEL_SPECS = [
    dict(label="GNN + trim/extend", construct="GNN", edit="trim/extend", adj_weight=E2_FIG2_ADJ_WEIGHT),
    dict(label="RPC + trim/extend", construct="RPC", edit="trim/extend", adj_weight=E2_FIG2_ADJ_WEIGHT),
    dict(label="GNN + type2", construct="GNN", edit="type2", adj_weight=E2_FIG2_ADJ_WEIGHT),
    dict(label="RPC + type2", construct="RPC", edit="type2", adj_weight=E2_FIG2_ADJ_WEIGHT),
    dict(label="trim 12 + extend 12", construct="none", edit="trim12+extend12", adj_weight=E2_FIG2_ADJ_WEIGHT),
]
_e2_abl_table = f"final_e2_5model_{E2_CITY}{_sfx}"
# RUN_E2_5MODEL gates this experiment (mirrors RUN_E2_OUR_PARETO above):
# when off, the grids are empty so no BCO runs and the display block below
# is skipped via the `if not E2_ABL_DF.empty` guard.
_e2_5model_alphas = E2_ALPHA_GRID if RUN_E2_5MODEL else []
_e2_5model_specs = E2_MODEL_SPECS if RUN_E2_5MODEL else []
if RUN_E2_5MODEL:
    reset_paper_table(_e2_abl_table)
else:
    print(f"[E2] {E2_CITY}: 5-model alpha sweep skipped (RUN_E2_5MODEL=False)")
_abl_rows = []
for _a in _e2_5model_alphas:
    _seed_m = _e2_seed_metrics(_ctx, _a, E2_FIXED_TARGET,
                              adj_weight=E2_FIG2_ADJ_WEIGHT)
    _seed_row = _e2_initial_row(_ctx, "model", f"Initial (LC+{EXP_INIT_TIER})",
                                _a, E2_FIXED_TARGET, _seed_m,
                                adj_weight=E2_FIG2_ADJ_WEIGHT)
    _seed_row["adj_enabled"] = bool(float(E2_FIG2_ADJ_WEIGHT) > 0)
    _seed_row["source"] = "e2_5model_initial_adj_off"
    _abl_rows.append(_seed_row)
    append_paper_row(_seed_row, _e2_abl_table, ndigits=4)
for _spec_m in tqdm(_e2_5model_specs, desc=f"E2 {E2_CITY} 5-model alpha sweep"):
    _label = _spec_m["label"]
    _adj_w = float(_spec_m["adj_weight"])
    for _a in _e2_5model_alphas:
        try:
            _seed_m = _e2_seed_metrics(_ctx, _a, E2_FIXED_TARGET, adj_weight=_adj_w)
            r, m, dt = _run_rttwmc(
                _ctx, _spec_m["construct"], _spec_m["edit"], alpha=_a,
                adj_target=E2_FIXED_TARGET, adj_weight=_adj_w,
                tag=(f"abl_{_label}_a{_a}".replace(" ", "_").replace("/", "_")),
                seed=SEEDS[0])
            row = _e2_row(_ctx, "model", _label, _a, E2_FIXED_TARGET, r, m,
                          duration_s=dt, seed_metrics=_seed_m,
                          adj_weight=_adj_w)
            row["adj_enabled"] = bool(_adj_w > 0)
            row["source"] = "e2_5model_adj_off"
            _abl_rows.append(row)
            append_paper_row(row, _e2_abl_table, ndigits=4)
            print(f"  [E2 model] {_label:26} a={_a:g} adj_w={_adj_w:g}: "
                  f"cost={row['cost']:.4f} delta={row.get('cost_delta', float('nan')):.4f} "
                  f"({dt:.0f}s)", flush=True)
        except Exception as exc:
            print(f"  {_label} alpha={_a} FAILED: {exc}")
E2_ABL_DF = pd.DataFrame(_abl_rows)
if not E2_ABL_DF.empty:
    display(E2_ABL_DF.round(4))
    save_paper_table(E2_ABL_DF.round(4), _e2_abl_table)
    E2_ABL_RUN_DF = E2_ABL_DF[~E2_ABL_DF["is_initial"].astype(bool)].copy()
    _plot_fronts(E2_ABL_RUN_DF, "model",
                 f"E2 ({E2_CITY}): 5 model RTT x WMC fronts (adj off)",
                 f"final_e2_5model_{E2_CITY}{_sfx}")
    _plot_fronts(E2_ABL_RUN_DF, "model",
                 f"E2 ({E2_CITY}): 5 model cost delta x WMC (adj off)",
                 f"final_e2_5model_costdelta_{E2_CITY}{_sfx}",
                 xcol="cost_delta", ycol="WMC")
    _p = E2_ABL_RUN_DF.pivot_table(index="alpha", columns="model", values="cost")
    _rpc_cols = [c for c in _p.columns if str(c).startswith("RPC")]
    if "GNN + trim/extend" in _p.columns and _rpc_cols:
        for _c in _rpc_cols:
            _p[f"GNN_trimext_minus_{_c}"] = (_p["GNN + trim/extend"] - _p[_c]).round(4)
    display(_p.round(4))
print(f"[E2] {E2_CITY}: 5-model alpha sweep done; "
      f"our-model alpha x target sweep run={RUN_E2_OUR_PARETO}.")


### E2 Route Visualisation

Draw selected `alpha x adj_target` route sets from the E2 our-model sweep. The first grid shows the route sets directly; the second grid highlights changes against the LC initial network.

In [ ]:
# Visualise E2 our-model routes for selected alpha x target combinations.
# Run the E2 cell above first so final_e2_ourpareto_<city>_routes.pt exists.
import math
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from eval_lib import plots as route_plots

E2_VIZ_CITY = E2_CITY
E2_VIZ_ALPHAS = [0.0, 0.5, 1.0]
E2_VIZ_TARGETS = [0.2, 0.6, 1.0]
E2_VIZ_OVERLAP_CURVES = True
E2_VIZ_SHOW_DIFF = True

_e2_viz_sfx = "_smoke" if globals().get("SMOKE", False) else ""
_e2_viz_table = f"final_e2_ourpareto_{E2_VIZ_CITY}{_e2_viz_sfx}"
_e2_viz_routes_path = PAPER_DIR / f"{_e2_viz_table}_routes.pt"
_e2_viz_csv_path = PAPER_DIR / f"{_e2_viz_table}.csv"
_e2_viz_ok = bool(globals().get("RUN_E2_ROUTE_VIZ", True)) and _e2_viz_routes_path.exists()
if not _e2_viz_ok:
    print(f"[E2 route viz] skipped (RUN_E2_ROUTE_VIZ="
          f"{globals().get('RUN_E2_ROUTE_VIZ', True)}, dump={_e2_viz_routes_path.name} "
          f"exists={_e2_viz_routes_path.exists()})")
else:
    _e2_dump = torch.load(_e2_viz_routes_path, weights_only=False)
    _e2_routes = _e2_dump["routes"]
    _e2_coords = _e2_dump["coords"]
    _e2_adj = _e2_dump["street_adj"]
    _e2_ref_key = next((k for k in _e2_routes if str(k).startswith("Initial")), list(_e2_routes)[0])
    _e2_ref_rt = as_route_tensor(_e2_routes[_e2_ref_key])
    _e2_ref_rr = _e2_ref_rt[0] if _e2_ref_rt.ndim == 3 else _e2_ref_rt

    _e2_viz_df = pd.read_csv(_e2_viz_csv_path) if _e2_viz_csv_path.exists() else pd.DataFrame()
    if not _e2_viz_df.empty:
        _e2_viz_df["alpha"] = pd.to_numeric(_e2_viz_df["alpha"], errors="coerce")
        _e2_viz_df["adj_target"] = pd.to_numeric(_e2_viz_df["adj_target"], errors="coerce")


    def _e2_viz_key(alpha, target):
        return f"target={float(target):g} | alpha={float(alpha):g}"


    def _e2_select_values(col, preferred):
        if _e2_viz_df.empty or col not in _e2_viz_df:
            return preferred
        available = sorted(float(v) for v in _e2_viz_df[col].dropna().unique())
        selected = [float(v) for v in preferred if any(np.isclose(v, a) for a in available)]
        return selected or available[:min(4, len(available))]


    _e2_plot_alphas = _e2_select_values("alpha", E2_VIZ_ALPHAS)
    _e2_plot_targets = _e2_select_values("adj_target", E2_VIZ_TARGETS)


    def _fmt_metric(value, digits=3):
        try:
            value = float(value)
        except (TypeError, ValueError):
            return "NA"
        if not np.isfinite(value):
            return "NA"
        return f"{value:.{digits}f}"


    def _e2_initial_flags(df):
        if "is_initial" in df:
            return df["is_initial"].astype(str).str.lower().isin(["true", "1", "yes"])
        if "method" in df:
            return df["method"].astype(str).str.startswith("Initial")
        return pd.Series(False, index=df.index)


    def _e2_metric_line(row, prefix, delta=None):
        _d = "" if delta is None else f"  d={_fmt_metric(delta)}"
        return (f"{prefix}: cost={_fmt_metric(row.get('cost'))}{_d}  "
                f"RTT={row['RTT']:.0f}  WMC={row['WMC']:.2f}  "
                f"adj={row['adj_vs_seed']:.2f}  redun={row['redun%']:.0f}%")


    def _e2_metric_subtitle(alpha, target):
        if _e2_viz_df.empty:
            return ""
        mask = (np.isclose(_e2_viz_df["alpha"], float(alpha)) &
                np.isclose(_e2_viz_df["adj_target"], float(target)))
        if not mask.any():
            return ""
        flags = _e2_initial_flags(_e2_viz_df)
        init_rows = _e2_viz_df.loc[mask & flags]
        result_rows = _e2_viz_df.loc[mask & ~flags]
        row = (result_rows.iloc[0] if not result_rows.empty
               else _e2_viz_df.loc[mask].iloc[0])
        init_line = ""
        if not init_rows.empty:
            init_line = _e2_metric_line(init_rows.iloc[0], "init", 0.0)
        elif pd.notna(row.get("seed_cost", np.nan)):
            init_line = f"init: cost={_fmt_metric(row.get('seed_cost'))}"
        result_line = _e2_metric_line(row, "result", row.get("cost_delta", np.nan))
        comps = (f"result: rtt_c={_fmt_metric(row.get('rtt_cost'))}  "
                 f"wmc_c={_fmt_metric(row.get('wmc_cost'))}  d_un={row['d_un']:.1f}%")
        return "\n".join([x for x in [init_line, result_line, comps] if x])


    def _draw_e2_route_grid(diff=False):
        if not _e2_plot_alphas or not _e2_plot_targets:
            print("No E2 alpha/target values available for plotting.")
            return
        fig, axes = plt.subplots(
            len(_e2_plot_targets), len(_e2_plot_alphas),
            figsize=(6.4 * len(_e2_plot_alphas), 6.1 * len(_e2_plot_targets)),
            squeeze=False)
        missing = []
        for row_idx, target in enumerate(_e2_plot_targets):
            for col_idx, alpha in enumerate(_e2_plot_alphas):
                ax = axes[row_idx, col_idx]
                key = _e2_viz_key(alpha, target)
                if key not in _e2_routes:
                    ax.axis("off")
                    ax.set_title(f"target={target:g}, alpha={alpha:g}\nmissing")
                    missing.append(key)
                    continue
                rt = as_route_tensor(_e2_routes[key])
                rr = rt[0] if rt.ndim == 3 else rt
                title = f"target={target:g}, alpha={alpha:g}"
                subtitle = _e2_metric_subtitle(alpha, target)
                if diff:
                    route_plots.plot_route_diff(
                        ax, rr, _e2_ref_rr, _e2_coords, _e2_adj, title=title,
                        subtitle=subtitle, palette="tab20",
                        with_overlap_curves=E2_VIZ_OVERLAP_CURVES)
                else:
                    route_plots.plot_plain_route_set(
                        ax, rr, _e2_coords, _e2_adj, title=title,
                        subtitle=subtitle, palette="tab20",
                        with_overlap_curves=E2_VIZ_OVERLAP_CURVES)
        mode = f"diff vs {_e2_ref_key}" if diff else "plain route sets"
        fig.suptitle(f"E2 {E2_VIZ_CITY}: our model routes by target x alpha ({mode})",
                     fontsize=15, fontweight="bold")
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show(); plt.close(fig)
        if missing:
            print(f"missing {len(missing)} route sets:", missing[:8])


    if not _e2_viz_df.empty:
        _e2_display = _e2_viz_df[
            _e2_viz_df["alpha"].apply(lambda x: any(np.isclose(x, a) for a in _e2_plot_alphas)) &
            _e2_viz_df["adj_target"].apply(lambda x: any(np.isclose(x, t) for t in _e2_plot_targets))
        ]
        display(_e2_display.round(4))

    _draw_e2_route_grid(diff=False)
    if E2_VIZ_SHOW_DIFF:
        _draw_e2_route_grid(diff=True)
    print(f"E2 route visualisation drawn from {_e2_viz_routes_path.name}; ref={_e2_ref_key!r}")


## E1 narrow -- neural BCO vs Our NBCO alpha sweep

Runs only two methods on every benchmark city (`Mandl`, `Mumford0`, `Mumford1`, `Mumford2`, `Mumford3`):
`neural BCO` and `Our NBCO (GNN rebuild + trim/extend)`. For each method we run
`alpha in {0, 0.5, 1}` with `adjustment_degree_target=0.5` and 100 BCO iterations.

All SA / GA / HH / heuristic BCO / NSGA-II / trim-only branches are disabled here. Tables are saved as
`paper_results/final_main_unified_<city><E1U_TABLE_SUFFIX>.csv` and a combined table as
`final_main_unified_comparison<E1U_TABLE_SUFFIX>.csv`.


In [ ]:
import torch


def _e1u_alpha_tag(alpha):
    return f"{float(alpha):.1f}".rstrip("0").rstrip(".").replace(".", "p") or "0"


def _e1u_alpha_weights(alpha):
    alpha = float(alpha)
    return {"demand_time_weight": 0.0,
            "route_time_weight": alpha,
            "median_connectivity_weight": 1.0 - alpha}


def _e1u_apply_alpha_and_target(cfg, *, alpha, adj_target):
    for key, value in _e1u_alpha_weights(alpha).items():
        _set_cfg_value(cfg, f"experiment.cost_function.kwargs.{key}", float(value))
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
    bco_cfg_set(cfg, n_iterations=int(E1U_BCO_ITERATIONS),
                **dict(UNIFIED_ADJ, adjustment_degree_target=float(adj_target)))
    return cfg


def _e1u_eval_fixed_routes(city, spec, routes, init, tensors, *, method, source,
                           alpha, adj_target, duration_s=None):
    cfg = _unify_weights(_eval_routes_cfg(city, spec))
    for key, value in _e1u_alpha_weights(alpha).items():
        _set_cfg_value(cfg, f"experiment.cost_function.kwargs.{key}", float(value))
    _set_cfg_value(cfg, "eval.csv", False)
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
    _run_name, metrics, _unserved, scored_routes = _run_baseline(
        None, cfg, routes, f"{city}_e1u_{method.lower().replace(' ', '_')}_score_",
        {}, tensors=tensors, use_weighted_connectivity=True,
        connectivity_mode=CONNECTIVITY_MODE,
        adjustment_seed_routes=init,
        **dict(UNIFIED_ADJ, adjustment_degree_target=float(adj_target)))
    row = _row(city, method, source, metrics, scored_routes, init,
               duration_s=duration_s)
    row.update(alpha=float(alpha), adj_target=float(adj_target),
               n_iterations=int(E1U_BCO_ITERATIONS),
               objective=f"alpha*RTT + (1-alpha)*WMC + adj(target={float(adj_target):g})")
    return row, as_route_tensor(scored_routes)


def _e1u_cache_key(method, alpha):
    return f"{method} (alpha={float(alpha):g}, target={float(E1U_ADJ_TARGET):g}, iter={int(E1U_BCO_ITERATIONS)})"


def _e1u_load_cache(table_name):
    if not globals().get("E1U_REUSE_BASELINE_CACHE", True):
        return {"rows": {}, "routes": {}, "history": {}}
    csv_path = PAPER_DIR / f"{table_name}.csv"
    routes_path = PAPER_DIR / f"{table_name}_routes.pt"
    history_path = PAPER_DIR / f"{table_name}_history.pt"
    if not (csv_path.exists() and routes_path.exists()):
        return {"rows": {}, "routes": {}, "history": {}}
    try:
        df = pd.read_csv(csv_path)
        dump = torch.load(routes_path, map_location="cpu", weights_only=False)
        routes = {k: as_route_tensor(v) for k, v in dump.get("routes", {}).items()}
        rows = {str(r["method"]): r for r in df.to_dict("records") if str(r.get("method")) in routes}
        history = torch.load(history_path, map_location="cpu", weights_only=False) if history_path.exists() else {}
        if not isinstance(history, dict):
            history = {}
        print(f"[E1 narrow cache] {table_name}: rows={len(rows)} routes={len(routes)}")
        return {"rows": rows, "routes": routes, "history": history}
    except Exception as exc:
        print(f"[E1 narrow cache] {table_name}: ignored ({exc})")
        return {"rows": {}, "routes": {}, "history": {}}


def _e1u_print_row(row, *, cached=False):
    prefix = "cached" if cached else "done"
    print(f"  [{prefix}] {row['method']:55} alpha={float(row['alpha']):.1f} "
          f"RTT={float(row['RTT']):.0f} WMC={float(row['WMC']):.2f} "
          f"adj={float(row['adj_vs_seed']):.3f} cost={float(row['cost']):.3f}",
          flush=True)


def run_city_e1u_nbco_our_alpha(spec, table_name=None, comparison_table_name=None):
    city = spec["city"]
    tensors, init = load_benchmark_graph(spec)
    init = as_route_tensor(init)
    rows, routes, histories = [], {}, {}
    cache = _e1u_load_cache(table_name) if table_name is not None else {"rows": {}, "routes": {}, "history": {}}
    if table_name is not None:
        reset_paper_table(table_name)

    def _write(row):
        if table_name is not None:
            append_paper_row(row, table_name, ndigits=3)
        if comparison_table_name is not None:
            append_paper_row(row, comparison_table_name, ndigits=3)

    def _add(row, route, hist=None, *, cached=False):
        rows.append(row)
        routes[str(row["method"])] = as_route_tensor(route)
        if hist is not None:
            try:
                histories[str(row["method"])] = _ravel_hist(hist)
            except Exception:
                histories[str(row["method"])] = [float(v) for v in np.asarray(hist).ravel()]
        _write(row)
        _e1u_print_row(row, cached=cached)

    for alpha in E1U_ALPHA_GRID:
        init_label = _e1u_cache_key(f"Initial (LC+{EXP_INIT_TIER})", alpha)
        init_row, init_scored = _e1u_eval_fixed_routes(
            city, spec, init, init, tensors, method=init_label, source="e1_initial",
            alpha=alpha, adj_target=E1U_ADJ_TARGET, duration_s=0.0)
        _add(init_row, init_scored)

        if E1U_RUN_NEURAL_BCO:   # baselines off -> skip neural BCO
            neural_label = _e1u_cache_key("neural BCO", alpha)
            if neural_label in cache["rows"] and neural_label in cache["routes"] and not E1U_RERUN_BASELINES:
                row = dict(cache["rows"][neural_label])
                row.update(city=city, method=neural_label)
                _add(row, cache["routes"][neural_label], cache["history"].get(neural_label), cached=True)
            else:
                print(f"  -> {neural_label} ...", flush=True)
                ch = {}
                v = next(x for x in BCO_VARIANTS if x["key"] == "seeded_neural")
                cfg = _variant_bco_cfg(city, spec, v, weights=_e1u_alpha_weights(alpha),
                                       run_name_suffix=f"e1_alpha{_e1u_alpha_tag(alpha)}_target05_")
                _e1u_apply_alpha_and_target(cfg, alpha=alpha, adj_target=E1U_ADJ_TARGET)
                t0 = _time.perf_counter()
                r = run_bco(cfg, init, tensors=tensors, run_name_scope=f"{city}_",
                            cost_history_out=ch)
                dt = _time.perf_counter() - t0
                row, scored = _e1u_eval_fixed_routes(
                    city, spec, r[3], init, tensors, method=neural_label,
                    source="e1_neural_bco", alpha=alpha, adj_target=E1U_ADJ_TARGET,
                    duration_s=dt)
                _add(row, scored, ch.get("history"))

        if E1U_RUN_OUR_MODEL:   # Our NBCO off -> neural-BCO-only run
            our_label = _e1u_cache_key("Our NBCO (GNN rebuild + trim/extend)", alpha)
            if our_label in cache["rows"] and our_label in cache["routes"] and not E1U_RERUN_OUR_MODEL:
                row = dict(cache["rows"][our_label])
                row.update(city=city, method=our_label)
                _add(row, cache["routes"][our_label], cache["history"].get(our_label), cached=True)
            else:
                print(f"  -> {our_label} ...", flush=True)
                ch = {}
                cfg = _our_model_cfg(city, spec, adj_target=E1U_ADJ_TARGET,
                                     seed=SEEDS[0], use_gnn=True)
                _e1u_apply_alpha_and_target(cfg, alpha=alpha, adj_target=E1U_ADJ_TARGET)
                t0 = _time.perf_counter()
                r = run_bco(cfg, init, tensors=tensors, run_name_scope=f"{city}_",
                            cost_history_out=ch)
                dt = _time.perf_counter() - t0
                row, scored = _e1u_eval_fixed_routes(
                    city, spec, r[3], init, tensors, method=our_label,
                    source="e1_our_nbco", alpha=alpha, adj_target=E1U_ADJ_TARGET,
                    duration_s=dt)
                _add(row, scored, ch.get("history"))

    return rows, routes, tensors, init, histories


_e1u_sfx = ("_smoke" if SMOKE else "") + E1U_TABLE_SUFFIX
_e1u_comparison_table = "final_main_unified_comparison" + _e1u_sfx
u_all_rows = []
if E1U_CITIES:
    reset_paper_table(_e1u_comparison_table)
for spec in [s for s in BENCHMARK_SPECS if s["city"] in E1U_CITIES]:
    _e1u_active = " + ".join([m for m, on in (("neural BCO", E1U_RUN_NEURAL_BCO),
                                              ("Our NBCO", E1U_RUN_OUR_MODEL)) if on]) or "Initial only"
    print(f"=== {spec['city']} E1 narrow: {_e1u_active}, alpha={E1U_ALPHA_GRID}, target={E1U_ADJ_TARGET}, iter={E1U_BCO_ITERATIONS} ===", flush=True)
    _city_table = f"final_main_unified_{spec['city']}{_e1u_sfx}"
    rows, routes, tensors, init, histories = run_city_e1u_nbco_our_alpha(
        spec, table_name=_city_table, comparison_table_name=_e1u_comparison_table)
    u_all_rows += rows
    save_paper_table(pd.DataFrame(rows).round(3), _city_table)
    save_paper_routes(_city_table, routes, tensors["node_locs"], tensors["street_adj"],
                      meta={"city": spec["city"],
                            "objective": "E1 narrow: neural BCO + Our NBCO alpha sweep",
                            "methods": ["neural BCO", "Our NBCO (GNN rebuild + trim/extend)"],
                            "alpha_grid": E1U_ALPHA_GRID,
                            "adj_target": float(E1U_ADJ_TARGET),
                            "bco_iterations": int(E1U_BCO_ITERATIONS),
                            "our_model_path": str(OUR_MODEL_PATH)})
    torch.save(histories, PAPER_DIR / f"{_city_table}_history.pt")
    print(f"=== {spec['city']} E1 narrow done ({len(rows)} rows) ===", flush=True)

unified_df = pd.DataFrame(u_all_rows).round(3)
if not unified_df.empty:
    display(unified_df)
    save_paper_table(unified_df, _e1u_comparison_table)
else:
    print("E1 narrow skipped: E1U_CITIES is empty.")
print("E1 narrow configured for neural BCO + Our NBCO only; "
      f"alphas={E1U_ALPHA_GRID}, target={E1U_ADJ_TARGET}, iter={E1U_BCO_ITERATIONS}.")


### Route visualisation (unified run: RTT + WMC + adj for all methods)

Drawn from the **unified** dumps (`final_main_unified_<city>`). Layout: OD demand -> init -> methods (plain & diff vs init), plus a focused 1x4 (our routes / init / neural-BCO diff / our-NBCO diff).

In [ ]:
# Autonomous route-set visualisation (reads paper_results dumps). Layout:
#   grid 1 (plain): [OD demand] [init] [methods...]
#   grid 2 (diff):  [OD demand] [init] [methods diff vs init]
#   + focused 1x4: our routes / init / neural-BCO diff / our-NBCO diff.
import math
import torch
import pandas as pd
import matplotlib.pyplot as plt
from eval_lib.results_io import ARTIFACTS_DIR
from eval_lib import plots as route_plots
from eval_lib import as_route_tensor, load_benchmark_tensors

VIZ_CITY = "Mumford0"          #  UNIFIED route dumps
VIZ_OVERLAP_CURVES = True      # overlap "ribbons"; set False for dense Mumford networks
VIZ_NCOL = 3
VIZ_DEMAND_TOP_FRAC = None     # None = all OD pairs; e.g. 0.15 keeps only busiest 15%

_prdir = ARTIFACTS_DIR / "paper_results"
_vsfx = ("_smoke" if SMOKE else "") + globals().get("E1U_TABLE_SUFFIX", "")
# ^ match the stem the E1 runner writes (final_main_unified_<city><E1U_TABLE_SUFFIX>)
_ptf = _prdir / f"final_main_unified_{VIZ_CITY}{_vsfx}_routes.pt"
_viz_ok = _ptf.exists()
if not _viz_ok:
    print(f"[unified route viz] skipped: no dump {_ptf.name} "
          f"(run E1u for {VIZ_CITY} first)")
else:
    _dump = torch.load(_ptf, weights_only=False)
    _routes, _coords, _adj = _dump["routes"], _dump["coords"], _dump["street_adj"]
    _methods = list(_routes)
    _demand = load_benchmark_tensors(VIZ_CITY)["demand"]   # OD matrix for the demand panel

    # initial network = reference for the diff version
    _ref_key = next((m for m in _methods if "Initial" in m), _methods[0])
    _ref_rt = as_route_tensor(_routes[_ref_key]); _ref_rr = _ref_rt[0] if _ref_rt.ndim == 3 else _ref_rt

    # optional metric subtitle from the matching results table
    _sub = {}
    _cmp_sub = {}
    _csvf = _prdir / f"final_main_unified_{VIZ_CITY}{_vsfx}.csv"
    if _csvf.exists():
        _df = pd.read_csv(_csvf).set_index("method")
        _init_cost = None
        if _ref_key in _df.index and "cost" in _df.columns:
            _init_cost = float(_df.loc[_ref_key, "cost"])

        def _viz_metric_line(_r, prefix, delta=None):
            _d = "" if delta is None else f"  d={delta:.3f}"
            return (f"{prefix}: cost={_r['cost']:.3f}{_d}  RTT={_r['RTT']:.0f}  "
                    f"WMC={_r['WMC']:.2f}  adj={_r['adj_vs_seed']:.2f}  "
                    f"d_un={_r['d_un']:.1f}%  redun={_r['redun%']:.0f}%")

        _init_line = (_viz_metric_line(_df.loc[_ref_key], "init", 0.0)
                      if _ref_key in _df.index else "")
        for _m, _r in _df.iterrows():
            _delta = (float(_r["cost"]) - _init_cost
                      if _init_cost is not None and "cost" in _df.columns else float("nan"))
            _prefix = "init" if _m == _ref_key else "result"
            _line = _viz_metric_line(_r, _prefix, _delta)
            _sub[_m] = _line
            _cmp_sub[_m] = ((_init_line + "\n" + _line)
                            if _init_line and _m != _ref_key else _line)


    def _draw_panel(ax, key, diff):
        if key == "__demand__":
            route_plots.plot_demand_graph(ax, _demand, _coords, _adj, title="OD demand",
                                          subtitle="edge color/width = demand volume",
                                          top_frac=VIZ_DEMAND_TOP_FRAC)
            return
        _rt = as_route_tensor(_routes[key]); _rr = _rt[0] if _rt.ndim == 3 else _rt
        try:
            if diff and key != _ref_key:
                route_plots.plot_route_diff(ax, _rr, _ref_rr, _coords, _adj, title=key,
                                            subtitle=_cmp_sub.get(key, _sub.get(key, "")), palette="tab20",
                                            with_overlap_curves=VIZ_OVERLAP_CURVES)
            else:
                route_plots.plot_plain_route_set(ax, _rr, _coords, _adj, title=key,
                                                 subtitle=_sub.get(key, ""), palette="tab20",
                                                 with_overlap_curves=VIZ_OVERLAP_CURVES)
        except Exception as exc:
            ax.set_title(f"{key}: plot failed ({exc})")


    def _draw_grid(diff):
        # panel order: demand first, init second, then every other method
        panels = ["__demand__", _ref_key] + [m for m in _methods if m != _ref_key]
        _nrow = math.ceil(len(panels) / VIZ_NCOL)
        fig, axes = plt.subplots(_nrow, VIZ_NCOL, figsize=(6.5 * VIZ_NCOL, 6.5 * _nrow), squeeze=False)
        for _ax, _panel in zip(axes.flat, panels):
            _draw_panel(_ax, _panel, diff)
        for _ax in axes.flat[len(panels):]:
            _ax.axis("off")
        _kind = f"DIFF vs {_ref_key}" if diff else "plain route sets"
        fig.suptitle(f"{VIZ_CITY} UNIFIED: demand + init + methods ({_kind})",
                     fontsize=15, fontweight="bold")
        plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.show(); plt.close(fig)


    _draw_grid(diff=False)   # version 1: plain route sets
    _draw_grid(diff=True)    # version 2: diff vs the initial network


    # --- focused 1x4: our routes / init / neural-BCO diff / our-NBCO diff ---
    _our_key = next((m for m in _methods if "Our NBCO" in m), None)
    _nbco_key = next((m for m in _methods if m == "neural BCO"),
                     next((m for m in _methods if "neural" in m.lower()), None))
    if _our_key and _nbco_key:
        _ourt = as_route_tensor(_routes[_our_key]); _our_rr = _ourt[0] if _ourt.ndim == 3 else _ourt
        _nbt = as_route_tensor(_routes[_nbco_key]); _nb_rr = _nbt[0] if _nbt.ndim == 3 else _nbt
        fig, axes = plt.subplots(1, 4, figsize=(26, 6.5), squeeze=False)
        route_plots.plot_plain_route_set(axes[0, 0], _our_rr, _coords, _adj,
                                         title="Our NBCO -- routes", subtitle=_sub.get(_our_key, ""),
                                         palette="tab20", with_overlap_curves=VIZ_OVERLAP_CURVES)
        route_plots.plot_plain_route_set(axes[0, 1], _ref_rr, _coords, _adj, title=_ref_key,
                                         subtitle=_sub.get(_ref_key, ""), palette="tab20",
                                         with_overlap_curves=VIZ_OVERLAP_CURVES)
        route_plots.plot_route_diff(axes[0, 2], _nb_rr, _ref_rr, _coords, _adj,
                                    title="neural BCO -- diff vs init", subtitle=_cmp_sub.get(_nbco_key, _sub.get(_nbco_key, "")),
                                    palette="tab20", with_overlap_curves=VIZ_OVERLAP_CURVES)
        route_plots.plot_route_diff(axes[0, 3], _our_rr, _ref_rr, _coords, _adj,
                                    title="Our NBCO -- diff vs init", subtitle=_cmp_sub.get(_our_key, _sub.get(_our_key, "")),
                                    palette="tab20", with_overlap_curves=VIZ_OVERLAP_CURVES)
        fig.suptitle(f"{VIZ_CITY} UNIFIED: routes / init / neural-BCO diff / our-NBCO diff",
                     fontsize=15, fontweight="bold")
        plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.show(); plt.close(fig)
        print(f"focused 4-panel: our={_our_key!r}, nbco={_nbco_key!r}, ref={_ref_key!r}")
    else:
        print(f"4-panel skipped (missing key): our={_our_key!r}, nbco={_nbco_key!r}")

    print(f"drawn demand + init + {len(_methods)} methods x2 (plain+diff) for {VIZ_CITY}; ref={_ref_key!r}")


## MACSA Table B -- paper routes + Our NBCO alpha sweep

This section replaces the old generic MACSA probe with the exact Mandl-8 Table-B workflow used for the paper figures. It reads the fixed route sets from `datasets/MACSA_data/mandl_8/routes_*.txt`, scores them with the same E1-style metric plumbing, then runs Our NBCO from the original network with `adjustment_degree_target = adj(MACSA, original)`.

Outputs:
- `final_macsa_mandl8_alpha_sweep_iter100.*`: Our NBCO alpha sweep, `alpha=0.0..1.0` step `0.1`, 100 BCO iterations per run.
- `final_macsa_mandl8_tableb.*`: Table-B paper methods plus the best Our NBCO sweep solution that beats MACSA on both RTT and WMC. The old cap-mode row is intentionally not included.


In [ ]:
# MACSA Table B helpers: scoring, Our NBCO sweep, and route grids.
import contextlib
import io
import math
import time as _t
from pathlib import Path
import sys

_MACSA_NOTEBOOK_DIR = Path("examples/route_generator").resolve()
if (_MACSA_NOTEBOOK_DIR / "eval_lib").exists() and str(_MACSA_NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(_MACSA_NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from IPython.display import Image, display

from connectpt.routes_generator.citygraph_dataset import load_macsa_tensors
from eval_lib import plots as route_plots
from eval_lib.baselines import _run_baseline
from eval_lib.context import DATASETS_DIR
from eval_lib.helpers import as_route_tensor, build_bco_cfg, run_bco
import eval_lib.helpers as _eh
from eval_lib.paper import (PAPER_DIR, UNIFIED_ADJ, adj_vs_init, bco_cfg_set,
                            eval_routes_cfg as _eval_routes_cfg,
                            paper_row as _row,
                            save_paper_routes, save_paper_table,
                            set_cfg_value as _set_cfg_value,
                            unify_weights as _unify_weights)
from eval_lib.params import ADJ_OBJECTIVE, ADJ_TARGET, CONNECTIVITY_MODE

# Keep this section runnable even when executed out of order during figure work.
try:
    OUR_MODEL_PATH
except NameError:
    from eval_lib.context import EDIT_MODEL_WEIGHTS_DIR
    OUR_MODEL_PATH = EDIT_MODEL_WEIGHTS_DIR / "improvement_lc_rttconn_adj_w10_t02_finetune100.pt"
try:
    SEEDS
except NameError:
    SEEDS = [0]
_eh.EDIT_MODEL_WEIGHTS_PATH = OUR_MODEL_PATH
_eh.EDIT_MODEL_N_ADJ_COND_FEATS = 0

MACSA_SCENARIO_NAME = "mandl_8"
MACSA_SCENARIO_DIR = DATASETS_DIR / "MACSA_data" / MACSA_SCENARIO_NAME
MACSA_METHOD_ORDER = ["original", "rga", "as", "ga", "ras", "mmas", "ma", "macsa"]
MACSA_METHOD_TITLE = {
    "original": "Original network [4]",
    "rga": "RGA",
    "as": "AS",
    "ga": "GA",
    "ras": "RAS",
    "mmas": "MMAS",
    "ma": "MA",
    "macsa": "MACSA",
}
MACSA_REF_METHOD = MACSA_METHOD_TITLE["original"]
MACSA_ARTICLE_STEM = "final_macsa_mandl8_tableb_article_only"
MACSA_SWEEP_BCO_ITERATIONS = 100
MACSA_SWEEP_BEES = 10
MACSA_SWEEP_SEED = int(SEEDS[0])
MACSA_SWEEP_FORCE_RERUN = False
MACSA_FORCE_CPU = False
MACSA_ALPHA_GRID = [round(i / 10.0, 1) for i in range(11)]
MACSA_SWEEP_STEM = f"final_macsa_mandl8_alpha_sweep_iter{MACSA_SWEEP_BCO_ITERATIONS}"
MACSA_COMPARISON_STEM = "final_macsa_mandl8_tableb"
MACSA_TABLEB_EVAL_ALPHA = 0.5
MACSA_TABLEB_EVAL_ADJ_TARGET = float(ADJ_TARGET)
MACSA_TABLEB_EVAL_ADJ_OBJECTIVE = str(ADJ_OBJECTIVE)
MACSA_NODE_SIZE = 70.0
MACSA_DPI = 220


def _macsa_alpha_tag(alpha):
    return f"{float(alpha):.1f}".rstrip("0").rstrip(".").replace("-", "m").replace(".", "p") or "0"


def _macsa_alpha_label(alpha):
    return f"Our NBCO alpha={float(alpha):.1f} (iter={MACSA_SWEEP_BCO_ITERATIONS})"


def _macsa_alpha_weights(alpha):
    alpha = float(alpha)
    return {"demand_time_weight": 0.0,
            "route_time_weight": alpha,
            "median_connectivity_weight": 1.0 - alpha}


def _macsa_set_alpha_weights(cfg, alpha):
    for key, value in _macsa_alpha_weights(alpha).items():
        _set_cfg_value(cfg, f"experiment.cost_function.kwargs.{key}", float(value))
    return cfg


def _macsa_read_routes_0indexed(path):
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.split("#", 1)[0].strip()
        if line:
            rows.append([int(tok) for tok in line.split()])
    if not rows:
        raise ValueError(f"No routes in {path}")
    width = max(len(row) for row in rows)
    out = torch.full((len(rows), width), -1, dtype=torch.long)
    for idx, row in enumerate(rows):
        out[idx, :len(row)] = torch.tensor(row, dtype=torch.long)
    return out


def _macsa_pad_routes(routes, n_routes, max_route_len):
    routes = as_route_tensor(routes).long()
    if routes.ndim == 3:
        routes = routes[0]
    if routes.shape[0] != int(n_routes):
        raise ValueError(f"Expected {n_routes} routes, got {tuple(routes.shape)}")
    if routes.shape[-1] > int(max_route_len):
        raise ValueError(f"Route width {routes.shape[-1]} exceeds {max_route_len}")
    if routes.shape[-1] < int(max_route_len):
        routes = torch.nn.functional.pad(routes, (0, int(max_route_len) - routes.shape[-1]), value=-1)
    return routes


def _macsa_2d(routes):
    routes = as_route_tensor(routes)
    return routes[0] if routes.ndim == 3 else routes


def _macsa_build_spec(raw_routes, n_nodes):
    n_routes = int(raw_routes[MACSA_REF_METHOD].shape[0])
    longest = max(int((rt > -1).sum(-1).max().item()) for rt in raw_routes.values())
    return {"city": MACSA_SCENARIO_NAME,
            "n_routes": n_routes,
            "min_route_len": 2,
            "max_route_len": min(int(n_nodes), max(12, longest))}


def _macsa_score_routes(method, source, routes, *, seed_routes, tensors, spec,
                        alpha, adj_target, adj_objective):
    cfg = _unify_weights(_eval_routes_cfg(MACSA_SCENARIO_NAME, spec))
    _macsa_set_alpha_weights(cfg, alpha)
    _set_cfg_value(cfg, "experiment.cpu", True)
    _set_cfg_value(cfg, "eval.csv", False)
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
    adj_kwargs = dict(UNIFIED_ADJ,
                      adjustment_degree_target=float(adj_target),
                      adjustment_degree_objective=str(adj_objective))
    t0 = _t.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()):
        _run_name, metrics, _unserved, scored_routes = _run_baseline(
            None, cfg, routes,
            f"{MACSA_SCENARIO_NAME}_{method.lower().replace(' ', '_')}_score_",
            {}, tensors=tensors,
            use_weighted_connectivity=True,
            connectivity_mode=CONNECTIVITY_MODE,
            adjustment_seed_routes=seed_routes,
            **adj_kwargs)
    row = _row(MACSA_SCENARIO_NAME, method, source, metrics, scored_routes,
               seed_routes, duration_s=_t.perf_counter() - t0)
    row.update(eval_alpha=float(alpha),
               eval_adj_target=float(adj_target),
               eval_adj_objective=str(adj_objective),
               objective=f"alpha*RTT + (1-alpha)*WMC + adj({adj_objective})")
    return row, as_route_tensor(scored_routes)


def _macsa_build_our_cfg(spec, *, alpha, adj_target, adj_objective,
                         n_iterations, n_bees, seed, force_cpu):
    n_bees = int(n_bees)
    rebuild_bees = max(1, n_bees // 2)
    trim_extend_bees = n_bees - rebuild_bees
    cfg = build_bco_cfg(
        run_name=f"{MACSA_SCENARIO_NAME}_alpha_sweep_our_nbco_gnn_rebuild_trimext",
        n_routes=spec["n_routes"], min_route_len=spec["min_route_len"],
        max_route_len=spec["max_route_len"], use_neural_bees=True,
        n_bees=n_bees, n_type1_bees=rebuild_bees, n_type2_bees=0,
        n_type4_bees=0, n_type5_bees=trim_extend_bees,
        n_type6_bees=0, n_type7_bees=0,
        force_cpu=bool(force_cpu), connectivity_mode=CONNECTIVITY_MODE,
        worse_accept_temperature=0.02, worse_accept_decay=0.985,
        worse_accept_min_temperature=0.001,
        worse_selection_temperature=0.02, worse_selection_decay=0.985,
        worse_selection_uniform_mix=0.10, worse_selection_elite_count=2,
        **_macsa_alpha_weights(alpha))
    bco_cfg_set(cfg, n_iterations=int(n_iterations),
                type4_allow_halt=False, type5_allow_halt=False,
                type6_allow_halt=False, type7_allow_halt=False,
                **dict(UNIFIED_ADJ,
                       adjustment_degree_target=float(adj_target),
                       adjustment_degree_objective=str(adj_objective)))
    _macsa_set_alpha_weights(cfg, alpha)
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
    _set_cfg_value(cfg, "eval.csv", False)
    _set_cfg_value(cfg, "experiment.seed", int(seed))
    return cfg


def _macsa_run_our_nbco(seed_routes, *, tensors, spec, alpha, adj_target,
                        adj_objective, n_iterations, n_bees, seed, force_cpu):
    cfg = _macsa_build_our_cfg(spec, alpha=alpha, adj_target=adj_target,
                               adj_objective=adj_objective,
                               n_iterations=n_iterations, n_bees=n_bees,
                               seed=seed, force_cpu=force_cpu)
    t0 = _t.perf_counter()
    _run_name, _metrics, _unserved, routes, _mutation_counts = run_bco(
        cfg, seed_routes, tensors=tensors, run_name_scope=f"{MACSA_SCENARIO_NAME}_")
    return as_route_tensor(routes), _t.perf_counter() - t0


def _macsa_metric_subtitle(row):
    if row is None:
        return ""
    return (f"ATT={float(row['ATT']):.2f}  WMC={float(row['WMC']):.2f}\n"
            f"RTT={float(row['RTT']):.0f}  cost={float(row['cost']):.3f}  "
            f"adj={float(row['adj_vs_seed']):.3f}")


def _macsa_relabel_nodes_1indexed(ax):
    for txt in ax.texts:
        value = txt.get_text()
        if value.isdigit():
            txt.set_text(str(int(value) + 1))


def _macsa_draw_grid(*, routes, rows_by_method, coords, street_adj, demand,
                     diff=False, ref_key=None, ref_routes=None,
                     include_demand=True, include_ref=True, ncol=5,
                     title="MACSA routes", node_size=MACSA_NODE_SIZE):
    panels = []
    if include_demand:
        panels.append("__demand__")
    if include_ref and ref_key is not None and ref_key in routes:
        panels.append(ref_key)
    panels += [name for name in routes if not (include_ref and name == ref_key)]
    ncol = max(1, int(ncol))
    nrow = math.ceil(len(panels) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.8 * ncol, 5.8 * nrow),
                             squeeze=False, constrained_layout=True)
    if ref_routes is None and ref_key is not None and ref_key in routes:
        ref_routes = routes[ref_key]
    ref_routes_2d = _macsa_2d(ref_routes) if ref_routes is not None else None
    for ax, panel in zip(axes.flat, panels):
        if panel == "__demand__":
            route_plots.plot_demand_graph(ax, demand, coords, street_adj,
                                          title="OD demand",
                                          subtitle="edge color/width = demand")
            _macsa_relabel_nodes_1indexed(ax)
            continue
        panel_routes = _macsa_2d(routes[panel])
        subtitle = _macsa_metric_subtitle(rows_by_method.get(panel))
        if diff and ref_routes_2d is not None and panel != ref_key:
            route_plots.plot_route_diff(ax, panel_routes, ref_routes_2d,
                                        coords, street_adj,
                                        title=f"{panel} vs original",
                                        subtitle=subtitle, palette="tab20",
                                        with_overlap_curves=True,
                                        show_node_labels=True,
                                        node_size=node_size)
        else:
            route_plots.plot_plain_route_set(ax, panel_routes, coords, street_adj,
                                             title=panel, subtitle=subtitle,
                                             palette="tab20",
                                             with_overlap_curves=True,
                                             show_node_labels=True,
                                             node_size=node_size)
        _macsa_relabel_nodes_1indexed(ax)
    for ax in axes.flat[len(panels):]:
        ax.axis("off")
    fig.suptitle(title, fontsize=15, fontweight="bold")
    return fig


def _macsa_save_fig(fig, stem, suffix):
    path = PAPER_DIR / f"{stem}_{suffix}.png"
    fig.savefig(path, dpi=MACSA_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"[paper] figure -> {path}")
    return path


def _macsa_display_image(path):
    try:
        display(Image(filename=str(path)))
    except Exception:
        print(path)


def _macsa_upsert_row(rows, row):
    return [r for r in rows if str(r.get("method")) != str(row.get("method"))] + [row]


def _macsa_select_best_sweep_row(sweep_df, macsa_row):
    work = sweep_df.copy()
    work["beats_macsa_rtt_wmc"] = ((work["RTT"].astype(float) < float(macsa_row["RTT"])) &
                                    (work["WMC"].astype(float) < float(macsa_row["WMC"])))
    work["joint_norm_rtt_wmc"] = (work["RTT"].astype(float) / float(macsa_row["RTT"]) +
                                  work["WMC"].astype(float) / float(macsa_row["WMC"]))
    pool = work[work["beats_macsa_rtt_wmc"]].copy()
    if pool.empty:
        pool = work.copy()
    pool = pool.sort_values(["joint_norm_rtt_wmc", "RTT", "WMC", "alpha"], ascending=True)
    return pool.iloc[0].to_dict()

print("MACSA Table-B config:", {
    "scenario_dir": str(MACSA_SCENARIO_DIR),
    "alpha_grid": MACSA_ALPHA_GRID,
    "bco_iterations": MACSA_SWEEP_BCO_ITERATIONS,
    "bco_bees": MACSA_SWEEP_BEES,
    "seed": MACSA_SWEEP_SEED,
    "force_cpu": MACSA_FORCE_CPU,
    "our_model": Path(OUR_MODEL_PATH).name,
})


In [ ]:
# Score the fixed Mandl-8 Table-B routes from the MACSA paper.
if not RUN_MACSA_EXPERIMENTS:
    MACSA_TABLEB_DF = pd.DataFrame()
    MACSA_TABLEB_ROUTES = {}
    print("MACSA Table-B skipped (RUN_MACSA_EXPERIMENTS=False).")
else:
    MACSA_TENSORS = load_macsa_tensors(MACSA_SCENARIO_DIR)
    MACSA_COORDS = MACSA_TENSORS["node_locs"]
    MACSA_STREET_ADJ = MACSA_TENSORS["street_adj"]
    MACSA_DEMAND = MACSA_TENSORS["demand"]
    MACSA_RAW_ROUTES = {
        MACSA_METHOD_TITLE[key]: _macsa_read_routes_0indexed(MACSA_SCENARIO_DIR / f"routes_{key}.txt")
        for key in MACSA_METHOD_ORDER
    }
    MACSA_SPEC = _macsa_build_spec(MACSA_RAW_ROUTES, int(MACSA_COORDS.shape[0]))
    MACSA_TABLEB_ROUTES = {
        method: _macsa_pad_routes(routes, MACSA_SPEC["n_routes"], MACSA_SPEC["max_route_len"])
        for method, routes in MACSA_RAW_ROUTES.items()
    }
    MACSA_SEED_ROUTES = MACSA_TABLEB_ROUTES[MACSA_REF_METHOD]
    MACSA_ADJ_TARGET_FROM_MACSA = float(adj_vs_init(MACSA_TABLEB_ROUTES["MACSA"], MACSA_SEED_ROUTES))

    MACSA_TABLEB_ROWS = []
    for method in MACSA_TABLEB_ROUTES:
        row, scored = _macsa_score_routes(
            method, "macsa_table_b", MACSA_TABLEB_ROUTES[method],
            seed_routes=MACSA_SEED_ROUTES, tensors=MACSA_TENSORS, spec=MACSA_SPEC,
            alpha=MACSA_TABLEB_EVAL_ALPHA,
            adj_target=MACSA_TABLEB_EVAL_ADJ_TARGET,
            adj_objective=MACSA_TABLEB_EVAL_ADJ_OBJECTIVE)
        row.update(alpha=np.nan, run_alpha=np.nan, n_iterations=np.nan,
                   macsa_adj_target=MACSA_ADJ_TARGET_FROM_MACSA)
        MACSA_TABLEB_ROWS.append(row)
        MACSA_TABLEB_ROUTES[method] = scored
        print(f"  {method:20} ATT={row['ATT']:.2f} RTT={row['RTT']:.0f} "
              f"WMC={row['WMC']:.2f} adj={row['adj_vs_seed']:.3f} cost={row['cost']:.3f}")

    MACSA_TABLEB_DF = pd.DataFrame(MACSA_TABLEB_ROWS).round(6)
    save_paper_table(MACSA_TABLEB_DF, MACSA_ARTICLE_STEM)
    save_paper_routes(MACSA_ARTICLE_STEM, MACSA_TABLEB_ROUTES,
                      MACSA_COORDS, MACSA_STREET_ADJ,
                      meta={"scenario": MACSA_SCENARIO_NAME,
                            "ref_method": MACSA_REF_METHOD,
                            "eval_alpha": MACSA_TABLEB_EVAL_ALPHA,
                            "eval_adj_target": MACSA_TABLEB_EVAL_ADJ_TARGET,
                            "macsa_adj_target": MACSA_ADJ_TARGET_FROM_MACSA})
    print(f"MACSA adj target from Table-B MACSA vs original: {MACSA_ADJ_TARGET_FROM_MACSA:.6f}")
    display(MACSA_TABLEB_DF)


In [ ]:
# Run Our NBCO alpha sweep: alpha=0.0..1.0 step 0.1, exactly 100 BCO iterations each.
if not RUN_MACSA_EXPERIMENTS or MACSA_TABLEB_DF.empty:
    MACSA_SWEEP_DF = pd.DataFrame()
    MACSA_SWEEP_ROUTES = {}
    print("MACSA alpha sweep skipped: no scored Table-B routes.")
else:
    sweep_csv = PAPER_DIR / f"{MACSA_SWEEP_STEM}.csv"
    sweep_routes_path = PAPER_DIR / f"{MACSA_SWEEP_STEM}_routes.pt"
    cached_routes = {}
    cached_duration = {}
    if sweep_routes_path.exists() and sweep_csv.exists() and not MACSA_SWEEP_FORCE_RERUN:
        try:
            payload = torch.load(sweep_routes_path, map_location="cpu", weights_only=False)
            meta = payload.get("meta", {})
            if int(meta.get("bco_iterations", -1)) == int(MACSA_SWEEP_BCO_ITERATIONS):
                cached_routes = {k: as_route_tensor(v) for k, v in payload.get("routes", {}).items()}
                cache_df = pd.read_csv(sweep_csv)
                if "method" in cache_df:
                    cached_duration = dict(zip(cache_df["method"].astype(str),
                                               cache_df.get("duration_s", pd.Series(dtype=float))))
                print(f"[macsa sweep] loaded cache: {sweep_routes_path.name}")
            else:
                print("[macsa sweep] cache ignored: iteration count mismatch")
        except Exception as exc:
            print(f"[macsa sweep] cache ignored: {exc}")

    MACSA_SWEEP_ROWS = []
    MACSA_SWEEP_ROUTES = {}

    def _macsa_save_sweep_progress():
        df = pd.DataFrame(MACSA_SWEEP_ROWS).sort_values("alpha").round(6)
        save_paper_table(df, MACSA_SWEEP_STEM)
        save_paper_routes(MACSA_SWEEP_STEM, MACSA_SWEEP_ROUTES,
                          MACSA_COORDS, MACSA_STREET_ADJ,
                          meta={"scenario": MACSA_SCENARIO_NAME,
                                "ref_method": MACSA_REF_METHOD,
                                "alpha_grid": MACSA_ALPHA_GRID,
                                "bco_iterations": MACSA_SWEEP_BCO_ITERATIONS,
                                "bco_bees": MACSA_SWEEP_BEES,
                                "seed": MACSA_SWEEP_SEED,
                                "adj_target": MACSA_ADJ_TARGET_FROM_MACSA,
                                "adj_objective": "target"})
        return df

    for alpha in tqdm(MACSA_ALPHA_GRID, desc="MACSA Our NBCO alpha sweep"):
        label = _macsa_alpha_label(alpha)
        if label in cached_routes and not MACSA_SWEEP_FORCE_RERUN:
            print(f"  [cache] {label}")
            routes = _macsa_pad_routes(cached_routes[label], MACSA_SPEC["n_routes"], MACSA_SPEC["max_route_len"])
            duration = float(cached_duration.get(label, np.nan)) if label in cached_duration else np.nan
            source = "our_nbco_alpha_sweep_cached"
        else:
            print(f"  [run] {label}: alpha={alpha:.1f}, iters={MACSA_SWEEP_BCO_ITERATIONS}", flush=True)
            routes, duration = _macsa_run_our_nbco(
                MACSA_SEED_ROUTES, tensors=MACSA_TENSORS, spec=MACSA_SPEC,
                alpha=float(alpha), adj_target=MACSA_ADJ_TARGET_FROM_MACSA,
                adj_objective="target", n_iterations=MACSA_SWEEP_BCO_ITERATIONS,
                n_bees=MACSA_SWEEP_BEES, seed=MACSA_SWEEP_SEED,
                force_cpu=MACSA_FORCE_CPU)
            source = "our_nbco_alpha_sweep"
        row, scored = _macsa_score_routes(
            label, source, routes, seed_routes=MACSA_SEED_ROUTES,
            tensors=MACSA_TENSORS, spec=MACSA_SPEC,
            alpha=float(alpha), adj_target=MACSA_ADJ_TARGET_FROM_MACSA,
            adj_objective="target")
        row.update(duration_s=duration, alpha=float(alpha), run_alpha=float(alpha),
                   n_iterations=int(MACSA_SWEEP_BCO_ITERATIONS),
                   macsa_adj_target=MACSA_ADJ_TARGET_FROM_MACSA,
                   beats_macsa_rtt_wmc=bool(
                       (float(row["RTT"]) < float(MACSA_TABLEB_DF.loc[MACSA_TABLEB_DF["method"] == "MACSA", "RTT"].iloc[0])) and
                       (float(row["WMC"]) < float(MACSA_TABLEB_DF.loc[MACSA_TABLEB_DF["method"] == "MACSA", "WMC"].iloc[0]))))
        MACSA_SWEEP_ROWS = _macsa_upsert_row(MACSA_SWEEP_ROWS, row)
        MACSA_SWEEP_ROUTES[label] = scored
        MACSA_SWEEP_DF = _macsa_save_sweep_progress()
        print(f"    -> RTT={row['RTT']:.0f} WMC={row['WMC']:.2f} "
              f"ATT={row['ATT']:.2f} adj={row['adj_vs_seed']:.3f} cost={row['cost']:.3f} "
              f"duration={duration:.1f}s", flush=True)

    MACSA_SWEEP_DF = pd.DataFrame(MACSA_SWEEP_ROWS).sort_values("alpha").round(6)
    display(MACSA_SWEEP_DF)


In [ ]:
# Pick the best Our NBCO sweep solution that beats MACSA on both RTT and WMC,
# then build the comparison table: all Table-B methods + that single Our solution.
if not RUN_MACSA_EXPERIMENTS or MACSA_SWEEP_DF.empty:
    MACSA_COMPARISON_DF = pd.DataFrame()
    MACSA_COMPARISON_ROUTES = {}
    print("MACSA comparison skipped: no sweep rows.")
else:
    macsa_ref_row = MACSA_TABLEB_DF[MACSA_TABLEB_DF["method"] == "MACSA"].iloc[0]
    MACSA_BEST_SWEEP_ROW = _macsa_select_best_sweep_row(MACSA_SWEEP_DF, macsa_ref_row)
    MACSA_BEST_SWEEP_LABEL = str(MACSA_BEST_SWEEP_ROW["method"])
    MACSA_BEST_ALPHA = float(MACSA_BEST_SWEEP_ROW["alpha"])
    MACSA_BEST_BEATS_MACSA = bool(MACSA_BEST_SWEEP_ROW["beats_macsa_rtt_wmc"])
    MACSA_BEST_COMPARE_LABEL = f"Our NBCO best (alpha={MACSA_BEST_ALPHA:.1f}, iter={MACSA_SWEEP_BCO_ITERATIONS})"

    best_compare_row, best_compare_routes = _macsa_score_routes(
        MACSA_BEST_COMPARE_LABEL, "our_nbco_alpha_sweep_best",
        MACSA_SWEEP_ROUTES[MACSA_BEST_SWEEP_LABEL],
        seed_routes=MACSA_SEED_ROUTES, tensors=MACSA_TENSORS, spec=MACSA_SPEC,
        alpha=MACSA_TABLEB_EVAL_ALPHA,
        adj_target=MACSA_TABLEB_EVAL_ADJ_TARGET,
        adj_objective=MACSA_TABLEB_EVAL_ADJ_OBJECTIVE)
    best_compare_row.update(alpha=MACSA_BEST_ALPHA,
                            run_alpha=MACSA_BEST_ALPHA,
                            n_iterations=int(MACSA_SWEEP_BCO_ITERATIONS),
                            macsa_adj_target=MACSA_ADJ_TARGET_FROM_MACSA,
                            beats_macsa_rtt_wmc=MACSA_BEST_BEATS_MACSA,
                            selected_from=MACSA_BEST_SWEEP_LABEL,
                            selection_rule="min RTT/MACSA_RTT + WMC/MACSA_WMC among rows beating MACSA on both")

    MACSA_COMPARISON_ROUTES = dict(MACSA_TABLEB_ROUTES)
    MACSA_COMPARISON_ROUTES[MACSA_BEST_COMPARE_LABEL] = best_compare_routes
    MACSA_COMPARISON_DF = pd.concat(
        [MACSA_TABLEB_DF, pd.DataFrame([best_compare_row])], ignore_index=True, sort=False).round(6)
    save_paper_table(MACSA_COMPARISON_DF, MACSA_COMPARISON_STEM)
    save_paper_routes(MACSA_COMPARISON_STEM, MACSA_COMPARISON_ROUTES,
                      MACSA_COORDS, MACSA_STREET_ADJ,
                      meta={"scenario": MACSA_SCENARIO_NAME,
                            "ref_method": MACSA_REF_METHOD,
                            "article_methods": [MACSA_METHOD_TITLE[k] for k in MACSA_METHOD_ORDER],
                            "best_our_method": MACSA_BEST_COMPARE_LABEL,
                            "best_sweep_method": MACSA_BEST_SWEEP_LABEL,
                            "best_beats_macsa_rtt_wmc": MACSA_BEST_BEATS_MACSA,
                            "bco_iterations": MACSA_SWEEP_BCO_ITERATIONS,
                            "sweep_stem": MACSA_SWEEP_STEM,
                            "cap_mode_included": False})
    print("Best Our NBCO sweep solution:", {
        "method": MACSA_BEST_SWEEP_LABEL,
        "alpha": MACSA_BEST_ALPHA,
        "beats_macsa_rtt_wmc": MACSA_BEST_BEATS_MACSA,
        "RTT": float(MACSA_BEST_SWEEP_ROW["RTT"]),
        "WMC": float(MACSA_BEST_SWEEP_ROW["WMC"]),
        "ATT": float(MACSA_BEST_SWEEP_ROW["ATT"]),
        "adj": float(MACSA_BEST_SWEEP_ROW["adj_vs_seed"]),
    })
    display(MACSA_COMPARISON_DF)


In [ ]:
# Visualize only Our NBCO alpha-sweep solutions.
if not RUN_MACSA_EXPERIMENTS or MACSA_SWEEP_DF.empty:
    print("MACSA our-only visualization skipped: no sweep rows.")
else:
    sweep_order = [str(row["method"]) for _, row in MACSA_SWEEP_DF.sort_values("alpha").iterrows()]
    sweep_routes_ordered = {name: MACSA_SWEEP_ROUTES[name] for name in sweep_order}
    sweep_rows_by_method = {str(row["method"]): row.to_dict()
                            for _, row in MACSA_SWEEP_DF.iterrows()}

    fig = _macsa_draw_grid(
        routes=sweep_routes_ordered, rows_by_method=sweep_rows_by_method,
        coords=MACSA_COORDS, street_adj=MACSA_STREET_ADJ, demand=MACSA_DEMAND,
        diff=False, ref_key=None, ref_routes=MACSA_SEED_ROUTES,
        include_demand=True, include_ref=False, ncol=4,
        title="Mandl-8 MACSA: Our NBCO alpha sweep, plain route sets")
    MACSA_SWEEP_PLAIN_PATH = _macsa_save_fig(fig, MACSA_SWEEP_STEM, "viz_plain")

    fig = _macsa_draw_grid(
        routes=sweep_routes_ordered, rows_by_method=sweep_rows_by_method,
        coords=MACSA_COORDS, street_adj=MACSA_STREET_ADJ, demand=MACSA_DEMAND,
        diff=True, ref_key=MACSA_REF_METHOD, ref_routes=MACSA_SEED_ROUTES,
        include_demand=True, include_ref=False, ncol=4,
        title="Mandl-8 MACSA: Our NBCO alpha sweep, diff vs original")
    MACSA_SWEEP_DIFF_PATH = _macsa_save_fig(fig, MACSA_SWEEP_STEM, "viz_diff")
    _macsa_display_image(MACSA_SWEEP_PLAIN_PATH)
    _macsa_display_image(MACSA_SWEEP_DIFF_PATH)


In [ ]:
# Visualize all paper Table-B solutions plus the selected best Our NBCO solution.
if not RUN_MACSA_EXPERIMENTS or MACSA_COMPARISON_DF.empty:
    print("MACSA comparison visualization skipped: no comparison table.")
else:
    comparison_rows_by_method = {str(row["method"]): row.to_dict()
                                 for _, row in MACSA_COMPARISON_DF.iterrows()}
    fig = _macsa_draw_grid(
        routes=MACSA_COMPARISON_ROUTES, rows_by_method=comparison_rows_by_method,
        coords=MACSA_COORDS, street_adj=MACSA_STREET_ADJ, demand=MACSA_DEMAND,
        diff=False, ref_key=MACSA_REF_METHOD, ref_routes=MACSA_SEED_ROUTES,
        include_demand=True, include_ref=True, ncol=5,
        title="Mandl-8 MACSA Table B: paper methods + best Our NBCO")
    MACSA_COMPARISON_PLAIN_PATH = _macsa_save_fig(fig, MACSA_COMPARISON_STEM, "viz_plain")

    fig = _macsa_draw_grid(
        routes=MACSA_COMPARISON_ROUTES, rows_by_method=comparison_rows_by_method,
        coords=MACSA_COORDS, street_adj=MACSA_STREET_ADJ, demand=MACSA_DEMAND,
        diff=True, ref_key=MACSA_REF_METHOD, ref_routes=MACSA_SEED_ROUTES,
        include_demand=True, include_ref=True, ncol=5,
        title="Mandl-8 MACSA Table B: paper methods + best Our NBCO, diff vs original")
    MACSA_COMPARISON_DIFF_PATH = _macsa_save_fig(fig, MACSA_COMPARISON_STEM, "viz_diff")
    _macsa_display_image(MACSA_COMPARISON_PLAIN_PATH)
    _macsa_display_image(MACSA_COMPARISON_DIFF_PATH)
